<a href="https://colab.research.google.com/github/Minahil-Habib/Indeed-Job-Pipeline/blob/main/indeed_pipline_finalized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Project Overview and Objectives












## 1.1 Project Overview

The objective of this project is to re-architect the existing Indeed data pipeline used to transfer job advertisement data from the development source table into the production job board environment. The current pipeline uses **Python** and **Google BigQuery**, with `daily_scrape_dev` acting as the source table and `vw_dca_jobboard` as the production target. The revised pipeline will introduce a controlled staging layer, `jobboard_staging`, where cleaned, deduplicated, transformed and AI-enriched records can be reviewed before final integration into the production dataset.

The pipeline will focus on improving data quality through four main areas:

1. field mapping and transformation,
2. cleaning and standardisation,
3. duplicate detection and removal,
4. AI-assisted data enrichment.

The pipeline will be developed and evaluated in a Google Colab notebook and will include experimental comparisons for location classification, deduplication and AI enrichment before selecting the most suitable production approach.

---

## 1.2 Pipeline Scope

The source-to-target mapping will use the supplied field mapping specification as the authoritative mapping definition.

A total of **20 target fields** will be included in the pipeline.

No additional recommended fields will be introduced into the field mapping.

The mapped fields will subsequently be classified according to whether they require:

* direct mapping,
* renaming,
* datatype conversion,
* cleaning,
* deterministic transformation,
* derived logic,
* or AI enrichment.

This ensures that all downstream processing is restricted to the agreed project scope.

---

## 1.3 Core Project Objectives

The pipeline has the following objectives.

### Objective 1 — Schema Analysis and Field Mapping

Analyse the schemas of `daily_scrape_dev` and `vw_dca_jobboard` and define the transformation required to map the source dataset into the agreed **20-field target structure**.

The mapping will identify:

* source and target fields,
* source and target datatypes,
* transformation rules,
* cleaning requirements,
* derived fields,
* and AI-enriched fields.

---

### Objective 2 — Data Cleaning and Standardisation

Clean and standardise the extracted Indeed records before any deduplication or AI enrichment is performed.

The cleaning stage will address issues including:

* null and blank values,
* unnecessary whitespace,
* HTML tags and HTML entities,
* inconsistent text formatting,
* location formatting,
* malformed dates,
* and incompatible datatypes.

This stage establishes a consistent dataset for subsequent experiments.

---

### Objective 3 — Location Type Determination

Develop and compare two approaches for classifying each job location into one of the following expected values:

* `Metro`
* `Regional`
* `Remote`

Two experimental cases will be evaluated.

**Case 1 — AI-only classification**

The cleaned location information will be supplied directly to an AI model, which will return the appropriate location type.

**Case 2 — Hybrid deterministic + AI classification**

The location will first be classified deterministically using the supplied **GCCSA 2026 shapefile**. Where the deterministic process cannot return a valid classification, such as when the result is `None`, `NaN`, null or otherwise unresolved, the record will be passed to the AI model as a fallback.

Both approaches will be evaluated using the same records and benchmark metrics.

---

### Objective 4 — Source-Level `jobKey` Deduplication

Before evaluating alternative deduplication strategies, duplicate records within the Indeed source dataset will first be identified using `jobKey`.

`jobKey` will be treated as the Indeed job identifier.

Where duplicate `jobKey` values exist:

* one record will be retained,
* additional records with the same `jobKey` will be discarded.

The resulting dataset will then be used consistently across the remaining deduplication experiments.

---

### Objective 5 — Deduplication Strategy Comparison

Three deduplication approaches will be developed and evaluated.

**Case 1**

Create a deduplication key using:

`company + job title + location`

**Case 2**

Use:

`company + job title + location + date`

The date component will use a percentage-based similarity or matching mechanism rather than relying only on exact date equality.

**Case 3**

Create a deduplication key using:

`company + job title + location + descriptionTextHash`

The approaches will be compared to determine which method provides the most reliable identification of duplicate job advertisements.

The production design will also retain the project's required **48-hour lookback principle** when comparing incoming Indeed records against recent records in the existing job board.

---

### Objective 6 — AI Enrichment Strategy Comparison

The AI enrichment stage will compare four implementation strategies.

The existing prompt configuration includes enrichment tasks such as technical skill extraction, role categorisation, business function, industry, experience level, residency or citizenship requirements and security-clearance requirements.

The four test cases will be:

**Case 1 — Single combined prompt defined in the notebook**

Use one prompt containing the required inputs, instructions and expected outputs for all enrichment fields.

**Case 2 — Separate prompt strings defined in the notebook**

Use separate prompt definitions for each enrichment task, with inputs, instructions and expected outputs handled independently.

**Case 3 — Single combined prompt using the JSON prompt library**

Use the same conceptual design as Case 1 while loading the underlying prompt definitions from `indeed_stage2_prompts.json`.

**Case 4 — Separate prompts using the JSON prompt library**

Use separate enrichment prompts, with the input fields, prompt instructions and expected outputs dynamically loaded from the JSON file.

---

## 1.4 Experimental Evaluation Framework

All competing approaches will be evaluated using consistent datasets wherever possible.

The evaluation will consider three broad dimensions.

### Output Quality

Depending on the experiment, quality measures will include:

* classification accuracy,
* duplicate-detection precision,
* duplicate-detection recall,
* F1 score,
* valid output rate,
* schema compliance,
* missing-output rate,
* and manually validated correctness.

### Processing Efficiency

Efficiency measures will include:

* total execution time,
* average processing time per record,
* records processed per unit of time,
* number of AI API calls,
* retry counts,
* and failed operations.

### Resource Usage

Where measurable, resource metrics will include:

* memory usage,
* token usage,
* input tokens,
* output tokens,
* total tokens,
* estimated AI cost,
* and other relevant computational-resource measures.

The purpose of the experiments is therefore not simply to determine whether an approach works, but to determine which approach provides the strongest balance between **data quality, processing efficiency, operational cost and maintainability**.

---

## 1.5 Expected Pipeline Outcome

The final pipeline will follow the general processing flow:

`daily_scrape_dev`

→ extraction

→ 20-field mapping

→ cleaning and standardisation

→ `jobKey` duplicate removal

→ location-type determination

→ cross-dataset deduplication

→ AI enrichment

→ data-quality validation

→ `jobboard_staging`

→ manual review and approval

→ `vw_dca_jobboard`

The staging-table approach ensures that records are reviewed before being promoted into the production job board, consistent with the governance requirements of the original pipeline brief.

---

## 1.6 Expected Deliverable

The primary deliverable will be a structured Google Colab notebook that:

* documents the complete pipeline,
* contains the implementation code,
* performs detailed exploratory data analysis,
* evaluates each experimental test case,
* records performance and resource metrics,
* compares alternative approaches,
* and provides evidence-based recommendations for the final production pipeline.

The final notebook should make it possible for both technical reviewers and stakeholders to understand not only the selected approach, but also **why it was selected over the alternatives**.

# 2. Environment Setup and Configuration

This section prepares the Google Colab environment required for the pipeline.

The project brief specifies Python and Google BigQuery as the core technologies and requires the final implementation to support data cleaning, deduplication, AI enrichment, staging-table ingestion, and data-quality validation.

The environment must therefore support:

BigQuery access,
Pandas-based processing,
geospatial processing of the GCCSA shapefile,
HTML cleaning,
AI enrichment using Gemini,
JSON prompt configuration,
execution-time measurement,
memory/resource measurement,
and structured experiment logging.

## 2.1 Install Required Packages

Google Colab already includes many of the required libraries. Install only the additional packages needed by the pipeline.

In [ ]:
%pip install -q \ google-genai \ geopandas \ shapely \ pyogrio \ beautifulsoup4 \ psutil \ rapidfuzz \ openpyxl

## 2.2 Import Python Libraries

Import the libraries required throughout the notebook.

In [ ]:
# Standard library
import os
import re
import gc
import json
import math
import time
import hashlib
import logging
import zipfile
import warnings
from pathlib import Path
from datetime import datetime, timedelta, timezone
from contextlib import contextmanager

# Data processing
import numpy as np
import pandas as pd

# BigQuery
from google.cloud import bigquery

# Google Colab
from google.colab import auth
from google.colab import userdata
from google.oauth2 import service_account

# AI
from google import genai

# Cleaning
from bs4 import BeautifulSoup

# Geospatial
import geopandas as gpd
from shapely.geometry import Point

# Similarity
from rapidfuzz import fuzz

# Resource monitoring
import psutil

# Display
from IPython.display import display

warnings.filterwarnings("ignore")
print("✓ Libraries imported successfully.")

✓ Libraries imported successfully.


## 2.3 Set BigQuery Project Configuration

In [ ]:
PROJECT_ID = "datacareerapp"
SOURCE_TABLE = "indeed_jobs.daily_scrape_dev"
TARGET_TABLE = "jobboard_prod_db.vw_dca_jobboard"
STAGING_TABLE = "jobboard_staging"
SOURCE_TABLE_ID = ( f"{PROJECT_ID}.{SOURCE_TABLE}" )
TARGET_TABLE_ID = ( f"{PROJECT_ID}.{TARGET_TABLE}" )
STAGING_TABLE_ID = ( f"{PROJECT_ID}.{STAGING_TABLE}" )

print("Source :", SOURCE_TABLE_ID)
print("Target :", TARGET_TABLE_ID)
print("Staging:", STAGING_TABLE_ID)

Source : datacareerapp.indeed_jobs.daily_scrape_dev
Target : datacareerapp.jobboard_prod_db.vw_dca_jobboard
Staging: datacareerapp.jobboard_staging


## 2.4 Initialise the BigQuery Client

In [ ]:
gcp_key_str = userdata.get('gcp_json_key')
key_dict = json.loads(gcp_key_str)

credentials = service_account.Credentials.from_service_account_info(key_dict)
bq_client = bigquery.Client(credentials= credentials, project=key_dict['project_id'])

print( f"✓ BigQuery client initialised " f"for project: {bq_client.project}" )

✓ BigQuery client initialised for project: datacareerapp


## 2.5 Test BigQuery Connection

In [ ]:
connection_test_query = """
SELECT
  CURRENT_TIMESTAMP() AS current_timestamp
"""
connection_test_df = (
    bq_client
    .query(connection_test_query)
    .to_dataframe()
)

display(connection_test_df)
print("✓ BigQuery connection successful.")

,current_timestamp
0,2026-08-31 05:20:17.848750+00:00


✓ BigQuery connection successful.


## 2.6 Validate Source and Target Tables

In [ ]:
def check_bigquery_table(client, table_id):
    """
    Check whether a BigQuery table/view can be accessed.

    Returns
    -------
    dict
        Validation result containing table id, existence,
        table type and row count where available.
    """

    try:
        table = client.get_table(table_id)

        return {
            "table_id": table_id,
            "accessible": True,
            "table_type": table.table_type,
            "num_rows": table.num_rows
        }

    except Exception as exc:

        return {
            "table_id": table_id,
            "accessible": False,
            "table_type": None,
            "num_rows": None,
            "error": str(exc)
        }


bigquery_table_checks = [
    check_bigquery_table(
        bq_client,
        SOURCE_TABLE_ID
    ),

    check_bigquery_table(
        bq_client,
        TARGET_TABLE_ID
    )
]

bigquery_table_checks_df = pd.DataFrame(
    bigquery_table_checks
)

display(bigquery_table_checks_df)

# Validate that both are available:

assert bigquery_table_checks_df["accessible"].all(), (
    "Source or target BigQuery object could not be accessed."
)

print("✓ Source and target BigQuery objects are accessible.")

,table_id,accessible,table_type,num_rows
0,datacareerapp.indeed_jobs.daily_scrape_dev,True,TABLE,20
1,datacareerapp.jobboard_prod_db.vw_dca_jobboard,True,TABLE,7250


✓ Source and target BigQuery objects are accessible.


## 2.7 Configure Secure Gemini API Access

In [ ]:
try:
    GEMINI_KEY = userdata.get("GOOGLE_API_KEY")

except Exception:
    GEMINI_KEY = None


assert GEMINI_KEY, (
    "GEMINI_KEY was not found. "
    "Add it to Google Colab Secrets before running "
    "AI enrichment sections."
)

gemini_client = genai.Client(
    api_key=GEMINI_KEY
)

print("✓ Gemini client initialised.")

✓ Gemini client initialised.


## 2.8 Define AI Model Configuration

In [ ]:
DEFAULT_AI_MODEL = "gemini-2.5-flash-lite"

print( f"Default AI model: {DEFAULT_AI_MODEL}" )

Default AI model: gemini-2.5-flash-lite


## 2.9 Define Uploaded File Paths

The notebook requires three project-supporting files:

field-mapping workbook,
AI prompt library,
GCCSA 2026 shapefile archive.

In [ ]:
PROMPTS_FILE = Path( "/content/indeed_stage2_prompts.json" )
GCCSA_SHAPEFILE_ZIP = Path( "/content/GCCSA_2026_AUST_SHP_GDA2020 (1).zip" )
PROJECT_FILES = {
    "Prompt Library": PROMPTS_FILE,
    "GCCSA Shapefile": GCCSA_SHAPEFILE_ZIP
}

## 2.10 Define the Approved 20-Field Mapping

In [ ]:
FIELD_MAPPING = {
    "jobKey": "id",
    "api_ini_dt_time": "api_ini_dt_time",
    "api_name": "api_name",
    "date": "posted_date",
    "title": "job_title",
    "company": "company_name",
    "city": "city",
    "state": "state",
    "location": "location",
    "description": "job_description",
    "jobUrl": "url",
    "source": "source",
    "industry": "industry",
    "function": "function",
    "top_tech_skills": "top_tech_skills",
    "exp_level": "exp_level",
    "pr_citizenship_req": "pr_citizenship_req",
    "sec_clearance": "sec_clearance",
    "isRemote": "location_type",
    "occupation": "role_cat"
}

Validate the mapping count

In [ ]:
assert len(FIELD_MAPPING) == 20, (
    f"Expected exactly 20 mappings, found {len(FIELD_MAPPING)}."
)

print("Field mapping contains exactly 20 fields.")

Field mapping contains exactly 20 fields.


Define final target schema

In [ ]:
TARGET_FIELDS = [
    "id",
    "api_ini_dt_time",
    "api_name",
    "posted_date",
    "job_title",
    "company_name",
    "city",
    "state",
    "location",
    "job_description",
    "url",
    "source",
    "industry",
    "function",
    "top_tech_skills",
    "exp_level",
    "pr_citizenship_req",
    "sec_clearance",
    "location_type",
    "role_cat"
]

Validate target schema

In [ ]:
assert len(TARGET_FIELDS) == 20, (
    f"Expected 20 target fields, found {len(TARGET_FIELDS)}."
)

assert len(set(TARGET_FIELDS)) == 20, (
    "Duplicate target field names detected."
)

print("Target schema contains exactly 20 unique fields.")

Target schema contains exactly 20 unique fields.


**Define Field Processing Categories**

The mapped fields can be grouped by how they are populated in the pipeline.

In [ ]:
DIRECT_OR_TRANSFORMED_FIELDS = [
    "id",
    "api_ini_dt_time",
    "api_name",
    "posted_date",
    "job_title",
    "company_name",
    "city",
    "state",
    "location",
    "job_description",
    "url",
    "source"
]

AI_ENRICHED_FIELDS = [
    "industry",
    "function",
    "top_tech_skills",
    "exp_level",
    "pr_citizenship_req",
    "sec_clearance",
    "role_cat"
]

DERIVED_FIELDS = [
    "location_type"
]

Validate That All 20 Fields Are Accounted For

In [ ]:
ALL_DEFINED_FIELDS = (
    DIRECT_OR_TRANSFORMED_FIELDS
    + AI_ENRICHED_FIELDS
    + DERIVED_FIELDS
)

assert set(ALL_DEFINED_FIELDS) == set(TARGET_FIELDS), (
    "One or more target fields are missing from the processing categories."
)

print("All 20 target fields are assigned to a processing category.")

All 20 target fields are assigned to a processing category.


Define the Location Type Output Contract

In [ ]:
VALID_LOCATION_TYPES = [
    "Metro",
    "Regional",
    "Remote"
]

LOCATION_TYPE_METHODS = {
    "case_1": "AI only",
    "case_2": "GCCSA shapefile first, then AI fallback"
}

Define the Mapping as a Review DataFrame

In [ ]:
field_mapping_df = pd.DataFrame(
    FIELD_MAPPING.items(),
    columns=["Source Field", "Target Field"]
)

display(field_mapping_df)

,Source Field,Target Field
0,jobKey,id
1,api_ini_dt_time,api_ini_dt_time
2,api_name,api_name
3,date,posted_date
4,title,job_title
5,company,company_name
6,city,city
7,state,state
8,location,location
9,description,job_description


## 2.11 Load AI Prompt Configuration

In [ ]:
with open(PROMPTS_FILE, "r", encoding="utf-8" ) as file:
  prompt_library = json.load(file)

print( f"✓ Loaded {len(prompt_library)} prompt definitions." )

for prompt_name in prompt_library:
  print(f" - {prompt_name}")

✓ Loaded 8 prompt definitions.
 - top_tech_skills
 - top_tech_skills_validation
 - role_cat
 - function
 - industry
 - exp_level
 - pr_citizenship_req
 - sec_clearance


## 2.12 Extract the GCCSA Shapefile and locate it

In [ ]:
GCCSA_EXTRACT_DIR = Path( "/content/gccsa_2026" )
GCCSA_EXTRACT_DIR.mkdir( parents=True, exist_ok=True )

with zipfile.ZipFile( GCCSA_SHAPEFILE_ZIP, "r" ) as zip_ref:
  zip_ref.extractall( GCCSA_EXTRACT_DIR )

print( f"✓ GCCSA archive extracted to: " f"{GCCSA_EXTRACT_DIR}" )

✓ GCCSA archive extracted to: /content/gccsa_2026


Locate shapefile

In [ ]:
shapefile_candidates = list(
    GCCSA_EXTRACT_DIR.rglob("*.shp")
)

assert len(shapefile_candidates) > 0, (
    "No .shp file was found inside the GCCSA archive."
)

print(
    f"Shapefiles found: "
    f"{len(shapefile_candidates)}"
)

for shp in shapefile_candidates:
    print(shp)

#Select the GCCSA shapefile.

GCCSA_SHP_PATH = shapefile_candidates[0]

print(
    f"\nSelected shapefile:\n"
    f"{GCCSA_SHP_PATH}"
)

Shapefiles found: 1
/content/gccsa_2026/GCCSA_2026_AUST_GDA2020.shp

Selected shapefile:
/content/gccsa_2026/GCCSA_2026_AUST_GDA2020.shp


## 2.13 Load GCCSA Spatial Data

In [ ]:
gccsa_gdf = gpd.read_file(
    GCCSA_SHP_PATH
)

print(
    f"GCCSA records: {len(gccsa_gdf):,}"
)

print(
    f"Coordinate reference system: "
    f"{gccsa_gdf.crs}"
)

display(
    gccsa_gdf.head()
)

GCCSA records: 35
Coordinate reference system: EPSG:7844


,GCC_CODE26,GCC_NAME26,CHG_FLAG26,CHG_LBL26,STE_CODE26,STE_NAME26,AUS_CODE26,AUS_NAME26,AREASQKM26,geometry
0,1GSYD,Greater Sydney,0,No change,1,New South Wales,AUS,Australia,12368.7250,"MULTIPOLYGON (((151.31497 -33.55578, 151.31496..."
1,1RNSW,Rest of NSW,0,No change,1,New South Wales,AUS,Australia,788428.9823,"MULTIPOLYGON (((153.57106 -28.87381, 153.57106..."
2,19499,No usual address (NSW),0,No change,1,New South Wales,AUS,Australia,NaN,None
3,19799,Migratory - Offshore - Shipping (NSW),0,No change,1,New South Wales,AUS,Australia,NaN,None
4,2GMEL,Greater Melbourne,0,No change,2,Victoria,AUS,Australia,9992.6078,"MULTIPOLYGON (((144.73142 -38.35024, 144.73148..."


## 2.14 Preserve the Original Spatial CRS

In [ ]:
GCCSA_ORIGINAL_CRS = gccsa_gdf.crs

assert GCCSA_ORIGINAL_CRS is not None, (
    "GCCSA shapefile has no defined CRS." )

print( f"✓ GCCSA CRS detected: " f"{GCCSA_ORIGINAL_CRS}" )

✓ GCCSA CRS detected: EPSG:7844


When latitude and longitude points are created later, they will first be created in:

POINT_INPUT_CRS = "EPSG:4326"

and then converted to the shapefile CRS before the spatial join.

This avoids incorrect spatial classifications caused by mismatched coordinate systems.

In [ ]:
POINT_INPUT_CRS = "EPSG:4326"

print( f"Input coordinate CRS: " f"{POINT_INPUT_CRS}" )

Input coordinate CRS: EPSG:4326


## 2.15 Configure Logging

In [ ]:
PIPELINE_NAME = "indeed_stage2_pipeline"

logger = logging.getLogger(
    PIPELINE_NAME
)

logger.setLevel(
    logging.INFO
)

if not logger.handlers:

    console_handler = logging.StreamHandler()

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    console_handler.setFormatter(
        formatter
    )

    logger.addHandler(
        console_handler
    )


logger.info(
    "Indeed pipeline environment initialised."
)

2026-08-31 05:20:25,197 | INFO | Indeed pipeline environment initialised.
INFO:indeed_stage2_pipeline:Indeed pipeline environment initialised.


## 2.16 Create General Timing Utility

All test cases must be measured consistently.

Create a reusable execution timer.

In [ ]:
# ============================================================
# SECTION 2.16 — EXECUTION TIMER
# ============================================================

@contextmanager
def execution_timer():
    """
    Measure wall-clock execution time.

    Usage
    -----
    with execution_timer() as timer:
        function()

    print(timer["seconds"])
    """

    result = {}

    start = time.perf_counter()

    try:
        yield result

    finally:
        end = time.perf_counter()

        result["seconds"] = (
            end - start
        )

with execution_timer() as timer:
    time.sleep(0.25)

print(
    f"Execution time: "
    f"{timer['seconds']:.4f} seconds"
)

Execution time: 0.2501 seconds


## 2.17 Create Memory Measurement Utility

In [ ]:
# ============================================================
# SECTION 2.17 — MEMORY MONITORING
# ============================================================

def get_memory_usage_mb():
    """
    Return memory currently used by the Python process
    in megabytes.
    """

    process = psutil.Process(
        os.getpid()
    )

    memory_bytes = (
        process
        .memory_info()
        .rss
    )

    return (
        memory_bytes /
        (1024 ** 2)
    )

#Test:

print(
    f"Current Python memory usage: "
    f"{get_memory_usage_mb():.2f} MB"
)

Current Python memory usage: 643.73 MB


## 2.18 Create Standard Experiment Benchmark Utility

Execution time and memory measurements should use the same calculation for all test cases.

In [ ]:
def benchmark_function( function, *args, **kwargs ):
  """ Run a function and capture basic execution metrics.
  Returns -------
  result Function output.
  metrics : dict Execution-time and memory statistics. """

  gc.collect()
  memory_before = get_memory_usage_mb()
  start_time = time.perf_counter()

  result = function( *args, **kwargs )
  end_time = time.perf_counter()
  memory_after = get_memory_usage_mb()

  metrics = {
      "execution_time_seconds": end_time - start_time,
      "memory_before_mb": memory_before,
      "memory_after_mb": memory_after,
      "memory_delta_mb": memory_after - memory_before }

  return result, metrics

## 2.19 Initialise Standard Metrics Schema

Define a common experiment-results structure.

Not every metric applies to every experiment.

For example:

AI token counts are not applicable to shapefile-only records.
precision/recall requires labelled validation data.
API-call counts are not applicable to deterministic deduplication.

Missing metrics will remain NaN rather than being artificially assigned zero.

In [ ]:
# ============================================================
# SECTION 2.19 — STANDARD EXPERIMENT METRICS
# ============================================================
EXPERIMENT_METRIC_COLUMNS = [
    "experiment_group",
    "test_case",
    "approach",
    "records_processed",

    # Quality
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "valid_output_rate",
    "unresolved_rate",

    # Performance
    "execution_time_seconds",
    "avg_time_per_record_seconds",
    "records_per_second",

    # Resources
    "memory_before_mb",
    "memory_after_mb",
    "memory_delta_mb",

    # AI usage
    "api_calls",
    "input_tokens",
    "output_tokens",
    "total_tokens",
    "estimated_cost",

    # Reliability
    "retry_count",
    "failure_count" ]

experiment_metrics_df = pd.DataFrame( columns=EXPERIMENT_METRIC_COLUMNS )
display( experiment_metrics_df )

,experiment_group,test_case,approach,records_processed,accuracy,precision,recall,f1_score,valid_output_rate,unresolved_rate,...,memory_before_mb,memory_after_mb,memory_delta_mb,api_calls,input_tokens,output_tokens,total_tokens,estimated_cost,retry_count,failure_count


## 2.20 Create Metric Recording Function

In [ ]:
# ============================================================
# SECTION 2.20 — RECORD EXPERIMENT RESULTS
# ============================================================

def record_experiment_metrics(
    metrics_df,
    **metrics
):
    """
    Append one experiment result to the common
    metrics dataframe.
    """

    row = {
        column: metrics.get(
            column,
            np.nan
        )
        for column
        in EXPERIMENT_METRIC_COLUMNS
    }

    return pd.concat(
        [
            metrics_df,
            pd.DataFrame([row])
        ],
        ignore_index=True
    )

#Example:

example_metrics = experiment_metrics_df.copy()

example_metrics = record_experiment_metrics(
    example_metrics,
    experiment_group="environment_test",
    test_case="example",
    approach="example_function",
    records_processed=100,
    execution_time_seconds=1.5
)

display(
    example_metrics
)

#The example dataframe does not need to be retained.

del example_metrics

,experiment_group,test_case,approach,records_processed,accuracy,precision,recall,f1_score,valid_output_rate,unresolved_rate,...,memory_before_mb,memory_after_mb,memory_delta_mb,api_calls,input_tokens,output_tokens,total_tokens,estimated_cost,retry_count,failure_count
0,environment_test,example,example_function,100,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2.21 Define Null-Like Values

The pipeline will encounter multiple representations of missing data.

Define them centrally.

This configuration will later be used carefully during cleaning.

It does not mean every occurrence of words such as "unknown" inside free-text descriptions will be removed. It applies only when the entire field represents a missing value.

In [ ]:
# ============================================================
# SECTION 2.21 — NULL CONFIGURATION
# ============================================================

NULL_STRINGS = {
    "",
    " ",
    "null",
    "none",
    "nan",
    "n/a",
    "na",
    "not available",
    "unknown"
}

## 2.22 Define Reproducibility Settings

The same seed will be used when drawing benchmark samples so that competing experimental approaches are evaluated using identical records.

In [ ]:
# ============================================================
# SECTION 2.22 — REPRODUCIBILITY
# ============================================================

RANDOM_STATE = 42

np.random.seed(
    RANDOM_STATE
)

print(
    f"✓ Random seed set to "
    f"{RANDOM_STATE}"
)

✓ Random seed set to 42


## 2.23 Environment Validation

Perform a consolidated check before continuing.

In [ ]:
# ============================================================
# SECTION 2.23 — ENVIRONMENT VALIDATION
# ============================================================

environment_checks = {
    "BigQuery client":
        bq_client is not None,

    "Source table":
        bigquery_table_checks_df.loc[
            bigquery_table_checks_df["table_id"]
            == SOURCE_TABLE_ID,
            "accessible"
        ].all(),

    "Target table":
        bigquery_table_checks_df.loc[
            bigquery_table_checks_df["table_id"]
            == TARGET_TABLE_ID,
            "accessible"
        ].all(),

    "Gemini client":
        gemini_client is not None,

    "20 field mappings":
        len(FIELD_MAPPING) == 20,

    "Prompt library":
        len(prompt_library) > 0,

    "GCCSA archive":
        GCCSA_SHAPEFILE_ZIP.exists(),

    "GCCSA shapefile":
        GCCSA_SHP_PATH.exists(),

    "GCCSA dataframe":
        len(gccsa_gdf) > 0,

    "GCCSA CRS":
        gccsa_gdf.crs is not None
}

environment_validation_df = pd.DataFrame(
    environment_checks.items(),
    columns=[
        "Environment Check",
        "Passed"
    ]
)

display(
    environment_validation_df
)

#Final assertion:

assert environment_validation_df["Passed"].all(), (
    "One or more environment checks failed. "
    "Resolve them before continuing."
)

print(
    "✓ SECTION 2 PASSED — "
    "Environment is ready for pipeline analysis."
)


,Environment Check,Passed
0,BigQuery client,True
1,Source table,True
2,Target table,True
3,Gemini client,True
4,20 field mappings,True
5,Prompt library,True
6,GCCSA archive,True
7,GCCSA shapefile,True
8,GCCSA dataframe,True
9,GCCSA CRS,True


✓ SECTION 2 PASSED — Environment is ready for pipeline analysis.


# 3. Source and Target Schema Analysis

This section inspects the schemas of the BigQuery source table daily_scrape_dev and the production job-board view/table vw_dca_jobboard.

The purpose is to understand:

available source fields,
target fields,
BigQuery data types,
nullability,
nested structures,
schema differences,
and which of the approved 20 target fields can be sourced directly versus derived or AI-enriched.

No cleaning, deduplication, transformation, or AI enrichment is performed in this section.

## 3.1 Retrieve Source Table Schema

In [ ]:
source_table = bq_client.get_table(SOURCE_TABLE_ID)
print(f"Source table: {SOURCE_TABLE_ID}")
print(f"Table type: {source_table.table_type}")
print(f"Rows: {source_table.num_rows:,}")
print(f"Columns: {len(source_table.schema)}")

Source table: datacareerapp.indeed_jobs.daily_scrape_dev
Table type: TABLE
Rows: 20
Columns: 44


## 3.2 Convert Source Schema to a DataFrame

Create a structured schema table for easier analysis.

In [ ]:
source_schema_df = pd.DataFrame(
    [
        {
            "field_name": field.name,
            "data_type": field.field_type,
            "mode": field.mode,
            "description": field.description
        }
        for field in source_table.schema
    ]
)
display(source_schema_df)

,field_name,data_type,mode,description
0,companyIndustry,STRING,NULLABLE,None
1,companyCeo,RECORD,NULLABLE,None
2,companyLinks,RECORD,NULLABLE,None
3,companyFounded,RECORD,NULLABLE,None
4,companyDescription,STRING,NULLABLE,None
5,companyNumEmployees,STRING,NULLABLE,None
6,companyAddresses,STRING,REPEATED,None
7,jobUrl,STRING,NULLABLE,None
8,language,STRING,NULLABLE,None
9,locale,STRING,NULLABLE,None


## 3.3 Retrieve Target Schema

Inspect the existing vw_dca_jobboard schema.

In [ ]:
target_table = bq_client.get_table(TARGET_TABLE_ID)
print(f"Target table: {TARGET_TABLE_ID}")
print(f"Table type: {target_table.table_type}")
print(f"Rows: {target_table.num_rows:,}")
print(f"Columns: {len(target_table.schema)}")

Target table: datacareerapp.jobboard_prod_db.vw_dca_jobboard
Table type: TABLE
Rows: 7,250
Columns: 20


## 3.4 Convert Target Schema to a DataFrame

In [ ]:
target_schema_df = pd.DataFrame(
    [
        {
            "field_name": field.name,
            "data_type": field.field_type,
            "mode": field.mode,
            "description": field.description
        }
        for field in target_table.schema
    ]
)
display(target_schema_df)

,field_name,data_type,mode,description
0,id,STRING,NULLABLE,None
1,api_ini_dt_time,TIMESTAMP,NULLABLE,None
2,api_name,STRING,NULLABLE,None
3,posted_date,DATE,NULLABLE,None
4,job_title,STRING,NULLABLE,None
5,company_name,STRING,NULLABLE,None
6,city,STRING,NULLABLE,None
7,state,STRING,NULLABLE,None
8,location,STRING,NULLABLE,None
9,job_description,STRING,NULLABLE,None


## 3.5 Compare Source and Target Column Counts

In [ ]:
schema_summary_df = pd.DataFrame(
    {
        "Dataset": [
            "daily_scrape_dev",
            "vw_dca_jobboard"
        ],
        "Column Count": [
            len(source_schema_df),
            len(target_schema_df)
        ]
    }
)
display(schema_summary_df)

,Dataset,Column Count
0,daily_scrape_dev,44
1,vw_dca_jobboard,20


## 3.6 Inspect Source Field Names

This is useful for checking whether the expected source fields exist before applying the approved 20-field mapping.

In [ ]:
source_columns = source_schema_df["field_name"].tolist()
print("Source fields:\n")

for column in source_columns:
    print(column)

Source fields:

companyIndustry
companyCeo
companyLinks
companyFounded
companyDescription
companyNumEmployees
companyAddresses
jobUrl
language
locale
isRemote
source
datePublished
hiringDemand
expired
title
shiftAndSchedule
emails
workingSystem
companyLogoUrl
socialInsurance
companyUrl
contacts
descriptionText
shifts
attributes
companyName
occupation
postedToday
applyUrl
benefits
companyBriefDescription
salary
companyHeaderUrl
descriptionHtml
age
location
requirements
jobType
companyRevenue
jobKey
numOfCandidates
rating
scrapingInfo


## 3.7 Inspect Target Field Names

In [ ]:
target_columns_existing = target_schema_df["field_name"].tolist()
print("Target fields:\n")

for column in target_columns_existing:
    print(column)

Target fields:

id
api_ini_dt_time
api_name
posted_date
job_title
company_name
city
state
location
job_description
url
source
top_tech_skills
function
industry
exp_level
pr_citizenship_req
sec_clearance
location_type
role_cat


## 3.8 Check the Approved 20 Source Fields Against the Source Schema

The approved mapping was defined directly in Section 2.

Check which required source fields exist in daily_scrape_dev.

In [ ]:
required_source_fields = list(FIELD_MAPPING.keys())
source_field_check_df = pd.DataFrame(
    {
        "source_field": required_source_fields,
        "exists_in_source": [
            field in source_columns
            for field in required_source_fields
        ]
    }
)
display(source_field_check_df)

,source_field,exists_in_source
0,jobKey,True
1,api_ini_dt_time,False
2,api_name,False
3,date,False
4,title,True
5,company,False
6,city,False
7,state,False
8,location,True
9,description,False


## 3.9 Identify Missing Source Fields

Some fields may legitimately not exist in the raw Indeed source because they are expected to be:

* generated,
* AI-enriched,
* or derived later in the pipeline.

Therefore, a missing source field does not automatically represent an error.

In [ ]:
missing_source_fields = source_field_check_df.loc[
    ~source_field_check_df["exists_in_source"],
    "source_field"
].tolist()
print("Missing mapped source fields:")

if missing_source_fields:
    for field in missing_source_fields:
        print(f"- {field}")
else:
    print("None")

Missing mapped source fields:
- api_ini_dt_time
- api_name
- date
- company
- city
- state
- description
- industry
- function
- top_tech_skills
- exp_level
- pr_citizenship_req
- sec_clearance


## 3.10 Check Approved Target Fields Against Existing Target Schema

In [ ]:
target_field_check_df = pd.DataFrame(
    {
        "target_field": TARGET_FIELDS,
        "exists_in_target": [
            field in target_columns_existing
            for field in TARGET_FIELDS
        ]
    }
)
display(target_field_check_df)

,target_field,exists_in_target
0,id,True
1,api_ini_dt_time,True
2,api_name,True
3,posted_date,True
4,job_title,True
5,company_name,True
6,city,True
7,state,True
8,location,True
9,job_description,True


## 3.11 Identify Missing Target Fields

This check helps identify whether the production target schema has changed since the mapping was defined.

In [ ]:
missing_target_fields = target_field_check_df.loc[
    ~target_field_check_df["exists_in_target"],
    "target_field"
].tolist()
print("Approved target fields missing from vw_dca_jobboard:")

if missing_target_fields:
    for field in missing_target_fields:
        print(f"- {field}")
else:
    print("None")

Approved target fields missing from vw_dca_jobboard:
None


## 3.12 Compare Source and Target Data Types

Join the approved source-to-target mapping with the actual BigQuery schemas.

In [ ]:
mapping_schema_comparison_df = pd.DataFrame(
    FIELD_MAPPING.items(),
    columns=[
        "source_field",
        "target_field"
    ]
)
mapping_schema_comparison_df = mapping_schema_comparison_df.merge(
    source_schema_df[
        [
            "field_name",
            "data_type",
            "mode"
        ]
    ].rename(
        columns={
            "field_name": "source_field",
            "data_type": "source_data_type",
            "mode": "source_mode"
        }
    ),
    on="source_field",
    how="left"
)
mapping_schema_comparison_df = mapping_schema_comparison_df.merge(
    target_schema_df[
        [
            "field_name",
            "data_type",
            "mode"
        ]
    ].rename(
        columns={
            "field_name": "target_field",
            "data_type": "target_data_type",
            "mode": "target_mode"
        }
    ),
    on="target_field",
    how="left"
)
display(mapping_schema_comparison_df)

,source_field,target_field,source_data_type,source_mode,target_data_type,target_mode
0,jobKey,id,STRING,NULLABLE,STRING,NULLABLE
1,api_ini_dt_time,api_ini_dt_time,NaN,NaN,TIMESTAMP,NULLABLE
2,api_name,api_name,NaN,NaN,STRING,NULLABLE
3,date,posted_date,NaN,NaN,DATE,NULLABLE
4,title,job_title,STRING,NULLABLE,STRING,NULLABLE
5,company,company_name,NaN,NaN,STRING,NULLABLE
6,city,city,NaN,NaN,STRING,NULLABLE
7,state,state,NaN,NaN,STRING,NULLABLE
8,location,location,RECORD,NULLABLE,STRING,NULLABLE
9,description,job_description,NaN,NaN,STRING,NULLABLE


## 3.13 Flag Data-Type Differences

In [ ]:
mapping_schema_comparison_df["type_match"] = (
    mapping_schema_comparison_df["source_data_type"]
    == mapping_schema_comparison_df["target_data_type"]
)
display(
    mapping_schema_comparison_df[
        [
            "source_field",
            "source_data_type",
            "target_field",
            "target_data_type",
            "type_match"
        ]
    ]
)

,source_field,source_data_type,target_field,target_data_type,type_match
0,jobKey,STRING,id,STRING,True
1,api_ini_dt_time,NaN,api_ini_dt_time,TIMESTAMP,False
2,api_name,NaN,api_name,STRING,False
3,date,NaN,posted_date,DATE,False
4,title,STRING,job_title,STRING,True
5,company,NaN,company_name,STRING,False
6,city,NaN,city,STRING,False
7,state,NaN,state,STRING,False
8,location,RECORD,location,STRING,False
9,description,NaN,job_description,STRING,False


## 3.14 Show Fields Requiring Data-Type Review

Do not automatically treat every mismatch as a problem.

For example:

source strings may need conversion to timestamps,
nested values may need flattening,
generated target fields may not have a direct source type,
AI fields may be populated later as strings or booleans.

These cases will be handled explicitly during transformation.

In [ ]:
type_mismatch_df = mapping_schema_comparison_df[
    mapping_schema_comparison_df["type_match"] == False
].copy()
display(type_mismatch_df)

,source_field,target_field,source_data_type,source_mode,target_data_type,target_mode,type_match
1,api_ini_dt_time,api_ini_dt_time,NaN,NaN,TIMESTAMP,NULLABLE,False
2,api_name,api_name,NaN,NaN,STRING,NULLABLE,False
3,date,posted_date,NaN,NaN,DATE,NULLABLE,False
5,company,company_name,NaN,NaN,STRING,NULLABLE,False
6,city,city,NaN,NaN,STRING,NULLABLE,False
7,state,state,NaN,NaN,STRING,NULLABLE,False
8,location,location,RECORD,NULLABLE,STRING,NULLABLE,False
9,description,job_description,NaN,NaN,STRING,NULLABLE,False
12,industry,industry,NaN,NaN,STRING,NULLABLE,False
13,function,function,NaN,NaN,STRING,NULLABLE,False


## 3.15 Detect Nested or Repeated Source Fields

Indeed source data contains nested RECORD or repeated fields.

Identify them before extraction.

If the result is empty, no top-level mapped source field is directly represented as a nested/repeated field.

In [ ]:
nested_source_fields_df = source_schema_df[
    (
        source_schema_df["data_type"] == "RECORD"
    )
    |
    (
        source_schema_df["mode"] == "REPEATED"
    )
].copy()
display(nested_source_fields_df)

,field_name,data_type,mode,description
1,companyCeo,RECORD,NULLABLE,None
2,companyLinks,RECORD,NULLABLE,None
3,companyFounded,RECORD,NULLABLE,None
6,companyAddresses,STRING,REPEATED,None
13,hiringDemand,RECORD,NULLABLE,None
16,shiftAndSchedule,STRING,REPEATED,None
17,emails,STRING,REPEATED,None
18,workingSystem,STRING,REPEATED,None
20,socialInsurance,STRING,REPEATED,None
22,contacts,RECORD,REPEATED,None


## 3.16 Recursively Inspect Nested Schema Fields

Use a helper function to inspect nested BigQuery structures in more detail.

In [ ]:
def flatten_bigquery_schema(schema, parent=""):
    rows = []

    for field in schema:
        full_name = (
            f"{parent}.{field.name}"
            if parent
            else field.name
        )

        rows.append(
            {
                "field_name": full_name,
                "data_type": field.field_type,
                "mode": field.mode
            }
        )

        if field.fields:
            rows.extend(
                flatten_bigquery_schema(
                    field.fields,
                    parent=full_name
                )
            )

    return rows
source_schema_flat_df = pd.DataFrame(
    flatten_bigquery_schema(
        source_table.schema
    )
)
display(source_schema_flat_df)

,field_name,data_type,mode
0,companyIndustry,STRING,NULLABLE
1,companyCeo,RECORD,NULLABLE
2,companyCeo.startDate,DATE,NULLABLE
3,companyCeo.photoUrl,STRING,NULLABLE
4,companyCeo.name,STRING,NULLABLE
...,...,...,...
76,rating.count,INTEGER,NULLABLE
77,rating.rating,FLOAT,NULLABLE
78,scrapingInfo,RECORD,NULLABLE
79,scrapingInfo.index,INTEGER,NULLABLE


## 3.17 Sample Source Records

Inspect a small sample without loading the full source dataset.

In [ ]:
source_sample_query = f"""
SELECT *
FROM `{SOURCE_TABLE_ID}`
LIMIT 5
"""
source_sample_df = (
    bq_client
    .query(source_sample_query)
    .to_dataframe()
)
display(source_sample_df)

,companyIndustry,companyCeo,companyLinks,companyFounded,companyDescription,companyNumEmployees,companyAddresses,jobUrl,language,locale,...,descriptionHtml,age,location,requirements,jobType,companyRevenue,jobKey,numOfCandidates,rating,scrapingInfo
0,None,"{'startDate': None, 'photoUrl': None, 'name': ...","{'corporateWebsite': None, 'customLinks': [], ...",None,None,None,[],https://au.indeed.com/viewjob?jk=9650a124388567d9,en,au,...,"<div><h2 class=""jobSectionHeader""><b>We&rsquo;...",13 hours ago,"{'latitude': -37.814, 'formattedAddressShort':...",[],[Full-time],None,9650a124388567d9,None,"{'count': 5, 'rating': 4.6}","{'index': 8, 'page': 3}"
1,None,"{'startDate': None, 'photoUrl': None, 'name': ...","{'corporateWebsite': None, 'customLinks': [], ...",None,None,None,[],https://au.indeed.com/viewjob?jk=9c2916ac1bdaf7e2,en,au,...,<div><p><b>This position commences 1 January 2...,6 hours ago,"{'latitude': -37.818, 'formattedAddressShort':...",[],[Full-time],None,9c2916ac1bdaf7e2,None,"{'count': 91, 'rating': 3.9}","{'index': 6, 'page': 1}"
2,GOVERNMENT,"{'startDate': None, 'photoUrl': None, 'name': ...","{'corporateWebsite': 'https://www.csiro.au', '...",{'year': 1926},"At CSIRO, we solve the greatest challenges thr...","1,001 to 5,000","[BLACK MOUNTAIN, AS]",https://au.indeed.com/viewjob?jk=1651d59366f4f3a1,en,au,...,<div></div><div><p><b>Acknowledgement of Count...,4 hours ago,"{'latitude': -27.46794, 'formattedAddressShort...",[],[Full-time],$100M to $500M (USD),1651d59366f4f3a1,None,"{'count': 71, 'rating': 4.0}","{'index': 10, 'page': 3}"
3,None,"{'startDate': None, 'photoUrl': 'https://d2q79...","{'corporateWebsite': 'https://www.anz.com.au',...",{'year': 1835},If reinventing the wheel is in your wheelhouse...,"10,000+",[Melbourne],https://au.indeed.com/viewjob?jk=10f4851d92195dbd,en,au,...,<div></div><div><div><div><div><div><h1 class=...,6 hours ago,"{'latitude': -37.814, 'formattedAddressShort':...",[],"[Permanent, Full-time]",more than $10B (USD),10f4851d92195dbd,None,"{'count': 1320, 'rating': 4.1}","{'index': 5, 'page': 1}"
4,None,"{'startDate': None, 'photoUrl': None, 'name': ...","{'corporateWebsite': None, 'customLinks': [], ...",None,None,None,[],https://au.indeed.com/viewjob?jk=1df3bf48f6eea6ca,en,au,...,Company Description\n<br>Showtime Consulting i...,14 hours ago,"{'latitude': -35.2772, 'formattedAddressShort'...",[],[],None,1df3bf48f6eea6ca,None,"{'count': 3, 'rating': 3.3}","{'index': 11, 'page': 2}"


## 3.18 Sample Target Records

In [ ]:
target_sample_query = f"""
SELECT *
FROM `{TARGET_TABLE_ID}`
LIMIT 5
"""
target_sample_df = (
    bq_client
    .query(target_sample_query)
    .to_dataframe()
)
display(target_sample_df)

,id,api_ini_dt_time,api_name,posted_date,job_title,company_name,city,state,location,job_description,url,source,top_tech_skills,function,industry,exp_level,pr_citizenship_req,sec_clearance,location_type,role_cat
0,de6ecb39-8190-42f3-aa67-0f5e4d065fa2,2026-08-18 13:00:51.832348+00:00,apify_websift_da,2026-08-18,Graduate Transport Planner (Bus Scheduling & O...,Swan Transit,New South,None,"{'area': None, 'displayLocation': 'New South W...","Support the planning, scheduling, and optimisa...",https://au.seek.com/job/94046089,Seek,"Microsoft Excel, Power BI","Transport Planning, Operations Management, Sch...","Public Transport, Logistics, Urban Mobility, T...",Entry-level,None mentioned,False,Major city,Data Analyst
1,8e675008-9bc0-4064-bb2f-3c29b8a8d752,2026-08-18 13:03:56.987856+00:00,brightdata_da,2026-08-18,Planning Analyst,MUFG,Greater Sydney Area,None,Greater Sydney Area,Do you want your voice heard and your actions ...,https://www.linkedin.com/jobs/view/planning-an...,LinkedIn,"Excel, PowerPoint, KPI management systems, Bal...","Business Planning, Project Management, Stakeho...","Banking, financial services",Associate,None mentioned,False,Major city,Data Analyst
2,ae80aa70-2d09-4275-928a-81f6b8a42ef4,2026-08-18 13:03:56.987856+00:00,brightdata_da,2026-08-18,Senior Investigations Analyst,Tabcorp,Greater Melbourne Area,None,Greater Melbourne Area,Your opportunity Looking for a role where your...,https://www.linkedin.com/jobs/view/senior-inve...,LinkedIn,Excel,"Investigations, Risk Management, Intelligence ...","Gambling, entertainment, technology",Senior,None mentioned,False,Major city,Data Analyst
3,23533ff4-2522-4818-80f3-3639a41322d0,2026-08-18 13:03:56.987856+00:00,brightdata_da,2026-08-18,Research and Support Consultant,"Department of Education, Western Australia",Greater Perth Area,None,Greater Perth Area,This is a permanent full-time position commenc...,https://www.linkedin.com/jobs/view/research-an...,LinkedIn,"GIS, ArcGIS","Research, Consulting, Policy Development, Proj...","Education, public sector",Senior,"Permanent Residency, Citizenship, Valid work visa",True,Major city,Data Analyst
4,447caf3f-3dec-4d2b-af77-667cee9f9d59,2026-08-18 13:03:56.987856+00:00,brightdata_da,2026-08-18,Performance Management Engineer,City of Melbourne,Greater Melbourne Area,None,Greater Melbourne Area,Synopsis Temporary Full-Time 4 August 2026 to ...,https://www.linkedin.com/jobs/view/performance...,LinkedIn,Microsoft Excel,"Contract Administration, Asset Management, Cus...","Public Services, Transportation, Engineering, ...",Associate,None mentioned,True,Major city,Data Analyst


## 3.19 Inspect Only the Approved Target Fields

The existing production table may contain fields outside the agreed project scope.

Restrict analysis to the approved 20 target fields that actually exist.

In [ ]:
existing_approved_target_fields = [
    field
    for field in TARGET_FIELDS
    if field in target_columns_existing
]
existing_approved_target_fields

['id',
 'api_ini_dt_time',
 'api_name',
 'posted_date',
 'job_title',
 'company_name',
 'city',
 'state',
 'location',
 'job_description',
 'url',
 'source',
 'industry',
 'function',
 'top_tech_skills',
 'exp_level',
 'pr_citizenship_req',
 'sec_clearance',
 'location_type',
 'role_cat']

## 3.20 Query Approved Target Fields Only

In [ ]:
approved_target_sql = ",\n".join(
    f"`{field}`"
    for field in existing_approved_target_fields
)
approved_target_sample_query = f"""
SELECT
    {approved_target_sql}
FROM `{TARGET_TABLE_ID}`
LIMIT 5
"""
approved_target_sample_df = (
    bq_client
    .query(approved_target_sample_query)
    .to_dataframe()
)
display(approved_target_sample_df)

,id,api_ini_dt_time,api_name,posted_date,job_title,company_name,city,state,location,job_description,url,source,industry,function,top_tech_skills,exp_level,pr_citizenship_req,sec_clearance,location_type,role_cat
0,ab2628fd-4e8c-41db-a4bf-d252d13fa158,2026-08-12 20:49:10+00:00,apify_indeed,2026-08-12,Lead Data Engineer,EPAM Systems,Remote,,"Remote, Australia",We are seeking a Lead Data Engineer to own end...,https://au.indeed.com/viewjob?jk=116aa9137a4c0eab,Indeed,Information Technology,"Data Engineering, Data Modeling, Data Architec...","Databricks, dbt, Confluence, Python, SQL",Senior,None mentioned,False,Regional/remote,Data Engineer
1,933ad3e1-4826-44ea-ae45-c80a04e205a4,2026-08-12 20:49:10+00:00,apify_indeed,2026-08-12,Senior/Staff Data Scientist (Optimisation),Rippling,Remote,,"Remote, Australia",About Gridsight Electricity grids are undergoi...,https://au.indeed.com/viewjob?jk=bb0a4c0304d6eff4,Indeed,"Electricity, Utilities, Renewable Energy","Optimisation, Software Engineering","Pyomo, CBC, Gurobi, CPLEX, OR-Tools, Python",Senior,None mentioned,False,Regional/remote,Data Engineer
2,2b2081bf-02c9-48cf-9573-560763b18122,2026-08-12 20:49:10+00:00,apify_indeed,2026-08-12,Senior/Staff Data Scientist (Power Flows),Rippling,Wollongong,NSW,"Wollongong, NSW, Australia",About Gridsight Electricity grids are undergoi...,https://au.indeed.com/viewjob?jk=b1f0d78bdc33963c,Indeed,"Energy, Utilities, Renewable Energy, Power Sys...","Electrical Engineering, Software Engineering, ...","OpenDSS, pandapower, lf-energy, Python",Senior,None mentioned,False,Major city,Data Engineer
3,981e3726-637a-4fc8-b7e6-d882e61f60d8,2026-08-12 20:49:10+00:00,apify_indeed,2026-08-12,"Associate Director People, Systems and Analytics",NSW Government,Sydney,NSW,"Sydney, NSW, Australia","Department of Creative Industries, Tourism, Ho...",https://au.indeed.com/viewjob?jk=abb57002fe6d48eb,Indeed,"Creative Industries, Tourism, Hospitality, Sport","Human Resources, HR Systems, Reporting, Leader...","SAP, Cornerstone, Power BI",Senior,None mentioned,False,Major city,Data Analyst
4,21c625a5-74db-42b0-812c-dcdc78867ecc,2026-08-12 20:49:10+00:00,apify_indeed,2026-08-12,Senior Data Analyst - Economics,NSW Government,Sydney,NSW,"Sydney, NSW, Australia","Department of Creative Industries, Tourism, Ho...",https://au.indeed.com/viewjob?jk=184fc52f9ae9b165,Indeed,"Creative industries, tourism, hospitality, spo...","Project Management, Stakeholder Management, Po...","Excel, Power BI, R, Stata, Python",Senior,None mentioned,False,Major city,Data Analyst


## 3.22 Determine Processing Method for Each Target Field

Use the field categories defined in Section 2 to make the schema analysis more informative.

In [ ]:
def get_processing_method(field):
    if field in DIRECT_OR_TRANSFORMED_FIELDS:
        return "Direct / Transformation"

    if field in AI_ENRICHED_FIELDS:
        return "AI Enrichment"

    if field in DERIVED_FIELDS:
        return "Derived"

    return "Unclassified"
mapping_schema_comparison_df[
    "processing_method"
] = mapping_schema_comparison_df[
    "target_field"
].apply(
    get_processing_method
)
display(
    mapping_schema_comparison_df[
        [
            "source_field",
            "source_data_type",
            "target_field",
            "target_data_type",
            "processing_method"
        ]
    ]
)

,source_field,source_data_type,target_field,target_data_type,processing_method
0,jobKey,STRING,id,STRING,Direct / Transformation
1,api_ini_dt_time,NaN,api_ini_dt_time,TIMESTAMP,Direct / Transformation
2,api_name,NaN,api_name,STRING,Direct / Transformation
3,date,NaN,posted_date,DATE,Direct / Transformation
4,title,STRING,job_title,STRING,Direct / Transformation
5,company,NaN,company_name,STRING,Direct / Transformation
6,city,NaN,city,STRING,Direct / Transformation
7,state,NaN,state,STRING,Direct / Transformation
8,location,RECORD,location,STRING,Direct / Transformation
9,description,NaN,job_description,STRING,Direct / Transformation


## 3.23 Schema Analysis Summary

Create a compact summary of the main schema findings.

In [ ]:
schema_analysis_summary = {
    "source_column_count":
        len(source_schema_df),

    "target_column_count":
        len(target_schema_df),

    "approved_mapping_count":
        len(FIELD_MAPPING),

    "missing_source_fields":
        len(missing_source_fields),

    "missing_target_fields":
        len(missing_target_fields),

    "source_nested_or_repeated_fields":
        len(nested_source_fields_df),

    "type_mismatches":
        int(
            (
                mapping_schema_comparison_df["type_match"]
                == False
            ).sum()
        )
}
schema_analysis_summary_df = pd.DataFrame(
    schema_analysis_summary.items(),
    columns=[
        "Metric",
        "Value"
    ]
)
display(schema_analysis_summary_df)

,Metric,Value
0,source_column_count,44
1,target_column_count,20
2,approved_mapping_count,20
3,missing_source_fields,13
4,missing_target_fields,0
5,source_nested_or_repeated_fields,20
6,type_mismatches,15


## 3.24 Validate the Approved 20-Field Scope

The schema inspection may reveal many additional fields in the raw source or production target. They are intentionally excluded unless they are required to derive one of the approved target fields.

In [ ]:
assert len(FIELD_MAPPING) == 20
assert len(TARGET_FIELDS) == 20
print(
    "✓ Schema analysis remains restricted "
    "to the approved 20-field pipeline."
)

✓ Schema analysis remains restricted to the approved 20-field pipeline.


## 3.25 Section 3 Outcome

At the end of this section, the notebook has:

retrieved the actual BigQuery schemas of daily_scrape_dev and vw_dca_jobboard;
documented source and target data types;
checked the availability of the approved mapped fields;
identified missing source or target fields;
compared source and target data types;
identified nested and repeated structures;
inspected representative source and target records;
classified each target field by processing method;
and confirmed that the analysis remains restricted to the approved 20-field pipeline.

No records have yet been cleaned, transformed, classified, enriched, or removed.

The next section should therefore perform source-data extraction and detailed pre-cleaning EDA, so that the actual quality problems in the incoming Indeed dataset are measured before any transformations are applied.

# 4. Exploratory Data Analysis

This section establishes the pre-cleaning baseline for the Indeed pipeline.

The purpose of the EDA is to understand the quality and structure of the current source records before any cleaning, deduplication, location classification, or AI enrichment is applied.

The analysis focuses specifically on the fields and conditions required for the later test cases:

source and target 48-hour working windows,
null and blank values,
jobKey duplication,
Company + Title + Location duplication,
date availability for Deduplication Case 2,
descriptionTextHash availability for Deduplication Case 3,
location-object and coordinate completeness for the location experiments,
raw isRemote values,
HTML and description quality,
and readiness of inputs required for AI enrichment.

The original project brief requires a 48-hour deduplication lookback and explicitly identifies cleaning, deduplication, AI enrichment, and staging quality checks as core pipeline requirements.

## 4.1 Load Working Datasets
### 4.1.1 Get the Indeed Table Modification Timestamp

In [ ]:
# Get source table metadata
source_table_metadata = bq_client.get_table(SOURCE_TABLE)

source_modified_time = source_table_metadata.modified

print("Indeed table last modified:")
print(source_modified_time)

Indeed table last modified:
2026-08-24 05:04:38.589000+00:00


### 4.1.2 Calculate the Dynamic 48-Hour Window

In [ ]:
lookback_start = (
    source_modified_time
    - timedelta(hours=48)
)
print(
    "Pipeline anchor time :",
    source_modified_time
)

print(
    "48-hour lookback start :",
    lookback_start
)

Pipeline anchor time : 2026-08-24 05:04:38.589000+00:00
48-hour lookback start : 2026-08-22 05:04:38.589000+00:00


**Compare real-time vs data-anchored window**

In [ ]:
real_time_anchor = pd.Timestamp.now(
    tz="UTC"
)

real_time_lookback = (
    real_time_anchor
    - timedelta(hours=48)
)

data_anchor = (
    source_modified_time
)

data_lookback = (
    data_anchor
    - timedelta(hours=48)
)

dedup_window_comparison_df = pd.DataFrame({
    "Approach": [
        "Real-time window",
        "Data-anchored window"
    ],

    "Anchor Time": [
        real_time_anchor,
        data_anchor
    ],

    "48h Lookback Start": [
        real_time_lookback,
        data_lookback
    ],

    "Suitable for Historical Batch": [
        "No",
        "Yes"
    ]
})

display(
    dedup_window_comparison_df
)

,Approach,Anchor Time,48h Lookback Start,Suitable for Historical Batch
0,Real-time window,2026-08-31 05:20:35.048436+00:00,2026-08-29 05:20:35.048436+00:00,No
1,Data-anchored window,2026-08-24 05:04:38.589000+00:00,2026-08-22 05:04:38.589000+00:00,Yes


### 4.1.3 Load the Source EDA Dataset


For exploratory analysis, load the latest 5,000 Indeed records that contain a valid datePublished value.

This is an EDA working sample, not the final production extraction logic.


In [ ]:
source_query = f"""
SELECT *
FROM `{SOURCE_TABLE}`
WHERE datePublished IS NOT NULL
ORDER BY datePublished DESC
"""
source_eda_df = (
    bq_client
    .query(source_query)
    .to_dataframe()
)
print(
    "Source rows:",
    len(source_eda_df)
)

print(
    "Source columns:",
    len(source_eda_df.columns)
)

Source rows: 20
Source columns: 44


### 4.1.5 Load the 48-Hour Target Dataset

The target records are restricted to the same 48-hour processing window using api_ini_dt_time.

In [ ]:
target_query = f"""
SELECT *
FROM `{TARGET_TABLE}`
WHERE api_ini_dt_time BETWEEN @lookback_start
                          AND @anchor_time
"""
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter(
            "lookback_start",
            "TIMESTAMP",
            lookback_start
        ),
        bigquery.ScalarQueryParameter(
            "anchor_time",
            "TIMESTAMP",
            source_modified_time
        )
    ]
)
target_48h_df = (
    bq_client
    .query(
        target_query,
        job_config=job_config
    )
    .to_dataframe()
)
print(
    f"Target records loaded: "
    f"{len(target_48h_df):,}"
)

print(
    f"Target columns loaded: "
    f"{len(target_48h_df.columns)}"
)

Target records loaded: 370
Target columns loaded: 20


## 4.2 Working Dataset Summary

In [ ]:
eda_window_summary = pd.DataFrame({
    "Metric": [
        "Indeed table last modified",
        "48-hour lookback start",
        "Source records loaded",
        "Source columns",
        "Target records loaded",
        "Target columns"
    ],

    "Value": [
        source_modified_time,
        lookback_start,
        len(source_eda_df),
        len(source_eda_df.columns),
        len(target_48h_df),
        len(target_48h_df.columns)
    ]
})
display(eda_window_summary)

,Metric,Value
0,Indeed table last modified,2026-08-24 05:04:38.589000+00:00
1,48-hour lookback start,2026-08-22 05:04:38.589000+00:00
2,Source records loaded,20
3,Source columns,44
4,Target records loaded,370
5,Target columns,20


## 4.3 Inspect Source Structure
### 4.3.1 Display Sample Records

In [ ]:
display( source_eda_df.head() )

,companyIndustry,companyCeo,companyLinks,companyFounded,companyDescription,companyNumEmployees,companyAddresses,jobUrl,language,locale,...,descriptionHtml,age,location,requirements,jobType,companyRevenue,jobKey,numOfCandidates,rating,scrapingInfo
0,None,"{'startDate': None, 'photoUrl': None, 'name': ...","{'corporateWebsite': None, 'customLinks': [], ...",None,None,None,[],https://au.indeed.com/viewjob?jk=9c2916ac1bdaf7e2,en,au,...,<div><p><b>This position commences 1 January 2...,6 hours ago,"{'latitude': -37.818, 'formattedAddressShort':...",[],[Full-time],None,9c2916ac1bdaf7e2,None,"{'count': 91, 'rating': 3.9}","{'index': 6, 'page': 1}"
1,GOVERNMENT,"{'startDate': None, 'photoUrl': None, 'name': ...","{'corporateWebsite': 'https://www.csiro.au', '...",{'year': 1926},"At CSIRO, we solve the greatest challenges thr...","1,001 to 5,000","[BLACK MOUNTAIN, AS]",https://au.indeed.com/viewjob?jk=1651d59366f4f3a1,en,au,...,<div></div><div><p><b>Acknowledgement of Count...,4 hours ago,"{'latitude': -27.46794, 'formattedAddressShort...",[],[Full-time],$100M to $500M (USD),1651d59366f4f3a1,None,"{'count': 71, 'rating': 4.0}","{'index': 10, 'page': 3}"
2,INTERNET_AND_SOFTWARE,"{'startDate': 2021-07-05 00:00:00, 'photoUrl':...",{'corporateWebsite': 'https://www.amazon.jobs'...,{'year': 1994},Amazon is guided by four principles: customer ...,"10,000+","[440 Terry Ave N, Seattle, WA, United States, ...",https://au.indeed.com/viewjob?jk=d140df231910c633,en,au,...,"<div><h2 class=""jobSectionHeader""><b>DESCRIPTI...",15 hours ago,"{'latitude': -33.86785, 'formattedAddressShort...",[],[Full-time],more than $10B (USD),d140df231910c633,None,"{'count': 55937, 'rating': 3.5}","{'index': 20, 'page': 1}"
3,None,"{'startDate': None, 'photoUrl': None, 'name': ...",{'corporateWebsite': 'http://www.wesfarmers.co...,{'year': 1914},Wesfarmers Limited is one of Australia's large...,"10,000+","[Perth, AS]",https://au.indeed.com/viewjob?jk=28e7a32d55dffa4f,en,au,...,<div><b>Work options:</b> Hybrid<p><br>\nWesfa...,3 hours ago,"{'latitude': -37.814, 'formattedAddressShort':...",[],"[Permanent, Full-time]",more than $10B (USD),28e7a32d55dffa4f,None,"{'count': 23, 'rating': 3.7}","{'index': 14, 'page': 1}"
4,None,"{'startDate': None, 'photoUrl': 'https://d2q79...","{'corporateWebsite': 'https://www.anz.com.au',...",{'year': 1835},If reinventing the wheel is in your wheelhouse...,"10,000+",[Melbourne],https://au.indeed.com/viewjob?jk=10f4851d92195dbd,en,au,...,<div></div><div><div><div><div><div><h1 class=...,6 hours ago,"{'latitude': -37.814, 'formattedAddressShort':...",[],"[Permanent, Full-time]",more than $10B (USD),10f4851d92195dbd,None,"{'count': 1320, 'rating': 4.1}","{'index': 5, 'page': 1}"


### 4.3.2 Inspect Source Data Types

In [ ]:
source_dtypes_df = pd.DataFrame({
    "field": source_eda_df.columns,
    "dtype": [
        str(dtype)
        for dtype in source_eda_df.dtypes
    ]
})
display(source_dtypes_df)

,field,dtype
0,companyIndustry,object
1,companyCeo,object
2,companyLinks,object
3,companyFounded,object
4,companyDescription,object
5,companyNumEmployees,object
6,companyAddresses,object
7,jobUrl,object
8,language,object
9,locale,object


### 4.3.3 Detect Nested Object Fields

The raw Indeed data contains nested structures such as location.

These should be identified before uniqueness or duplicate operations because dictionaries cannot safely be used directly in some Pandas operations.

In [ ]:
def contains_nested_object(series):

    return series.apply(
        lambda value:
            isinstance(
                value,
                (dict, list)
            )
    ).any()
nested_columns = [
    column
    for column in source_eda_df.columns
    if contains_nested_object(
        source_eda_df[column]
    )
]
print("Nested object columns:")

for column in nested_columns:
    print(column)

Nested object columns:
companyCeo
companyLinks
companyFounded
hiringDemand
salary
location
rating
scrapingInfo


## 4.4 Missing and Blank Values
### 4.4.1 Null-Value Analysis

In [ ]:
source_missing = pd.DataFrame({
    "null_count":
        source_eda_df
        .isna()
        .sum(),

    "null_pct":
        (
            source_eda_df
            .isna()
            .mean()
            * 100
        ).round(2)
})
source_missing = (
    source_missing
    .sort_values(
        "null_pct",
        ascending=False
    )
)
display(source_missing)

,null_count,null_pct
numOfCandidates,20,100.0
companyIndustry,13,65.0
companyHeaderUrl,11,55.0
companyBriefDescription,10,50.0
companyRevenue,7,35.0
companyFounded,6,30.0
companyDescription,5,25.0
companyLogoUrl,5,25.0
companyNumEmployees,4,20.0
companyLinks,0,0.0


### 4.4.2 Fields Directly Impacting the Pipeline

Focus first on fields that directly influence the final mapped pipeline or later experiments.

In [ ]:
core_source_fields = [
    "jobKey",
    "title",
    "companyName",
    "datePublished",
    "descriptionText",
    "descriptionTextHash",
    "jobUrl",
    "location",
    "isRemote"
]
core_source_fields_existing = [
    field
    for field in core_source_fields
    if field in source_eda_df.columns
]
display(
    source_missing.loc[
        source_missing.index.intersection(
            core_source_fields_existing
        )
    ]
)

,null_count,null_pct
isRemote,0,0.0
jobUrl,0,0.0
datePublished,0,0.0
descriptionText,0,0.0
title,0,0.0
companyName,0,0.0
location,0,0.0
jobKey,0,0.0


### 4.4.3 Blank-String Analysis

Null and blank values should be analysed separately.

In [ ]:
blank_results = []
for col in (
    source_eda_df
    .select_dtypes(
        include="object"
    )
    .columns
):

    if col in nested_columns:
        continue

    blank_count = (
        source_eda_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    blank_results.append({
        "field":
            col,

        "blank_count":
            blank_count,

        "blank_pct":
            round(
                blank_count
                / len(source_eda_df)
                * 100,
                2
            )
    })
source_blank_df = (
    pd.DataFrame(
        blank_results
    )
    .sort_values(
        "blank_pct",
        ascending=False
    )
)
display(source_blank_df)

,field,blank_count,blank_pct
31,numOfCandidates,20,100.0
0,companyIndustry,13,65.0
24,companyHeaderUrl,11,55.0
23,companyBriefDescription,10,50.0
29,companyRevenue,7,35.0
1,companyDescription,5,25.0
12,companyLogoUrl,5,25.0
2,companyNumEmployees,4,20.0
7,source,0,0.0
8,title,0,0.0


## 4.5 Preliminary Duplicate Investigation

This stage performs EDA only.

No records are removed yet.

Actual deduplication will occur later after the cleaning and standardisation stage.

The project requires duplicate analysis using:

* jobKey,
* Company + Job Title + Location,
* Company + Job Title + Location + Date,
* Company + Job Title + Location + descriptionTextHash.

### 4.5.1 Preliminary Stable-Field Duplicate Analysis

In [ ]:
duplicate_check_cols = [
    "jobKey",
    "companyName",
    "title",
    "jobUrl",
    "datePublished"
]
duplicate_check_cols = [
    col
    for col in duplicate_check_cols
    if col in source_eda_df.columns
]
duplicate_rows = (
    source_eda_df
    .duplicated(
        subset=duplicate_check_cols,
        keep=False
    )
)
print(
    "Potential duplicate rows:",
    duplicate_rows.sum()
)
display(
    source_eda_df.loc[
        duplicate_rows,
        duplicate_check_cols
    ].head(50)
)

Potential duplicate rows: 0


,jobKey,companyName,title,jobUrl,datePublished


## 4.6 jobKey Quality Analysis

jobKey is the Indeed source identifier and will later be used as the first source-level deduplication check.

In EDA, only inspect its completeness and duplication level. Do not remove any records here.



In [ ]:
jobkey_null_count = (
    source_eda_df["jobKey"]
    .isna()
    .sum()
)
jobkey_unique_count = (
    source_eda_df["jobKey"]
    .nunique(
        dropna=True
    )
)
jobkey_duplicate_rows = (
    source_eda_df["jobKey"]
    .notna()
    &
    source_eda_df["jobKey"]
    .duplicated(
        keep=False
    )
)
print(
    "Missing jobKey values:",
    jobkey_null_count
)

print(
    "Unique non-null jobKey values:",
    jobkey_unique_count
)

print(
    "Rows belonging to duplicated jobKey groups:",
    jobkey_duplicate_rows.sum()
)

Missing jobKey values: 0
Unique non-null jobKey values: 20
Rows belonging to duplicated jobKey groups: 0


### 4.6.1 Inspect Duplicate jobKey Records

In [ ]:
display(
    source_eda_df.loc[
        jobkey_duplicate_rows,
         [ "jobKey", "companyName", "title", "datePublished" ]
        ]
    .sort_values( "jobKey" )
    .head(50) )

,jobKey,companyName,title,datePublished


## 4.7 Deduplication Field Readiness

The later deduplication experiments require the following source information:

* company,
* job title,
* location,
* posting date,
* description hash.

Section 4 only evaluates whether these fields are sufficiently populated.

### 4.7.1 Define Required Source Deduplication Fields

In [ ]:
dedup_source_fields = [
    "companyName",
    "title",
    "location",
    "datePublished",
    "descriptionTextHash"
]
dedup_source_fields = [
    field
    for field in dedup_source_fields
    if field in source_eda_df.columns
]

### 4.7.2 Measure Source Deduplication Field Completeness

In [ ]:
dedup_source_readiness = []
for field in dedup_source_fields:

    null_count = (
        source_eda_df[field]
        .isna()
        .sum()
    )

    dedup_source_readiness.append({
        "field": field,
        "missing_count": null_count,
        "missing_pct": round(
            null_count
            / len(source_eda_df)
            * 100,
            2
        )
    })
dedup_source_readiness_df = pd.DataFrame(
    dedup_source_readiness
)
display(
    dedup_source_readiness_df
)

,field,missing_count,missing_pct
0,companyName,0,0.0
1,title,0,0.0
2,location,0,0.0
3,datePublished,0,0.0


## 4.8 Target Deduplication Readiness

The later cross-dataset deduplication stage compares daily_scrape_dev against the 48-hour vw_dca_jobboard dataset.

At this stage, only confirm that the required target fields exist and inspect their completeness.

### 4.8.1 Define Required Target Fields

In [ ]:
dedup_target_fields = [
    "company_name",
    "job_title",
    "location",
    "posted_date",
    "job_description"
    ]

### 4.8.2 Check Target Field Availability

In [ ]:
target_field_availability = pd.DataFrame({
    "field": dedup_target_fields,
    "exists": [
        field in target_48h_df.columns
        for field in dedup_target_fields
    ]
})
display(
    target_field_availability
)

,field,exists
0,company_name,True
1,job_title,True
2,location,True
3,posted_date,True
4,job_description,True


### 4.8.3 Measure Available Target Field Completeness

In [ ]:
existing_dedup_target_fields = [
    field
    for field in dedup_target_fields
    if field in target_48h_df.columns
]
target_dedup_readiness = []
for field in existing_dedup_target_fields:

    null_count = (
        target_48h_df[field]
        .isna()
        .sum()
    )

    target_dedup_readiness.append({
        "field": field,
        "missing_count": null_count,
        "missing_pct": round(
            null_count
            / len(target_48h_df)
            * 100,
            2
        ) if len(target_48h_df) > 0 else np.nan
    })
target_dedup_readiness_df = pd.DataFrame(
    target_dedup_readiness
)
display(
    target_dedup_readiness_df
)

,field,missing_count,missing_pct
0,company_name,0,0.0
1,job_title,0,0.0
2,location,0,0.0
3,posted_date,0,0.0
4,job_description,0,0.0


## 4.9 Location Data Analysis

The later location experiments require information from the nested source location object.

### 4.9.1 Measure Location Completeness

In [ ]:
location_eda = pd.DataFrame(
    { "metric":
     [
         "Total source rows",
         "Missing location object",
         "Missing formatted address",
         "Missing city", "Missing state",
         "Missing latitude", "Missing longitude" ],
       "count": [
           len(source_eda_df),
           source_eda_df["location"] .isna() .sum(),
           source_eda_df["location"] .apply(
               lambda x: not isinstance(x, dict)
                or not x.get("formattedAddressShort") ) .sum(),
           source_eda_df["location"] .apply(
               lambda x: not isinstance(x, dict)
               or not x.get("city") ) .sum(),
           source_eda_df["location"] .apply(
               lambda x: not isinstance(x, dict)
                or not x.get("state") ) .sum(),
           source_eda_df["location"] .apply(
               lambda x: not isinstance(x, dict)
               or x.get("latitude") is None ) .sum(),
           source_eda_df["location"] .apply(
               lambda x: not isinstance(x, dict)
                or x.get("longitude") is None ) .sum()
                ]
      })

### 4.9.2 Add Percentages

In [ ]:
location_eda["pct"] = (
    location_eda["count"]
    / len(source_eda_df)
    * 100
).round(2)
display(
    location_eda
)

,metric,count,pct
0,Total source rows,20,100.0
1,Missing location object,0,0.0
2,Missing formatted address,0,0.0
3,Missing city,0,0.0
4,Missing state,20,100.0
5,Missing latitude,0,0.0
6,Missing longitude,0,0.0


## 4.10 Coordinate Availability

The hybrid location approach will later depend on usable latitude and longitude values.

For EDA, only measure coverage.

In [ ]:
coordinate_available_mask = (
    source_eda_df["location"]
    .apply(
        lambda x:
        isinstance(x, dict)
        and x.get("latitude") is not None
        and x.get("longitude") is not None
    )
)
coordinate_coverage_pct = (
    coordinate_available_mask.mean()
    * 100
)
print(
    "Records with both latitude and longitude:",
    coordinate_available_mask.sum()
)

print(
    f"Coordinate coverage: "
    f"{coordinate_coverage_pct:.2f}%"
)

Records with both latitude and longitude: 20
Coordinate coverage: 100.00%


## 4.11 Inspect Common Raw Locations

In [ ]:
raw_location_counts = (
    source_eda_df["location"]
    .apply(
        lambda x:
        x.get("formattedAddressShort")
        if isinstance(x, dict)
        else None
    )
    .value_counts(
        dropna=False
    )
    .head(20)
)
display(
    raw_location_counts
    .to_frame(
        "record_count"
    )
)

,record_count
location,
Sydney NSW,7
Melbourne City Centre VIC,4
Melbourne VIC,4
Brisbane QLD,1
North Sydney NSW,1
Parramatta NSW,1
Clayton VIC,1
Canberra ACT,1


## 4.12 Inspect City Distribution

In [ ]:
city_counts = (
    source_eda_df["location"]
    .apply(
        lambda x:
        x.get("city")
        if isinstance(x, dict)
        else None
    )
    .value_counts(
        dropna=False
    )
    .head(20)
)
display(
    city_counts
    .to_frame(
        "record_count"
    )
)

,record_count
location,
Sydney,7
Melbourne City Centre,4
Melbourne,4
Brisbane,1
North Sydney,1
Parramatta,1
Clayton,1
Canberra,1


## 4.13 Inspect State Distribution

In [ ]:
state_counts = (
    source_eda_df["location"]
    .apply(
        lambda x:
        x.get("state")
        if isinstance(x, dict)
        else None
    )
    .value_counts(
        dropna=False
    )
)
display(
    state_counts
    .to_frame(
        "record_count"
    )
)

,record_count
location,
None,20


## 4.14 Inspect Existing isRemote

The existing source isRemote value is useful for understanding the raw data, but it should not be converted into the final location_type during EDA.

In [ ]:
if "isRemote" in source_eda_df.columns:

    isremote_counts = (
        source_eda_df["isRemote"]
        .value_counts(
            dropna=False
        )
    )
if "isRemote" in source_eda_df.columns:

    display(
        isremote_counts
        .to_frame(
            "record_count"
        )
    )

,record_count
isRemote,
False,20


## 4.15 Description Quality Analysis

The description field is central to later cleaning and AI enrichment.

EDA should establish:

missing descriptions,
blank descriptions,
HTML prevalence,
description lengths.

No description cleaning is performed here.

### 4.15.1 Missing and Blank Descriptions

In [ ]:
description_null_count = (
    source_eda_df["descriptionText"]
    .isna()
    .sum()
)
description_blank_count = (
    source_eda_df["descriptionText"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)
print(
    "Null descriptions:",
    description_null_count
)

print(
    "Blank descriptions:",
    description_blank_count
)

Null descriptions: 0
Blank descriptions: 0


## 4.16 HTML Prevalence

In [ ]:
html_mask = (
    source_eda_df["descriptionText"]
    .fillna("")
    .astype(str)
    .str.contains(
        r"<[^>]+>",
        regex=True
    )
)
html_count = html_mask.sum()
html_pct = (
    html_mask.mean()
    * 100
)
print(
    "Descriptions containing HTML:",
    html_count
)

print( f"Descriptions containing HTML %: " f"{html_pct:.2f}%" )

Descriptions containing HTML: 0
Descriptions containing HTML %: 0.00%


### 4.16.1 Inspect HTML Examples

In [ ]:
display( source_eda_df.loc[ html_mask, ["descriptionText"] ].head(5) )

,descriptionText


## 4.17 Description Length Distribution

This baseline will later help interpret AI token and resource usage.

In [ ]:
description_length = (
    source_eda_df["descriptionText"]
    .fillna("")
    .astype(str)
    .str.len()
)
display(
    description_length
    .describe()
    .to_frame(
        "description_length"
    )
)

,description_length
count,20.000000
mean,4758.050000
std,1269.118072
min,2604.000000
25%,4105.750000
50%,4832.000000
75%,5216.000000
max,7213.000000


## 4.18 Posting Date Quality Analysis

### 4.18.1 Test Date Parseability

In [ ]:
parsed_date_published = pd.to_datetime(
    source_eda_df["datePublished"],
    errors="coerce", utc=True )

### 4.18.2 Count Invalid Values

In [ ]:
invalid_date_count = (
    source_eda_df["datePublished"].notna()
    &
    parsed_date_published.isna()
).sum()
print(
    "Invalid non-null datePublished values:",
    invalid_date_count
)

Invalid non-null datePublished values: 0


### 4.18.3 Inspect Posting Date Range

In [ ]:
print(
    "Earliest datePublished:",
    parsed_date_published.min()
)
print(
    "Latest datePublished:",
    parsed_date_published.max()
)

Earliest datePublished: 2026-08-10 00:00:00+00:00
Latest datePublished: 2026-08-21 00:00:00+00:00


### 4.19 descriptionTextHash Quality Analysis

The description hash will later be used by one deduplication test case.

EDA should only determine its availability and completeness.

In [ ]:
if "descriptionTextHash" in source_eda_df.columns:

    description_hash_summary = pd.DataFrame({
        "Metric": [
            "Total source records",
            "Missing descriptionTextHash",
            "Non-null descriptionTextHash",
            "Unique descriptionTextHash"
        ],

        "Value": [
            len(source_eda_df),

            source_eda_df[
                "descriptionTextHash"
            ]
            .isna()
            .sum(),

            source_eda_df[
                "descriptionTextHash"
            ]
            .notna()
            .sum(),

            source_eda_df[
                "descriptionTextHash"
            ]
            .nunique(
                dropna=True
            )
        ]
    })
if "descriptionTextHash" in source_eda_df.columns:

    display(
        description_hash_summary
    )

## 4.20 AI-Enrichment Input Readiness

The supplied prompt library uses fields such as job description, job title, and existing industry information as AI inputs.

### 4.20.1 Define Relevant Raw AI Inputs

In [ ]:
ai_source_input_fields = [
    "descriptionText",
    "title",
    "industry"
]
ai_source_input_fields = [
    field
    for field in ai_source_input_fields
    if field in source_eda_df.columns
]

### 4.20.2 Measure AI Input Completeness

In [ ]:
ai_input_readiness = []
for field in ai_source_input_fields:

    series = source_eda_df[field]

    null_count = (
        series
        .isna()
        .sum()
    )

    blank_count = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    available_count = (
        len(series)
        - null_count
        - blank_count
    )

    ai_input_readiness.append({
        "field":
            field,

        "available_count":
            available_count,

        "available_pct":
            round(
                available_count
                / len(series)
                * 100,
                2
            ),

        "missing_count":
            null_count
            + blank_count
    })
ai_input_readiness_df = pd.DataFrame(
    ai_input_readiness
)
display(
    ai_input_readiness_df
)

,field,available_count,available_pct,missing_count
0,descriptionText,20,100.0,0
1,title,20,100.0,0


## 4.21 Final EDA Summary

Create one compact summary of the most important data-quality observations.

In [ ]:
eda_summary = pd.DataFrame({
    "Metric": [
        "Source records loaded",
        "Target records in 48h window",
        "Missing jobKey",
        "Rows in duplicated jobKey groups",
        "Missing companyName",
        "Missing title",
        "Missing location object",
        "Records with coordinates",
        "Coordinate coverage %",
        "Invalid datePublished",
        "Missing descriptionTextHash",
        "Descriptions containing HTML",
        "Descriptions containing HTML %",
        "Missing descriptions"
    ],

    "Value": [
        len(source_eda_df),

        len(target_48h_df),

        source_eda_df[
            "jobKey"
        ]
        .isna()
        .sum(),

        jobkey_duplicate_rows.sum(),

        source_eda_df[
            "companyName"
        ]
        .isna()
        .sum(),

        source_eda_df[
            "title"
        ]
        .isna()
        .sum(),

        source_eda_df[
            "location"
        ]
        .isna()
        .sum(),

        coordinate_available_mask.sum(),

        round(
            coordinate_coverage_pct,
            2
        ),

        invalid_date_count,

        (
            source_eda_df[
                "descriptionTextHash"
            ]
            .isna()
            .sum()
            if "descriptionTextHash"
            in source_eda_df.columns
            else np.nan
        ),

        html_count,

        round(
            html_pct,
            2
        ),

        description_null_count
        + description_blank_count
    ]
})
display(
    eda_summary
)

,Metric,Value
0,Source records loaded,20.0
1,Target records in 48h window,370.0
2,Missing jobKey,0.0
3,Rows in duplicated jobKey groups,0.0
4,Missing companyName,0.0
5,Missing title,0.0
6,Missing location object,0.0
7,Records with coordinates,20.0
8,Coordinate coverage %,100.0
9,Invalid datePublished,0.0


## 4.22 Store Baseline EDA Metrics

These metrics can later be compared against the cleaned dataset.

In [ ]:
BASELINE_EDA_METRICS = {
    "source_records":
        len(source_eda_df),

    "target_48h_records":
        len(target_48h_df),

    "missing_jobkey":
        int(
            source_eda_df[
                "jobKey"
            ]
            .isna()
            .sum()
        ),

    "duplicated_jobkey_group_rows":
        int(
            jobkey_duplicate_rows.sum()
        ),

    "coordinate_coverage_pct":
        float(
            coordinate_coverage_pct
        ),

    "invalid_dates":
        int(
            invalid_date_count
        ),

    "html_descriptions":
        int(
            html_count
        ),

    "missing_descriptions":
        int(
            description_null_count
            + description_blank_count
        )
}
if "descriptionTextHash" in source_eda_df.columns:

    BASELINE_EDA_METRICS[
        "missing_description_hash"
    ] = int(
        source_eda_df[
            "descriptionTextHash"
        ]
        .isna()
        .sum()
    )

## 4.23 Section 4 Outcome

Section 4 established the baseline quality and readiness of the raw Indeed source data and the corresponding 48-hour production comparison dataset before any pipeline processing was applied.

The EDA was intentionally limited to observation and measurement. No cleaning, transformation, deduplication, location classification, AI enrichment, or record removal was performed.

The analysis first established the working data window using the last modification timestamp of daily_scrape_dev as the pipeline anchor and derived the corresponding 48-hour lookback window for vw_dca_jobboard. This provides the production comparison dataset that will later be used for cross-dataset duplicate detection, consistent with the project's requirement to compare incoming Indeed jobs against recent production records.

Source Data Quality

The source dataset was analysed for general completeness and structural quality. The EDA identified:

the number of source records and fields available for processing;
null rates across the source dataset;
blank-string prevalence in text fields;
nested fields such as the location object;
and the completeness of fields that directly support the final pipeline.

Particular attention was given to fields required by downstream processing rather than attempting to analyse every source field equally.

jobKey Quality

jobKey was evaluated because it will later serve as the first source-level duplicate control.

The EDA measured:

missing jobKey values;
the number of unique non-null jobKey values;
and the number of source records belonging to duplicated jobKey groups.

Duplicate jobKey records were inspected but not removed. Actual jobKey deduplication will take place in the dedicated deduplication stage.

Deduplication Readiness

The fields required by the planned deduplication approaches were assessed for completeness.

For daily_scrape_dev, this included:

companyName,
title,
location,
datePublished,
and descriptionTextHash.

For the 48-hour vw_dca_jobboard comparison dataset, the availability and completeness of fields such as:

company_name,
job_title,
location,
posted_date,
and job_description

were also assessed.

No source-to-target duplicate matching was performed during EDA. The purpose of these checks was only to establish whether the datasets contain sufficient information for the later Case 1, Case 2, and Case 3 deduplication experiments.

Location Data Readiness

The nested source location object was analysed to determine the availability of geographic information required by the later location-type experiments.

The EDA measured missing values for:

location objects,
formatted addresses,
cities,
states,
latitude,
and longitude.

Coordinate coverage was also calculated by determining the proportion of source records containing both latitude and longitude.

This provides an important baseline for the later hybrid location-classification approach because it indicates how many records are potentially suitable for deterministic GCCSA spatial processing.

However, no GCCSA spatial join or Metro/Regional/Remote classification was performed in Section 4.

The existing isRemote field was also inspected as a raw source characteristic, but it was not used to derive the final location_type.

Description Quality

The raw descriptionText field was analysed because it is important for both cleaning and AI enrichment.

The EDA measured:

null descriptions;
blank descriptions;
the number and percentage of descriptions containing HTML;
and the distribution of description lengths.

These measurements create a baseline for the subsequent cleaning stage, where HTML and unnecessary whitespace will be removed.

Description length also provides useful context for the later AI-enrichment experiments because longer job descriptions can increase prompt size, token consumption, processing time, and API cost.

Posting-Date Quality

datePublished was assessed because it will later contribute to the date-based deduplication experiment.

The EDA evaluated:

whether non-null values could be successfully parsed as dates;
the number of invalid date values;
the earliest available posting date;
and the latest available posting date.

No date matching, date scoring, or similarity thresholds were applied at this stage.

Description Hash Readiness

descriptionTextHash was examined because it is required for the later description-hash deduplication approach.

The analysis measured:

total source records;
missing hash values;
non-null hash values;
and the number of unique hashes.

No hash-based duplicate comparison was performed.

AI-Enrichment Readiness

The availability of the primary raw inputs required by the AI enrichment process was also evaluated.

Fields such as:

descriptionText,
title,
and existing industry information

were checked for missing and blank values.

This establishes whether missing source inputs may affect the completeness or accuracy of later AI-enrichment outputs. The supplied prompt library uses job description, job title, technical skills, and source-industry information across its enrichment tasks.

No AI model calls were made during EDA.

Baseline Metrics Established

The key outcomes from Section 4 were stored in BASELINE_EDA_METRICS so that later stages can objectively compare the condition of the dataset before and after processing.

The stored baseline includes measures such as:

source record count;
48-hour target record count;
missing jobKey count;
duplicated jobKey group rows;
coordinate coverage;
invalid posting-date count;
descriptions containing HTML;
missing descriptions;
and missing description hashes where available.

These metrics will provide the reference point for later before-versus-after data-quality analysis.

Overall Conclusion

Section 4 confirms whether the available source and target data are sufficiently complete to proceed with the planned pipeline stages while documenting the main quality issues that must be addressed.

The processing sequence from this point remains:

Raw EDA Baseline
        ↓
Cleaning & Standardisation
        ↓
Location-Type Experiments
        ↓
jobKey Duplicate Removal
        ↓
Cross-Dataset Deduplication Experiments
        ↓
AI-Enrichment Experiments
        ↓
Post-Processing Data Quality Analysis
        ↓
Recommended Production Pipeline

Therefore, Section 4 describes the current state of the data only. All corrective actions and experimental comparisons are deliberately deferred to their respective downstream sections.

# 5. Data Cleaning and Standardisation

This section cleans and standardises the working source and target datasets before they are used by the later location, deduplication, and AI-enrichment stages.

The purpose is to ensure that downstream comparisons are performed on consistent values rather than differences caused by:

leading or trailing whitespace,
repeated internal whitespace,
HTML tags and HTML entities,
inconsistent null representations,
inconsistent text casing,
malformed dates,
nested location structures,
or incompatible field formats.

The project brief explicitly requires HTML removal, whitespace normalisation, location formatting, and date validation before downstream processing.

No deduplication, location classification, or AI enrichment is performed in this section.

## 5.1 Create Clean Working Copies

Preserve the EDA datasets and create separate working copies for cleaning.

In [ ]:
source_clean_df = source_eda_df.copy(deep=True)
target_clean_df = target_48h_df.copy(deep=True)
print(
    "Source rows:",
    len(source_clean_df)
)

print(
    "Target rows:",
    len(target_clean_df)
)

Source rows: 20
Target rows: 370


## 5.2 Define Null-Like Text Values

Some text fields may contain strings that represent missing values rather than true information.

In [ ]:
NULL_LIKE_VALUES = { "", "none", "null", "nan", "n/a", "na" }

## 5.3 Define a General Text-Cleaning Function

The general text-cleaning function:

handles null values,
trims leading and trailing whitespace,
normalises repeated internal whitespace,
and converts configured null-like strings to None.

It deliberately does not apply title case or other aggressive formatting that could alter legitimate company or job names.

In [ ]:
def clean_text(value):
  if pd.isna(value):
    return None

  value = str(value)

  value = re.sub( r"\s+", " ", value ).strip()

  if value.lower() in NULL_LIKE_VALUES:
    return None

  return value

## 5.4 Test the General Text Function

In [ ]:
text_cleaning_examples = [
    "  Data Analyst  ",
    "Senior   Data    Engineer",
    "",
    "NULL",
    None
]
for value in text_cleaning_examples:

    print(
        repr(value),
        "->",
        repr(
            clean_text(value)
        )
    )

'  Data Analyst  ' -> 'Data Analyst'
'Senior   Data    Engineer' -> 'Senior Data Engineer'
'' -> None
'NULL' -> None
None -> None


## 5.5 Define HTML Description Cleaning

Job descriptions may contain HTML tags and encoded HTML entities.

Use BeautifulSoup to convert them into readable plain text.

In [ ]:
def clean_description(value):
  if pd.isna(value):
    return None

  value = str(value)

  soup = BeautifulSoup( value, "html.parser" )

  text = soup.get_text( separator=" " )

  text = re.sub( r"\s+", " ", text ).strip()

  if not text:
    return None

  return text

## 5.6 Test Description Cleaning

In [ ]:
description_example = """
<p>We are looking for a <strong>Data Analyst</strong>.</p>
<ul>
<li>SQL</li>
<li>Python</li>
</ul>
"""
print(
    clean_description(
        description_example
    )
)

We are looking for a Data Analyst . SQL Python


## 5.7 Define Normalised Matching Text

Later deduplication requires comparable company, title, and location values.

Create a separate normalisation function for matching purposes.

The display values themselves will remain readable.

In [ ]:
def normalise_match_text(value):
  value = clean_text(value)

  if value is None:
    return None

  return value.lower()

## 5.8 Extract Source Location Components

The source location field is a nested dictionary.

Create clean scalar fields for:

formatted location,
city,
state,
latitude,
longitude.

These are working fields used during processing.

In [ ]:
def extract_location_value( location, key ):

  if not isinstance( location, dict ):
    return None

  return location.get(key)

### 5.8.1 Extract Formatted Location

In [ ]:
source_clean_df[ "clean_location" ] = (
    source_clean_df[ "location" ]
    .apply(
        lambda x:
        extract_location_value( x, "formattedAddressShort" ) )
    .apply( clean_text ) )

### 5.8.2 Extract City

In [ ]:
source_clean_df[ "clean_city" ] = (
    source_clean_df[ "location" ]
    .apply( lambda x: extract_location_value( x, "city" ) )
    .apply( clean_text ) )

### 5.8.3 Extract State

In [ ]:
source_clean_df[ "clean_state" ] = (
    source_clean_df[ "location" ]
    .apply( lambda x: extract_location_value( x, "state" ) )
    .apply( clean_text ) )

### 5.8.4 Extract Latitude

In [ ]:
source_clean_df[ "latitude" ] = (
    source_clean_df[ "location" ]
    .apply( lambda x: extract_location_value( x, "latitude" ) ) )

### 5.8.5 Extract Longitude

In [ ]:
source_clean_df[ "longitude" ] = (
    source_clean_df[ "location" ]
    .apply( lambda x: extract_location_value( x, "longitude" ) ) )

## 5.9 Convert Coordinates to Numeric Values

Invalid coordinate strings should become null rather than causing spatial-processing errors later.

In [ ]:
source_clean_df[
    "latitude"
] = pd.to_numeric(
    source_clean_df[
        "latitude"
    ],
    errors="coerce"
)
source_clean_df[
    "longitude"
] = pd.to_numeric(
    source_clean_df[
        "longitude"
    ],
    errors="coerce"
)

## 5.10 Validate Coordinate Ranges

Valid latitude values range from -90 to 90.

Valid longitude values range from -180 to 180.

In [ ]:
invalid_latitude_mask = (
    source_clean_df[
        "latitude"
    ].notna()
    &
    ~source_clean_df[
        "latitude"
    ].between(
        -90,
        90
    )
)
invalid_longitude_mask = (
    source_clean_df[
        "longitude"
    ].notna()
    &
    ~source_clean_df[
        "longitude"
    ].between(
        -180,
        180
    )
)
print(
    "Invalid latitude values:",
    invalid_latitude_mask.sum()
)

print(
    "Invalid longitude values:",
    invalid_longitude_mask.sum()
)

Invalid latitude values: 0
Invalid longitude values: 0


### 5.10.1 Replace Invalid Coordinates With Null

In [ ]:
source_clean_df.loc[
    invalid_latitude_mask,
    "latitude"
] = np.nan
source_clean_df.loc[
    invalid_longitude_mask,
    "longitude"
] = np.nan

## 5.11 Clean Source Company Names

In [ ]:
source_clean_df[ "clean_company" ] = (
    source_clean_df[ "companyName" ]
    .apply( clean_text ) )

## 5.12 Clean Source Job Titles

In [ ]:
source_clean_df[ "clean_title" ] = (
    source_clean_df[ "title" ]
    .apply( clean_text ) )

## 5.13 Clean Source Descriptions

In [ ]:
source_clean_df[ "clean_description" ] = (
    source_clean_df[ "descriptionText" ]
    .apply( clean_description )
    )

## 5.14 Clean Source Job URLs

In [ ]:
source_clean_df[ "clean_url" ] = (
    source_clean_df[ "jobUrl" ]
    .apply( clean_text ) )

## 5.15 Clean Source Industry Information

Only clean the field if it exists in the source dataset.

In [ ]:
source_clean_df[ "clean_industry" ] = (
    source_clean_df[ "companyIndustry" ]
    .apply( clean_text ) )

## 5.16 Clean jobKey

In [ ]:
source_clean_df[ "clean_jobKey" ] = (
    source_clean_df[ "jobKey" ]
    .apply( clean_text ) )

## 5.17 Parse Source Posting Date

Convert datePublished into a consistent timezone-aware datetime.

In [ ]:
source_clean_df[ "clean_posted_date" ] = pd.to_datetime(
    source_clean_df[ "datePublished" ],
    errors="coerce", utc=True )

## 5.18 Measure Failed Date Conversions

In [ ]:
source_failed_date_conversion = (
    source_clean_df[
        "datePublished"
    ].notna()
    &
    source_clean_df[
        "clean_posted_date"
    ].isna()
)
print(
    "Source dates that could not be parsed:",
    source_failed_date_conversion.sum()
)

Source dates that could not be parsed: 0


## 5.19 Prepare Matching Versions of Source Fields

Create normalised versions that can later be used for deduplication.

No matching is performed here.

In [ ]:
source_clean_df[
    "match_company"
] = (
    source_clean_df[
        "clean_company"
    ]
    .apply(
        normalise_match_text
    )
)
source_clean_df[
    "match_title"
] = (
    source_clean_df[
        "clean_title"
    ]
    .apply(
        normalise_match_text
    )
)
source_clean_df[
    "match_location"
] = (
    source_clean_df[
        "clean_location"
    ]
    .apply(
        normalise_match_text
    )
)

## 5.21 Clean Target Company Names

The target comparison dataset must use the same cleaning logic so that later comparisons are fair.

In [ ]:
target_clean_df[ "clean_company" ] = (
    target_clean_df[ "company_name" ]
    .apply( clean_text ) )

## 5.22 Clean Target Job Titles

In [ ]:
target_clean_df[ "clean_title" ] = (
    target_clean_df[ "job_title" ]
    .apply( clean_text ) )

## 5.23 Clean Target Locations

In [ ]:
target_clean_df[ "clean_location" ] = (
    target_clean_df[ "location" ]
    .apply( clean_text ) )

## 5.24 Clean Target Descriptions

In [ ]:
target_clean_df[ "clean_description" ] = (
    target_clean_df[ "job_description" ]
    .apply( clean_description ) )

## 5.25 Parse Target Posting Dates

In [ ]:
target_clean_df[ "clean_posted_date" ] = pd.to_datetime(
    target_clean_df[ "posted_date" ],
    errors="coerce", utc=True )

## 5.26 Prepare Target Matching Fields

Use exactly the same matching normalisation applied to the source.

In [ ]:
target_clean_df[
    "match_company"
] = (
    target_clean_df[
        "clean_company"
    ]
    .apply(
        normalise_match_text
    )
)
target_clean_df[
    "match_title"
] = (
    target_clean_df[
        "clean_title"
    ]
    .apply(
        normalise_match_text
    )
)
target_clean_df[
    "match_location"
] = (
    target_clean_df[
        "clean_location"
    ]
    .apply(
        normalise_match_text
    )
)

## 5.27 Measure Source Description Cleaning Effect

Compare the number of HTML-containing descriptions before and after cleaning.

In [ ]:
cleaned_html_mask = (
    source_clean_df[
        "clean_description"
    ]
    .fillna("")
    .astype(str)
    .str.contains(
        r"<[^>]+>",
        regex=True
    )
)
print(
    "HTML descriptions before cleaning:",
    html_count
)

print(
    "HTML descriptions after cleaning:",
    cleaned_html_mask.sum()
)

HTML descriptions before cleaning: 0
HTML descriptions after cleaning: 0


## 5.28 Measure Whitespace Cleaning Effect

Create a helper function to identify text values containing leading, trailing, or repeated whitespace.

In [ ]:
def has_whitespace_issue(value):
  if pd.isna(value):
    return False

  value = str(value)

  return (
      value != value.strip() or
      bool( re.search( r"\s{2,}", value ) ) )

### 5.28.1 Company Whitespace Issues Before Cleaning

In [ ]:
company_whitespace_before = (
    source_eda_df[ "companyName" ]
    .apply( has_whitespace_issue )
    .sum() )

### 5.28.2 Company Whitespace Issues After Cleaning

In [ ]:
company_whitespace_after = (
    source_clean_df[
        "clean_company"
    ]
    .apply(
        has_whitespace_issue
    )
    .sum()
)
print(
    "Company whitespace issues before:",
    company_whitespace_before
)

print(
    "Company whitespace issues after:",
    company_whitespace_after
)

Company whitespace issues before: 0
Company whitespace issues after: 0


## 5.29 Job Title Whitespace Comparison

In [ ]:
title_whitespace_before = (
    source_eda_df[
        "title"
    ]
    .apply(
        has_whitespace_issue
    )
    .sum()
)
title_whitespace_after = (
    source_clean_df[
        "clean_title"
    ]
    .apply(
        has_whitespace_issue
    )
    .sum()
)
print(
    "Title whitespace issues before:",
    title_whitespace_before
)

print(
    "Title whitespace issues after:",
    title_whitespace_after
)

Title whitespace issues before: 0
Title whitespace issues after: 0


## 5.30 Measure Cleaned Source Null Rates

Evaluate the principal cleaned fields.

In [ ]:
cleaned_quality_fields = [
    "clean_jobKey",
    "clean_company",
    "clean_title",
    "clean_location",
    "clean_city",
    "clean_state",
    "clean_description",
    "clean_posted_date",
    "clean_url"
]
cleaned_null_profile = pd.DataFrame({
    "field":
        cleaned_quality_fields,

    "null_count":
        [
            source_clean_df[
                field
            ].isna().sum()
            for field
            in cleaned_quality_fields
        ],

    "null_pct":
        [
            round(
                source_clean_df[
                    field
                ].isna().mean()
                * 100,
                2
            )
            for field
            in cleaned_quality_fields
        ]
})
display(
    cleaned_null_profile
)

,field,null_count,null_pct
0,clean_jobKey,0,0.0
1,clean_company,0,0.0
2,clean_title,0,0.0
3,clean_location,0,0.0
4,clean_city,0,0.0
5,clean_state,20,100.0
6,clean_description,0,0.0
7,clean_posted_date,0,0.0
8,clean_url,0,0.0


## 5.31 Compare Before and After Data Quality

Create a compact cleaning-effect summary.

In [ ]:
cleaning_summary = pd.DataFrame({
    "Metric": [
        "Descriptions containing HTML",
        "Company whitespace issues",
        "Title whitespace issues",
        "Invalid source dates",
        "Invalid latitude values",
        "Invalid longitude values"
    ],

    "Before": [
        html_count,
        company_whitespace_before,
        title_whitespace_before,
        invalid_date_count,
        invalid_latitude_mask.sum(),
        invalid_longitude_mask.sum()
    ],

    "After": [
        cleaned_html_mask.sum(),
        company_whitespace_after,
        title_whitespace_after,
        source_clean_df[
            "clean_posted_date"
        ].isna().sum(),
        source_clean_df[
            "latitude"
        ].notna().sum()
        - source_clean_df[
            "latitude"
        ].between(
            -90,
            90
        ).sum(),
        source_clean_df[
            "longitude"
        ].notna().sum()
        - source_clean_df[
            "longitude"
        ].between(
            -180,
            180
        ).sum()
    ]
})
display(
    cleaning_summary
)

,Metric,Before,After
0,Descriptions containing HTML,0,0
1,Company whitespace issues,0,0
2,Title whitespace issues,0,0
3,Invalid source dates,0,0
4,Invalid latitude values,0,0
5,Invalid longitude values,0,0


In [ ]:
row_index = 0

cleaning_comparison_df = pd.DataFrame([
    {
        "Field": "jobKey",
        "Raw": source_eda_df.loc[row_index, "jobKey"],
        "Cleaned": source_clean_df.loc[row_index, "clean_jobKey"]
    },
    {
        "Field": "companyName",
        "Raw": source_eda_df.loc[row_index, "companyName"],
        "Cleaned": source_clean_df.loc[row_index, "clean_company"]
    },
    {
        "Field": "title",
        "Raw": source_eda_df.loc[row_index, "title"],
        "Cleaned": source_clean_df.loc[row_index, "clean_title"]
    },
    {
        "Field": "descriptionText",
        "Raw": str(
            source_eda_df.loc[
                row_index,
                "descriptionText"
            ]
        )[:500],
        "Cleaned": str(
            source_clean_df.loc[
                row_index,
                "clean_description"
            ]
        )[:500]
    },
    {
        "Field": "jobUrl",
        "Raw": source_eda_df.loc[row_index, "jobUrl"],
        "Cleaned": source_clean_df.loc[row_index, "clean_url"]
    },
    {
        "Field": "location",
        "Raw": str(
            source_eda_df.loc[
                row_index,
                "location"
            ]
        ),
        "Cleaned": source_clean_df.loc[
            row_index,
            "clean_location"
        ]
    },
    {
        "Field": "city",
        "Raw": (
            source_eda_df.loc[
                row_index,
                "location"
            ].get("city")
            if isinstance(
                source_eda_df.loc[
                    row_index,
                    "location"
                ],
                dict
            )
            else None
        ),
        "Cleaned": source_clean_df.loc[
            row_index,
            "clean_city"
        ]
    },
    {
        "Field": "state",
        "Raw": str(
            source_eda_df.loc[
                row_index,
                "location"
            ]
        ),
        "Cleaned": source_clean_df.loc[
            row_index,
            "clean_state"
        ]
    },
    {
        "Field": "datePublished",
        "Raw": source_eda_df.loc[
            row_index,
            "datePublished"
        ],
        "Cleaned": source_clean_df.loc[
            row_index,
            "clean_posted_date"
        ]
    }
])

cleaning_comparison_df[
    "Changed?"
] = (
    cleaning_comparison_df[
        "Raw"
    ].astype(str)
    !=
    cleaning_comparison_df[
        "Cleaned"
    ].astype(str)
)

display(
    cleaning_comparison_df
)

,Field,Raw,Cleaned,Changed?
0,jobKey,9c2916ac1bdaf7e2,9c2916ac1bdaf7e2,False
1,companyName,Department of Education,Department of Education,False
2,title,Data Analyst,Data Analyst,False
3,descriptionText,This position commences 1 January 2027\n\nAre ...,This position commences 1 January 2027 Are you...,True
4,jobUrl,https://au.indeed.com/viewjob?jk=9c2916ac1bdaf7e2,https://au.indeed.com/viewjob?jk=9c2916ac1bdaf7e2,False
5,location,"{'latitude': -37.818, 'formattedAddressShort':...",Melbourne City Centre VIC,True
6,city,Melbourne City Centre,Melbourne City Centre,False
7,state,"{'latitude': -37.818, 'formattedAddressShort':...",None,True
8,datePublished,2026-08-21,2026-08-21 00:00:00+00:00,True


## 5.32 Validate Description HTML Removal

In [ ]:
assert cleaned_html_mask.sum() == 0, (
    "HTML tags remain in one or more cleaned descriptions."
)
print(
    "✓ HTML cleaning validation passed."
)

✓ HTML cleaning validation passed.


## 5.33 Validate Coordinate Ranges

In [ ]:
valid_latitude_check = (
    source_clean_df[
        "latitude"
    ]
    .dropna()
    .between(
        -90,
        90
    )
    .all()
)
valid_longitude_check = (
    source_clean_df[
        "longitude"
    ]
    .dropna()
    .between(
        -180,
        180
    )
    .all()
)
assert valid_latitude_check
assert valid_longitude_check
print(
    "✓ Coordinate validation passed."
)

✓ Coordinate validation passed.


## 5.34 Validate Row Preservation

Cleaning should change field values but should not remove records.

In [ ]:
assert (
    len(source_clean_df)
    ==
    len(source_eda_df)
), (
    "Source records were removed during cleaning."
)
assert (
    len(target_clean_df)
    ==
    len(target_48h_df)
), (
    "Target records were removed during cleaning."
)
print(
    "✓ Cleaning preserved all source and target rows."
)

✓ Cleaning preserved all source and target rows.


## 5.35 Store Cleaning Metrics

In [ ]:
CLEANING_METRICS = {
    "source_rows_before": len(source_eda_df),
    "source_rows_after": len(source_clean_df),
    "html_before": int( html_count ),
    "html_after": int( cleaned_html_mask.sum() ),
    "company_whitespace_before": int( company_whitespace_before ),
    "company_whitespace_after": int( company_whitespace_after ),
    "title_whitespace_before": int( title_whitespace_before ),
    "title_whitespace_after": int( title_whitespace_after ),
    "date_parse_failures": int( source_failed_date_conversion.sum() ) }

## 5.36 Create the Clean Pipeline Working Dataset

The cleaned source dataframe contains raw fields plus temporary processing fields.

Create a concise working dataframe containing only the source information required by later stages.

In [ ]:
pipeline_working_df = source_clean_df[ [
    "clean_jobKey", "clean_company",
    "clean_title", "clean_location",
    "clean_city", "clean_state",
    "latitude", "longitude",
    "clean_posted_date", "clean_description",
    "clean_url" ] ].copy()

### 5.36.1 Retain Matching Fields

In [ ]:
pipeline_working_df[
    "match_company"
] = (
    source_clean_df[
        "match_company"
    ]
)
pipeline_working_df[
    "match_title"
] = (
    source_clean_df[
        "match_title"
    ]
)
pipeline_working_df[
    "match_location"
] = (
    source_clean_df[
        "match_location"
    ]
)

## 5.37 Retain Raw Fields Required by Later AI Processing

The AI experiments will need the cleaned source description, title, and any available source industry information.

In [ ]:
if ( "clean_industry" in source_clean_df.columns ):
  pipeline_working_df[ "clean_industry" ] = (
    source_clean_df[ "clean_industry" ] )

## 5.38 Inspect Clean Working Dataset

In [ ]:
display( pipeline_working_df.head() )

,clean_jobKey,clean_company,clean_title,clean_location,clean_city,clean_state,latitude,longitude,clean_posted_date,clean_description,clean_url,match_company,match_title,match_location,clean_industry
0,9c2916ac1bdaf7e2,Department of Education,Data Analyst,Melbourne City Centre VIC,Melbourne City Centre,None,-37.81800,144.97600,2026-08-21 00:00:00+00:00,This position commences 1 January 2027 Are you...,https://au.indeed.com/viewjob?jk=9c2916ac1bdaf7e2,department of education,data analyst,melbourne city centre vic,None
1,1651d59366f4f3a1,CSIRO,Manager - Governance of Indigenous Data and In...,Brisbane QLD,Brisbane,None,-27.46794,153.02809,2026-08-21 00:00:00+00:00,Acknowledgement of Country CSIRO acknowledges ...,https://au.indeed.com/viewjob?jk=1651d59366f4f3a1,csiro,manager - governance of indigenous data and in...,brisbane qld,GOVERNMENT
2,d140df231910c633,Amazon.com,"Technical Program Manager, Data Center Delivery",Sydney NSW,Sydney,None,-33.86785,151.20732,2026-08-21 00:00:00+00:00,DESCRIPTION AWS Infrastructure Services owns t...,https://au.indeed.com/viewjob?jk=d140df231910c633,amazon.com,"technical program manager, data center delivery",sydney nsw,INTERNET_AND_SOFTWARE
3,28e7a32d55dffa4f,Wesfarmers,Data Scientist - Hybrid,Melbourne VIC,Melbourne,None,-37.81400,144.96332,2026-08-21 00:00:00+00:00,Work options: Hybrid Wesfarmers OneDigital is ...,https://au.indeed.com/viewjob?jk=28e7a32d55dffa4f,wesfarmers,data scientist - hybrid,melbourne vic,None
4,10f4851d92195dbd,ANZ,Data Analyst,Melbourne VIC,Melbourne,None,-37.81400,144.96332,2026-08-20 00:00:00+00:00,Data Analyst Req ID: 122042 Department: AR Cus...,https://au.indeed.com/viewjob?jk=10f4851d92195dbd,anz,data analyst,melbourne vic,None


## 5.39 Section 5 Outcome

Section 5 transformed the raw EDA datasets into cleaned and standardised working datasets suitable for subsequent pipeline processing.

The cleaning stage:

preserved all source and target records;
normalised leading, trailing, and repeated whitespace;
standardised null-like text values;
extracted readable location components from the nested Indeed location object;
extracted city, state, latitude, and longitude for later geographic processing;
converted coordinate values to numeric form;
removed invalid coordinate values;
cleaned company names and job titles without applying aggressive casing transformations;
removed HTML tags and normalised whitespace in job descriptions;
cleaned URLs;
parsed source and target posting dates into consistent datetime values;
preserved the source-provided descriptionTextHash;
prepared consistent matching representations of company, title, and location for later cross-dataset deduplication;
and cleaned both daily_scrape_dev and the 48-hour vw_dca_jobboard comparison dataset using the same text-normalisation rules.

The section also quantified the effect of cleaning by comparing:

HTML prevalence before and after cleaning;
whitespace problems before and after cleaning;
date conversion quality;
and coordinate validity.

Importantly, Section 5 does not:

remove duplicate jobKey records;
create or execute deduplication test cases;
classify locations as Metro, Regional, or Remote;
call the AI model;
or create final enrichment outputs.

These operations remain separated into their dedicated experimental sections.

The resulting datasets are:

source_clean_df

for the fully cleaned source records,

target_clean_df

for the cleaned 48-hour production comparison dataset, and

pipeline_working_df

for the concise source-side working dataset required by subsequent pipeline stages.

The next section will use the cleaned geographic data to evaluate the two planned location-type determination approaches:

AI-only classification;
GCCSA deterministic classification with AI fallback.

# 6. Location Type Determination Experiments

This section evaluates the two required approaches for generating the final location_type field.

The approved output values are:

Metro
Regional
Remote

The two experimental cases are:

**Case 1 — AI Only**

Every record is sent to the AI model for classification.

**Case 2 — Hybrid GCCSA + AI Fallback**

The cleaned coordinates are first spatially matched against the supplied 2026 GCCSA shapefile.

The deterministic result is used where available.

If the deterministic method returns:

* None,
* NaN,
* null,
* or no spatial match,

the record is passed to the AI model.

Both cases will operate on the same cleaned records and will be evaluated using common performance and resource metrics.

No deduplication is performed in this section.

## 6.1 Prepare the Location Experiment Dataset

Use a copy of the cleaned working dataset so the original cleaned dataframe remains unchanged.

In [ ]:
location_test_df = (
    pipeline_working_df
    .copy(deep=True)
)
print(
    "Location experiment records:",
    len(location_test_df)
)

Location experiment records: 20


## 6.2 Inspect Required Location Fields

The location experiments require:

cleaned location,
city,
state,
latitude,
longitude.

In [ ]:
LOCATION_REQUIRED_FIELDS = [
    "clean_location",
    "clean_city",
    "clean_state",
    "latitude",
    "longitude"
]
location_field_check = pd.DataFrame({
    "field": LOCATION_REQUIRED_FIELDS,
    "exists": [
        field in location_test_df.columns
        for field in LOCATION_REQUIRED_FIELDS
    ]
})
display(
    location_field_check
)

,field,exists
0,clean_location,True
1,clean_city,True
2,clean_state,True
3,latitude,True
4,longitude,True


## 6.3 Validate Required Location Fields

In [ ]:
assert location_field_check["exists"].all(), (
    "One or more required location fields "
    "are missing from the cleaned dataset."
)
print(
    "✓ Required location fields available."
)

✓ Required location fields available.


## 6.4 Define Valid Location Outputs

In [ ]:
VALID_LOCATION_TYPES = [ "Metro", "Regional", "Remote" ]

## 6.5 Define the AI Location Prompt

The location-classification prompt is defined directly in the notebook because the existing prompt JSON does not contain a location-type prompt.

The AI is restricted to the three approved output values.

In [ ]:
LOCATION_AI_PROMPT = """
You are classifying Australian job locations.
INPUT Location: {location}
City: {city}
State: {state}
TASK Classify the job location into exactly one of:
Metro
Regional
Remote
Definitions:
Metro: A location within a major Australian metropolitan or large urban area.
Regional: A populated city, town or regional centre outside the major metropolitan areas.
Remote: A geographically isolated or sparsely populated location that would reasonably be considered remote in the Australian context.
Use the location, city and state together.
Return only one value:
Metro
Regional
Remote
Do not provide an explanation. """

## 6.6 Define AI Location Response Validation

In [ ]:
def validate_location_type(value):
  if value is None:
    return None

  value = str(value).strip()

  for valid_value in VALID_LOCATION_TYPES:
      if value.lower() == valid_value.lower():
        return valid_value

  return None

## 6.7 Define a Single AI Location Classification Function

This function:

* builds the prompt,
* calls Gemini,
* validates the response,
* captures token usage,
* and records whether the call succeeded.

In [ ]:
def classify_location_ai( location, city, state, model=DEFAULT_AI_MODEL ):
    prompt = LOCATION_AI_PROMPT.format( location=location or "None", city=city or "None", state=state or "None" )
    start_time = time.perf_counter()

    try:
      response = ( gemini_client .models .generate_content( model=model, contents=prompt ) )

      elapsed = ( time.perf_counter() - start_time )

      raw_output = ( response.text.strip()
      if response.text else None )

      prediction = ( validate_location_type( raw_output ) )

      usage = getattr( response, "usage_metadata", None )

      input_tokens = (
          getattr( usage, "prompt_token_count", 0 )
          if usage else 0 )

      output_tokens = (
          getattr( usage, "candidates_token_count", 0 )
          if usage else 0 )

      total_tokens = (
          getattr( usage, "total_token_count", input_tokens + output_tokens )
          if usage else 0 )

      return {
          "prediction": prediction,
          "raw_output": raw_output,
          "success": prediction is not None,
          "execution_time_seconds": elapsed,
          "input_tokens": input_tokens,
          "output_tokens": output_tokens,
          "total_tokens": total_tokens,
          "error": None }

    except Exception as exc:
      elapsed = ( time.perf_counter() - start_time )
      return {
          "prediction": None,
          "raw_output": None,
          "success": False,
          "execution_time_seconds": elapsed,
          "input_tokens": 0,
          "output_tokens": 0,
          "total_tokens": 0,
          "error": str(exc) }

## 6.8 Create a Reproducible Experiment Sample

Running AI against every source record during development may consume unnecessary API resources.

Use a reproducible sample for the experimental comparison.

The sample size can later be increased.

In [ ]:
LOCATION_TEST_SAMPLE_SIZE = min(
    200,
    len(location_test_df)
)
location_sample_df = (
    location_test_df
    .sample(
        n=LOCATION_TEST_SAMPLE_SIZE,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)
print(
    "Location test sample size:",
    len(location_sample_df)
)


Location test sample size: 20


## 6.9 Case 1 — AI-Only Classification

In Case 1, every record is sent directly to Gemini.

In [ ]:
case1_ai_results = []
case1_start_time = (
    time.perf_counter()
)
for index, row in location_sample_df.iterrows():

    result = classify_location_ai(
        location=row["clean_location"],
        city=row["clean_city"],
        state=row["clean_state"]
    )

    result["record_index"] = index

    case1_ai_results.append(
        result
    )
case1_total_time = (
    time.perf_counter()
    - case1_start_time
)

## 6.10 Convert Case 1 Results to DataFrame

In [ ]:
case1_results_df = pd.DataFrame(
    case1_ai_results
)
display(
    case1_results_df.head()
)

,prediction,raw_output,success,execution_time_seconds,input_tokens,output_tokens,total_tokens,error,record_index
0,Metro,Metro,True,0.400354,131,1,132,None,0
1,Metro,Metro,True,0.241879,127,1,128,None,1
2,Metro,Metro,True,0.265711,127,1,128,None,2
3,Metro,Metro,True,0.233930,128,1,129,None,3
4,Metro,Metro,True,0.234458,129,1,130,None,4


## 6.11 Attach Case 1 Predictions

In [ ]:
case1_output_df = (
    location_sample_df
    .copy()
)
case1_output_df[
    "location_type"
] = (
    case1_results_df[
        "prediction"
    ]
)

## 6.12 Case 1 Classification Distribution

In [ ]:
case1_distribution = (
    case1_output_df[
        "location_type"
    ]
    .value_counts(
        dropna=False
    )
)
display(
    case1_distribution
    .to_frame(
        "record_count"
    )
)

,record_count
location_type,
Metro,20


## 6.13 Case 1 Performance Metrics

In [ ]:
case1_success_count = (
    case1_results_df[
        "success"
    ]
    .sum()
)
case1_failure_count = (
    (~case1_results_df[
        "success"
    ])
    .sum()
)
case1_valid_output_rate = (
    case1_success_count
    / len(case1_results_df)
    * 100
)

## 6.14 Case 1 Token Usage

In [ ]:
case1_input_tokens = (
    case1_results_df[
        "input_tokens"
    ]
    .sum()
)
case1_output_tokens = (
    case1_results_df[
        "output_tokens"
    ]
    .sum()
)
case1_total_tokens = (
    case1_results_df[
        "total_tokens"
    ]
    .sum()
)

## 6.15 Create Case 1 Metrics Summary

In [ ]:
case1_metrics = {
    "case":
        "Case 1",

    "approach":
        "AI Only",

    "records_processed":
        len(case1_output_df),

    "classified_records":
        int(
            case1_success_count
        ),

    "unresolved_records":
        int(
            case1_failure_count
        ),

    "valid_output_rate_pct":
        round(
            case1_valid_output_rate,
            2
        ),

    "ai_calls":
        len(case1_results_df),

    "total_execution_time_seconds":
        case1_total_time,

    "avg_time_per_record_seconds":
        (
            case1_total_time
            / len(case1_results_df)
        ),

    "input_tokens":
        int(
            case1_input_tokens
        ),

    "output_tokens":
        int(
            case1_output_tokens
        ),

    "total_tokens":
        int(
            case1_total_tokens
        )
}
display(
    pd.DataFrame(
        case1_metrics.items(),
        columns=[
            "Metric",
            "Value"
        ]
    )
)

,Metric,Value
0,case,Case 1
1,approach,AI Only
2,records_processed,20
3,classified_records,20
4,unresolved_records,0
5,valid_output_rate_pct,100.0
6,ai_calls,20
7,total_execution_time_seconds,5.277819
8,avg_time_per_record_seconds,0.263891
9,input_tokens,2561


## 6.16 Prepare the GCCSA Spatial Dataset

Case 2 uses the GCCSA shapefile loaded in Section 2.

Inspect its columns before classification.

In [ ]:
print( gccsa_gdf.columns.tolist() )

['GCC_CODE26', 'GCC_NAME26', 'CHG_FLAG26', 'CHG_LBL26', 'STE_CODE26', 'STE_NAME26', 'AUS_CODE26', 'AUS_NAME26', 'AREASQKM26', 'geometry']


## 6.17 Identify the GCCSA Name Field

For the supplied 2026 GCCSA shapefile, the expected geography-name field is:

GCC_NAME26

Validate that it exists.

In [ ]:
GCCSA_NAME_FIELD = "GCC_NAME26"
assert (
    GCCSA_NAME_FIELD
    in gccsa_gdf.columns
), (
    f"{GCCSA_NAME_FIELD} not found "
    "in GCCSA shapefile."
)

## 6.18 Inspect GCCSA Categories

In [ ]:
gccsa_names = (
    gccsa_gdf[
        GCCSA_NAME_FIELD
    ]
    .dropna()
    .sort_values()
    .unique()
)
for value in gccsa_names:
    print(value)

Australian Capital Territory
Greater Adelaide
Greater Brisbane
Greater Darwin
Greater Hobart
Greater Melbourne
Greater Perth
Greater Sydney
Migratory - Offshore - Shipping (ACT)
Migratory - Offshore - Shipping (NSW)
Migratory - Offshore - Shipping (NT)
Migratory - Offshore - Shipping (OT)
Migratory - Offshore - Shipping (Qld)
Migratory - Offshore - Shipping (SA)
Migratory - Offshore - Shipping (Tas)
Migratory - Offshore - Shipping (Vic)
Migratory - Offshore - Shipping (WA)
No usual address (ACT)
No usual address (NSW)
No usual address (NT)
No usual address (OT)
No usual address (Qld)
No usual address (SA)
No usual address (Tas)
No usual address (Vic)
No usual address (WA)
Other Territories
Outside Australia
Rest of NSW
Rest of NT
Rest of Qld
Rest of SA
Rest of Tas
Rest of Vic
Rest of WA


## 6.19 Define Deterministic GCCSA Classification

The supplied GCCSA geography distinguishes capital-city areas from the remainder of each state.

For the deterministic experiment:

* Greater ... areas are classified as
* Metro;
* Rest of ... areas are classified as Regional;
No usual address and migratory/offshore categories are left unresolved;
an unsuccessful spatial join also returns unresolved.

In [ ]:
def classify_gccsa_name( gccsa_name ):
  if pd.isna(gccsa_name):
    return None

  value = str( gccsa_name ).strip()

  if value.startswith( "Greater " ):
    return "Metro"

  if value.startswith( "Rest of " ):
    return "Regional"

  if ( "No usual address" in value ):
    return None

  if ( "Migratory" in value or "Offshore" in value or "Shipping" in value ):
    return None

  return None

## 6.20 Create Spatial Points From Coordinates

Only records containing both latitude and longitude can be passed to the deterministic spatial method.

In [ ]:
spatial_input_df = (
    location_sample_df.loc[
        location_sample_df[
            "latitude"
        ].notna()
        &
        location_sample_df[
            "longitude"
        ].notna()
    ]
    .copy()
)
print(
    "Records with coordinates:",
    len(spatial_input_df)
)

Records with coordinates: 20


## 6.21 Convert Sample Records to GeoDataFrame

The source coordinates are latitude/longitude values in WGS84.

In [ ]:
spatial_points_gdf = (
    gpd.GeoDataFrame(
        spatial_input_df,
        geometry=gpd.points_from_xy(
            spatial_input_df[ "longitude" ],
            spatial_input_df[ "latitude" ] ),
        crs="EPSG:4326" ) )

## 6.22 Convert Points to the GCCSA CRS

The points and polygons must use the same coordinate-reference system before the spatial join.

In [ ]:
spatial_points_gdf = (
    spatial_points_gdf
    .to_crs(
        gccsa_gdf.crs
    )
)
print(
    "Point CRS:",
    spatial_points_gdf.crs
)

print(
    "GCCSA CRS:",
    gccsa_gdf.crs
)

Point CRS: EPSG:7844
GCCSA CRS: EPSG:7844


## 6.23 Perform GCCSA Spatial Join

Measure the deterministic processing time separately.

In [ ]:
gccsa_start_time = (
    time.perf_counter()
)
gccsa_join_df = (
    gpd.sjoin(
        spatial_points_gdf,
        gccsa_gdf[
            [
                GCCSA_NAME_FIELD,
                "geometry"
            ]
        ],
        how="left",
        predicate="within"
    )
)
gccsa_execution_time = (
    time.perf_counter()
    - gccsa_start_time
)

## 6.24 Apply Deterministic Location Classification

In [ ]:
gccsa_join_df[ "deterministic_location_type" ] = (
    gccsa_join_df[ GCCSA_NAME_FIELD ]
    .apply( classify_gccsa_name ) )

## 6.25 Inspect Deterministic Results

In [ ]:
display(
    gccsa_join_df[
        [
            "clean_location",
            "clean_city",
            "clean_state",
            "latitude",
            "longitude",
            GCCSA_NAME_FIELD,
            "deterministic_location_type"
            ] ]
    .head(20) )

,clean_location,clean_city,clean_state,latitude,longitude,GCC_NAME26,deterministic_location_type
0,Melbourne City Centre VIC,Melbourne City Centre,None,-37.818000,144.97600,Greater Melbourne,Metro
1,Canberra ACT,Canberra,None,-35.277200,149.10540,Australian Capital Territory,None
2,Clayton VIC,Clayton,None,-37.926056,145.11943,Greater Melbourne,Metro
3,Brisbane QLD,Brisbane,None,-27.467940,153.02809,Greater Brisbane,Metro
4,Parramatta NSW,Parramatta,None,-33.816670,151.00000,Greater Sydney,Metro
5,Sydney NSW,Sydney,None,-33.867850,151.20732,Greater Sydney,Metro
6,Melbourne City Centre VIC,Melbourne City Centre,None,-37.818000,144.97600,Greater Melbourne,Metro
7,Melbourne VIC,Melbourne,None,-37.814000,144.96332,Greater Melbourne,Metro
8,Sydney NSW,Sydney,None,-33.867850,151.20732,Greater Sydney,Metro
9,Melbourne VIC,Melbourne,None,-37.814000,144.96332,Greater Melbourne,Metro


## 6.26 Deterministic Classification Coverage

In [ ]:
deterministic_classified_count = (
    gccsa_join_df[
        "deterministic_location_type"
    ]
    .notna()
    .sum()
)
deterministic_coverage_pct = (
    deterministic_classified_count
    / len(location_sample_df)
    * 100
)
print(
    "Deterministically classified:",
    deterministic_classified_count
)

print(
    f"Deterministic coverage: "
    f"{deterministic_coverage_pct:.2f}%"
)

Deterministically classified: 19
Deterministic coverage: 95.00%


## 6.27 Initialise Hybrid Results

Start with the full experiment sample.

In [ ]:
case2_output_df = (
    location_sample_df
    .copy()
)
case2_output_df[
    "location_type"
] = None
case2_output_df[
    "classification_method"
] = None

## 6.28 Copy Deterministic Results Into Hybrid Dataset

In [ ]:
for index, row in gccsa_join_df.iterrows():
  prediction = row[ "deterministic_location_type" ]

  if pd.notna(prediction):
    case2_output_df.loc[
        index, "location_type"
      ] = prediction

    case2_output_df.loc[
        index, "classification_method"
        ] = "GCCSA"

## 6.29 Identify AI Fallback Records

Any record not classified deterministically requires AI fallback.

In [ ]:
fallback_mask = (
    case2_output_df[
        "location_type"
    ]
    .isna()
)
fallback_count = (
    fallback_mask.sum()
)
fallback_pct = (
    fallback_count
    / len(case2_output_df)
    * 100
)
print(
    "Records requiring AI fallback:",
    fallback_count
)

print(
    f"AI fallback rate: "
    f"{fallback_pct:.2f}%"
)

Records requiring AI fallback: 1
AI fallback rate: 5.00%


## 6.30 Run AI Only on Fallback Records

In [ ]:
case2_ai_results = []
case2_start_time = (
    time.perf_counter()
)
for index, row in (
    case2_output_df.loc[
        fallback_mask
    ]
    .iterrows()
):

    result = classify_location_ai(
        location=row["clean_location"],
        city=row["clean_city"],
        state=row["clean_state"]
    )

    result["record_index"] = index

    case2_ai_results.append(
        result
    )
case2_ai_execution_time = (
    time.perf_counter()
    - case2_start_time
)

## 6.31 Convert Hybrid AI Results to DataFrame

In [ ]:
case2_ai_results_df = (
    pd.DataFrame(
        case2_ai_results
    )
)
display(
    case2_ai_results_df.head()
)

,prediction,raw_output,success,execution_time_seconds,input_tokens,output_tokens,total_tokens,error,record_index
0,Metro,Metro,True,0.295558,127,1,128,None,1


## 6.32 Add AI Fallback Predictions

In [ ]:
for _, result in ( case2_ai_results_df .iterrows() ):
  record_index = int( result[ "record_index" ] )
  prediction = result[ "prediction" ]
  case2_output_df.loc[ record_index, "location_type" ] = prediction

  case2_output_df.loc[
      record_index, "classification_method" ] = (
          "AI Fallback"
          if pd.notna( prediction )
          else "Unresolved" )

## 6.33 Calculate Total Hybrid Runtime

In [ ]:
case2_total_time = (
    gccsa_execution_time +
    case2_ai_execution_time )

## 6.34 Inspect Hybrid Output Distribution

In [ ]:
case2_distribution = (
    case2_output_df[
        "location_type"
    ]
    .value_counts(
        dropna=False
    )
)
display(
    case2_distribution
    .to_frame(
        "record_count"
    )
)

,record_count
location_type,
Metro,20


## 6.35 Inspect Hybrid Processing Methods

In [ ]:
classification_method_distribution = (
    case2_output_df[
        "classification_method"
    ]
    .value_counts(
        dropna=False
    )
)
display(
    classification_method_distribution
    .to_frame(
        "record_count"
    )
)

,record_count
classification_method,
GCCSA,19
AI Fallback,1


## 6.36 Calculate Hybrid Token Usage

In [ ]:
case2_input_tokens = (
    case2_ai_results_df[
        "input_tokens"
    ].sum()
    if len(
        case2_ai_results_df
    ) > 0
    else 0
)
case2_output_tokens = (
    case2_ai_results_df[
        "output_tokens"
    ].sum()
    if len(
        case2_ai_results_df
    ) > 0
    else 0
)
case2_total_tokens = (
    case2_ai_results_df[
        "total_tokens"
    ].sum()
    if len(
        case2_ai_results_df
    ) > 0
    else 0
)

## 6.37 Calculate Hybrid Valid Output Rate

In [ ]:
case2_classified_count = (
    case2_output_df[
        "location_type"
    ]
    .notna()
    .sum()
)
case2_valid_output_rate = (
    case2_classified_count
    / len(case2_output_df)
    * 100
)

## 6.38 Create Case 2 Metrics

In [ ]:
case2_metrics = {
    "case":
        "Case 2",

    "approach":
        "GCCSA + AI Fallback",

    "records_processed":
        len(case2_output_df),

    "classified_records":
        int(
            case2_classified_count
        ),

    "unresolved_records":
        int(
            len(case2_output_df)
            - case2_classified_count
        ),

    "valid_output_rate_pct":
        round(
            case2_valid_output_rate,
            2
        ),

    "deterministic_records":
        int(
            deterministic_classified_count
        ),

    "deterministic_coverage_pct":
        round(
            deterministic_coverage_pct,
            2
        ),

    "ai_fallback_records":
        int(
            fallback_count
        ),

    "ai_fallback_rate_pct":
        round(
            fallback_pct,
            2
        ),

    "ai_calls":
        len(
            case2_ai_results_df
        ),

    "total_execution_time_seconds":
        case2_total_time,

    "avg_time_per_record_seconds":
        (
            case2_total_time
            / len(case2_output_df)
        ),

    "input_tokens":
        int(
            case2_input_tokens
        ),

    "output_tokens":
        int(
            case2_output_tokens
        ),

    "total_tokens":
        int(
            case2_total_tokens
        )
}
display(
    pd.DataFrame(
        case2_metrics.items(),
        columns=[
            "Metric",
            "Value"
        ]
    )
)

,Metric,Value
0,case,Case 2
1,approach,GCCSA + AI Fallback
2,records_processed,20
3,classified_records,20
4,unresolved_records,0
5,valid_output_rate_pct,100.0
6,deterministic_records,19
7,deterministic_coverage_pct,95.0
8,ai_fallback_records,1
9,ai_fallback_rate_pct,5.0


## 6.39 Compare AI-Only and Hybrid Predictions

Join both predictions for the same records.

In [ ]:
location_comparison_df = (
    location_sample_df[
        [
            "clean_location",
            "clean_city",
            "clean_state"
        ]
    ]
    .copy()
)
location_comparison_df[
    "ai_only"
] = (
    case1_output_df[
        "location_type"
    ]
)
location_comparison_df[
    "hybrid"
] = (
    case2_output_df[
        "location_type"
    ]
)

## 6.40 Calculate Agreement Between Cases

Agreement is not the same as accuracy, but it is useful for identifying records where the two methods behave differently.

In [ ]:
location_comparison_df[
    "agree"
] = (
    location_comparison_df[
        "ai_only"
    ]
    ==
    location_comparison_df[
        "hybrid"
    ]
)
agreement_rate = (
    location_comparison_df[
        "agree"
    ]
    .mean()
    * 100
)
print(
    f"AI-only vs Hybrid agreement: "
    f"{agreement_rate:.2f}%"
)

AI-only vs Hybrid agreement: 100.00%


## 6.41 Inspect Disagreements

These disagreements should be reviewed when creating the manually validated accuracy sample.

In [ ]:
display(
    location_comparison_df.loc[
        ~location_comparison_df[ "agree" ] ].
    head(50) )

,clean_location,clean_city,clean_state,ai_only,hybrid,agree


## 6.42 Create a Manual Validation Sample

True accuracy cannot be calculated simply by comparing one automated method against another.

Create a reproducible validation sample for manual review.


In [ ]:
LOCATION_VALIDATION_SAMPLE_SIZE = min(
    50,
    len(
        location_comparison_df
    )
)
location_validation_df = (
    location_comparison_df
    .sample(
        n=LOCATION_VALIDATION_SAMPLE_SIZE,
        random_state=RANDOM_STATE
    )
    .copy()
)

## 6.43 Add Manual Ground-Truth Column

The analyst should populate only:

Metro
Regional
Remote

after reviewing the actual location.

In [ ]:
location_validation_df[
    "manual_location_type"
] = None

manual_location_labels = {
    0: "Metro",
    1: "Metro",
    2: "Metro",
    3: "Metro",
    4: "Metro",
    5: "Metro",
    6: "Metro",
    7: "Metro",
    8: "Metro",
    9: "Metro",
    10: "Metro",
    11: "Metro",
    12: "Metro",
    13: "Metro",
    14: "Metro",
    15: "Metro",
    16: "Metro",
    17: "Metro",
    18: "Metro",
    19: "Metro",
}

for index, label in manual_location_labels.items():

    location_validation_df.loc[
        index,
        "manual_location_type"
    ] = label


display(
    location_validation_df
)

,clean_location,clean_city,clean_state,ai_only,hybrid,agree,manual_location_type
0,Melbourne City Centre VIC,Melbourne City Centre,None,Metro,Metro,True,Metro
17,Sydney NSW,Sydney,None,Metro,Metro,True,Metro
15,Sydney NSW,Sydney,None,Metro,Metro,True,Metro
1,Canberra ACT,Canberra,None,Metro,Metro,True,Metro
8,Sydney NSW,Sydney,None,Metro,Metro,True,Metro
5,Sydney NSW,Sydney,None,Metro,Metro,True,Metro
11,Sydney NSW,Sydney,None,Metro,Metro,True,Metro
3,Brisbane QLD,Brisbane,None,Metro,Metro,True,Metro
18,Melbourne City Centre VIC,Melbourne City Centre,None,Metro,Metro,True,Metro
16,Sydney NSW,Sydney,None,Metro,Metro,True,Metro


## 6.44 Accuracy Function

Once the manual labels have been completed, use the same function for both cases.

In [ ]:
def calculate_location_accuracy( validation_df, prediction_column ):
  labelled = (
      validation_df[
          validation_df[ "manual_location_type" ]
          .isin( VALID_LOCATION_TYPES ) ] .copy() )

  if len(labelled) == 0:
    return np.nan

  return ( (
      labelled[ prediction_column ] == labelled[ "manual_location_type" ] )
   .mean() * 100 )

## 6.45 Calculate Accuracy After Manual Labelling

Run these cells after manual_location_type has been populated.

In [ ]:
case1_accuracy = (
    calculate_location_accuracy(
        location_validation_df,
        "ai_only"
    )
)
case2_accuracy = (
    calculate_location_accuracy(
        location_validation_df,
        "hybrid"
    )
)
print(
    "AI-only accuracy:",
    case1_accuracy
)

print(
    "Hybrid accuracy:",
    case2_accuracy
)

AI-only accuracy: 100.0
Hybrid accuracy: 100.0


## 6.46 Compare Resource Usage

In [ ]:
location_metrics_df = pd.DataFrame([
    {
        "Case":
            "Case 1",

        "Approach":
            "AI Only",

        "Records":
            case1_metrics[
                "records_processed"
            ],

        "Valid Output %":
            case1_metrics[
                "valid_output_rate_pct"
            ],

        "AI Calls":
            case1_metrics[
                "ai_calls"
            ],

        "AI Fallback %":
            100.0,

        "Execution Time (s)":
            case1_metrics[
                "total_execution_time_seconds"
            ],

        "Avg Time / Record (s)":
            case1_metrics[
                "avg_time_per_record_seconds"
            ],

        "Input Tokens":
            case1_metrics[
                "input_tokens"
            ],

        "Output Tokens":
            case1_metrics[
                "output_tokens"
            ],

        "Total Tokens":
            case1_metrics[
                "total_tokens"
            ],

        "Accuracy %":
            case1_accuracy
    },

    {
        "Case":
            "Case 2",

        "Approach":
            "GCCSA + AI Fallback",

        "Records":
            case2_metrics[
                "records_processed"
            ],

        "Valid Output %":
            case2_metrics[
                "valid_output_rate_pct"
            ],

        "AI Calls":
            case2_metrics[
                "ai_calls"
            ],

        "AI Fallback %":
            case2_metrics[
                "ai_fallback_rate_pct"
            ],

        "Execution Time (s)":
            case2_metrics[
                "total_execution_time_seconds"
            ],

        "Avg Time / Record (s)":
            case2_metrics[
                "avg_time_per_record_seconds"
            ],

        "Input Tokens":
            case2_metrics[
                "input_tokens"
            ],

        "Output Tokens":
            case2_metrics[
                "output_tokens"
            ],

        "Total Tokens":
            case2_metrics[
                "total_tokens"
            ],

        "Accuracy %":
            case2_accuracy
    }
])
display(
    location_metrics_df
)

,Case,Approach,Records,Valid Output %,AI Calls,AI Fallback %,Execution Time (s),Avg Time / Record (s),Input Tokens,Output Tokens,Total Tokens,Accuracy %
0,Case 1,AI Only,20,100.0,20,100.0,5.277819,0.263891,2561,20,2581,100.0
1,Case 2,GCCSA + AI Fallback,20,100.0,1,5.0,0.373134,0.018657,127,1,128,100.0


## 6.47 Calculate AI Call Reduction

One of the major expected advantages of the hybrid method is reducing AI requests

In [ ]:
ai_call_reduction_pct = (
    (
        case1_metrics[
            "ai_calls"
        ]
        -
        case2_metrics[
            "ai_calls"
        ]
    )
    /
    case1_metrics[
        "ai_calls"
    ]
    * 100
)
print(
    f"AI call reduction using hybrid approach: "
    f"{ai_call_reduction_pct:.2f}%"
)

AI call reduction using hybrid approach: 95.00%


## 6.48 Calculate Token Reduction

In [ ]:
if (
    case1_metrics[
        "total_tokens"
    ] > 0
):

    token_reduction_pct = (
        (
            case1_metrics[
                "total_tokens"
            ]
            -
            case2_metrics[
                "total_tokens"
            ]
        )
        /
        case1_metrics[
            "total_tokens"
        ]
        * 100
    )

else:

    token_reduction_pct = np.nan
print(
    f"Token reduction using hybrid approach: "
    f"{token_reduction_pct:.2f}%"
)

Token reduction using hybrid approach: 95.04%


## 6.49 Store Location Experiment Results

In [ ]:
LOCATION_EXPERIMENT_RESULTS = {
    "case_1": case1_metrics,
    "case_2": case2_metrics,
    "agreement_rate_pct": agreement_rate,
    "ai_call_reduction_pct": ai_call_reduction_pct,
    "token_reduction_pct": token_reduction_pct,
    "case_1_accuracy_pct": case1_accuracy,
    "case_2_accuracy_pct": case2_accuracy }

## 6.50 Section 6 Outcome

Section 6 evaluates the two required approaches for determining location_type.

**Case 1 — AI Only**

Every record is processed by the AI model and classified as:

Metro,
Regional,
or Remote.

This approach provides broad semantic flexibility but requires one AI request for every processed job.

**Case 2 — GCCSA + AI Fallback**

The hybrid approach first attempts deterministic spatial classification using the supplied GCCSA 2026 polygons.

Records with usable coordinates are spatially joined against the GCCSA geography.

For this experimental rule:

Greater ... GCCSA areas are mapped to Metro;
Rest of ... areas are mapped to Regional;
unresolved, non-geographic, or unmatched records are passed to AI.

AI is therefore used only when the deterministic stage cannot provide a usable classification.

Evaluation Metrics

The two cases are compared using:

records processed;
valid classification rate;
unresolved records;
deterministic coverage;
AI fallback rate;
number of AI calls;
total runtime;
average runtime per record;
input tokens;
output tokens;
total tokens;
AI-only versus hybrid prediction agreement;
and manually validated accuracy when ground-truth labels are supplied.

The hybrid experiment also quantifies:

AI-call reduction;
token reduction;
and deterministic coverage.
Important Interpretation

Agreement between Case 1 and Case 2 does not establish accuracy.

A manually reviewed sample is therefore created so that both approaches can ultimately be evaluated against the same independent ground truth.

The deterministic GCCSA rule also has an important limitation: GCCSA distinguishes Greater Capital City Statistical Areas from the remainder of each state, but it does not by itself provide a complete semantic distinction between all Australian metropolitan, regional, and remote locations.

The resulting accuracy results should therefore be considered together with resource efficiency when choosing the production approach.

No duplicate records are removed in Section 6.

The next section will perform the dedicated deduplication experiments, beginning with source-level jobKey removal and then comparing incoming daily_scrape_dev records against the 48-hour vw_dca_jobboard dataset using the three required deduplication approaches.

# 7. Deduplication Experiments

This section evaluates the required deduplication strategies used to determine whether incoming records from daily_scrape_dev already exist in the recent vw_dca_jobboard dataset.

The deduplication process contains two stages:

**Stage 1 — Source-Level jobKey Deduplication**

Duplicate jobKey values are removed from the incoming Indeed dataset before any cross-dataset comparison is performed.

**Stage 2 — Cross-Dataset Deduplication Experiments**

The remaining incoming source records are compared against the 48-hour vw_dca_jobboard comparison dataset using three approaches:

**Case 1**

Company + Job Title + Location

**Case 2**

Company + Job Title + Location + Date Similarity

**Case 3**

Company + Job Title + Location + descriptionTextHash

For Case 3, descriptionTextHash is generated during this experiment from the cleaned job description. It is not treated as an existing source or target field.

All three cases use the same cleaned and jobKey-deduplicated source dataset so that their results can be compared fairly.

The project brief requires incoming source records to be checked against recent production records within a two-day lookback before staging.

## 7.1 Prepare the Deduplication Datasets

Create experiment-specific copies so that the cleaned datasets from Section 5 remain unchanged.

In [ ]:
dedup_source_df = (
    source_clean_df
    .copy(deep=True)
)
dedup_target_df = (
    target_clean_df
    .copy(deep=True)
)
print(
    "Source records before jobKey deduplication:",
    len(dedup_source_df)
)

print(
    "Target records in 48-hour window:",
    len(dedup_target_df)
)

Source records before jobKey deduplication: 20
Target records in 48-hour window: 370


## 7.2 Validate Required Source Fields

The source dataset requires:

* cleaned jobKey,
* cleaned company,
* cleaned title,
* cleaned location,
* cleaned posting date,
* cleaned description.

In [ ]:
SOURCE_DEDUP_REQUIRED_FIELDS = [
    "clean_jobKey",
    "clean_company",
    "clean_title",
    "clean_location",
    "clean_posted_date",
    "clean_description",
    "match_company",
    "match_title",
    "match_location"
]
source_dedup_field_check = pd.DataFrame({
    "field":
        SOURCE_DEDUP_REQUIRED_FIELDS,

    "exists":
        [
            field in dedup_source_df.columns
            for field in SOURCE_DEDUP_REQUIRED_FIELDS
        ]
})
display(
    source_dedup_field_check
)
assert source_dedup_field_check["exists"].all(), (
    "One or more required source deduplication "
    "fields are missing."
)

,field,exists
0,clean_jobKey,True
1,clean_company,True
2,clean_title,True
3,clean_location,True
4,clean_posted_date,True
5,clean_description,True
6,match_company,True
7,match_title,True
8,match_location,True


## 7.3 Validate Required Target Fields

In [ ]:
TARGET_DEDUP_REQUIRED_FIELDS = [
    "clean_company",
    "clean_title",
    "clean_location",
    "clean_posted_date",
    "clean_description",
    "match_company",
    "match_title",
    "match_location"
]
target_dedup_field_check = pd.DataFrame({
    "field":
        TARGET_DEDUP_REQUIRED_FIELDS,

    "exists":
        [
            field in dedup_target_df.columns
            for field in TARGET_DEDUP_REQUIRED_FIELDS
        ]
})
display(
    target_dedup_field_check
)
assert target_dedup_field_check["exists"].all(), (
    "One or more required target deduplication "
    "fields are missing."
)

,field,exists
0,clean_company,True
1,clean_title,True
2,clean_location,True
3,clean_posted_date,True
4,clean_description,True
5,match_company,True
6,match_title,True
7,match_location,True


## 7.4 Stage 1 — Remove Duplicate jobKey Records

Before testing Cases 1–3, duplicate non-null jobKey records are removed from the incoming Indeed dataset.

Missing jobKey records are retained because they may still be evaluated using the cross-dataset deduplication methods.

### 7.4.1 Measure Source Records Before Removal

In [ ]:
jobkey_rows_before = ( len(dedup_source_df) )

### 7.4.2 Identify Duplicate Non-Null jobKey Records

In [ ]:
jobkey_remove_mask = (
    dedup_source_df[
        "clean_jobKey"
    ].notna()
    &
    dedup_source_df[
        "clean_jobKey"
    ].duplicated(
        keep="first"
    )
)
print(
    "Duplicate jobKey records to remove:",
    jobkey_remove_mask.sum()
)

Duplicate jobKey records to remove: 0


### 7.4.3 Remove Duplicate jobKey Records

Measure the processing time for this source-level operation.

In [ ]:
jobkey_start_time = (
    time.perf_counter()
)
dedup_source_df = (
    dedup_source_df.loc[
        ~jobkey_remove_mask
    ]
    .copy()
    .reset_index(drop=True)
)
jobkey_execution_time = (
    time.perf_counter()
    - jobkey_start_time
)

### 7.4.4 Validate jobKey Removal

In [ ]:
remaining_duplicate_jobkeys = (
    dedup_source_df.loc[
        dedup_source_df[
            "clean_jobKey"
        ].notna(),
        "clean_jobKey"
    ]
    .duplicated(
        keep=False
    )
    .sum()
)
assert remaining_duplicate_jobkeys == 0, (
    "Duplicate non-null jobKey values remain."
)
print(
    "Source rows before:",
    jobkey_rows_before
)

print(
    "Source rows after:",
    len(dedup_source_df)
)

print(
    "jobKey records removed:",
    jobkey_rows_before
    - len(dedup_source_df)
)

print(
    f"jobKey processing time: "
    f"{jobkey_execution_time:.6f} seconds"
)

Source rows before: 20
Source rows after: 20
jobKey records removed: 0
jobKey processing time: 0.003253 seconds


### 7.4.5 JobKey Duplicate Check Summary

In [ ]:
jobkey_duplicate_summary_df = pd.DataFrame({
    "Metric": [
        "Total source rows",
        "Missing jobKey",
        "Unique jobKey",
        "Rows in duplicate jobKey groups",
        "Duplicate rows removed"
    ],
    "Value": [
        len(source_clean_df),
        source_clean_df["clean_jobKey"].isna().sum(),
        source_clean_df["clean_jobKey"].nunique(dropna=True),
        (
            source_clean_df["clean_jobKey"].notna()
            &
            source_clean_df["clean_jobKey"].duplicated(
                keep=False
            )
        ).sum(),
        production_rows_before_jobkey
        - len(production_df)
    ]
})

display(jobkey_duplicate_summary_df)

,Metric,Value
0,Total source rows,20
1,Missing jobKey,0
2,Unique jobKey,20
3,Rows in duplicate jobKey groups,0
4,Duplicate rows removed,0


## 7.5 Preserve the Common Test Input

All three deduplication cases must use exactly the same source records.

In [ ]:
dedup_test_source_df = (
    dedup_source_df
    .copy(deep=True)
)
print(
    "Common experiment source records:",
    len(dedup_test_source_df)
)

Common experiment source records: 20


## 7.6 Define Valid Match-Field Logic

A deduplication key should not be created when required values are missing.

Otherwise multiple incomplete records could incorrectly produce the same key.

In [ ]:
def has_required_match_values( row, fields ):
  return all(
      pd.notna( row[field] ) and
      str( row[field] ).strip() != "" for field in fields )

## 7.7 Case 1 — Company + Job Title + Location

Case 1 uses the cleaned and normalised fields:

company + title + location

The source record is considered a duplicate when the same combination exists in the 48-hour target dataset.

### 7.7.1 Define Case 1 Fields

In [ ]:
CASE1_FIELDS = [ "match_company", "match_title", "match_location" ]

### 7.7.2 Build the Source Case 1 Key

In [ ]:
case1_source_df = (
    dedup_test_source_df
    .copy()
)
case1_source_valid = (
    case1_source_df[
        CASE1_FIELDS
    ]
    .notna()
    .all(axis=1)
)
case1_source_df[
    "dedup_key"
] = None
case1_source_df.loc[
    case1_source_valid,
    "dedup_key"
] = (
    case1_source_df.loc[
        case1_source_valid,
        CASE1_FIELDS
    ]
    .astype(str)
    .agg(
        "|".join,
        axis=1
    )
)

### 7.7.3 Build the Target Case 1 Key

In [ ]:
case1_target_df = (
    dedup_target_df
    .copy()
)
case1_target_valid = (
    case1_target_df[
        CASE1_FIELDS
    ]
    .notna()
    .all(axis=1)
)
case1_target_df[
    "dedup_key"
] = None
case1_target_df.loc[
    case1_target_valid,
    "dedup_key"
] = (
    case1_target_df.loc[
        case1_target_valid,
        CASE1_FIELDS
    ]
    .astype(str)
    .agg(
        "|".join,
        axis=1
    )
)

## 7.8 Execute Case 1

Measure processing time and memory usage.

In [ ]:
gc.collect()
case1_memory_before = (
    get_memory_usage_mb()
)
case1_start_time = (
    time.perf_counter()
)
case1_target_keys = set(
    case1_target_df[
        "dedup_key"
    ]
    .dropna()
)
case1_source_df[
    "is_duplicate"
] = (
    case1_source_df[
        "dedup_key"
    ]
    .notna()
    &
    case1_source_df[
        "dedup_key"
    ]
    .isin(
        case1_target_keys
    )
)
case1_execution_time = (
    time.perf_counter()
    - case1_start_time
)
case1_memory_after = (
    get_memory_usage_mb()
)

## 7.9 Case 1 Results

In [ ]:
case1_duplicate_count = (
    case1_source_df[
        "is_duplicate"
    ]
    .sum()
)
case1_duplicate_rate = (
    case1_duplicate_count
    / len(case1_source_df)
    * 100
)
print(
    "Case 1 duplicates identified:",
    case1_duplicate_count
)

print(
    f"Case 1 duplicate rate: "
    f"{case1_duplicate_rate:.2f}%"
)

print(
    f"Case 1 execution time: "
    f"{case1_execution_time:.6f} seconds"
)

Case 1 duplicates identified: 0
Case 1 duplicate rate: 0.00%
Case 1 execution time: 0.002601 seconds


## 7.10 Inspect Case 1 Matches

In [ ]:
display(
    case1_source_df.loc[
        case1_source_df[ "is_duplicate" ], [
            "clean_jobKey", "clean_company",
            "clean_title", "clean_location",
            "clean_posted_date" ] ].head(50) )

,clean_jobKey,clean_company,clean_title,clean_location,clean_posted_date


## 7.11 Case 2 — Company + Job Title + Location + Date Similarity

Case 2 first identifies candidates with the same:

* company,
* job title,
* location.

Posting dates are then compared using a percentage similarity score.

The project's production lookback is 48 hours.

A date difference of:

* 0 hours = 100% similarity,
* 24 hours = 50% similarity,
* 48 hours or more = 0% similarity.

This provides a transparent percentage-based measure tied directly to the two-day deduplication window.

## 7.12 Define the Date Similarity Function

In [ ]:
def calculate_date_similarity( source_date, target_date, max_hours=48 ):
  if ( pd.isna(source_date) or pd.isna(target_date) ):
    return np.nan

  difference_hours = abs( ( source_date - target_date ).total_seconds() ) / 3600

  similarity = ( 1 - min( difference_hours, max_hours ) / max_hours ) * 100

  return max( 0.0, similarity )

## 7.13 Validate the Date Similarity Function

In [ ]:
example_date = pd.Timestamp(
    "2026-08-20 12:00:00",
    tz="UTC"
)
print(
    "0 hour difference:",
    calculate_date_similarity(
        example_date,
        example_date
    )
)
print(
    "24 hour difference:",
    calculate_date_similarity(
        example_date,
        example_date
        + pd.Timedelta(
            hours=24
        )
    )
)
print(
    "48 hour difference:",
    calculate_date_similarity(
        example_date,
        example_date
        + pd.Timedelta(
            hours=48
        )
    )
)

0 hour difference: 100.0
24 hour difference: 50.0
48 hour difference: 0.0


## 7.14 Create Case 2 Candidate Pairs

Rather than comparing every source record against every target record, first match on:

Company + Title + Location

This reduces unnecessary date comparisons.

In [ ]:
case2_source_df = (
    dedup_test_source_df
    .copy()
)
case2_target_df = (
    dedup_target_df
    .copy()
)

### 7.14.1 Create the Base Key

In [ ]:
case2_source_df[
    "base_key"
] = (
    case2_source_df[
        "match_company"
    ]
    + "|"
    + case2_source_df[
        "match_title"
    ]
    + "|"
    + case2_source_df[
        "match_location"
    ]
)
case2_target_df[
    "base_key"
] = (
    case2_target_df[
        "match_company"
    ]
    + "|"
    + case2_target_df[
        "match_title"
    ]
    + "|"
    + case2_target_df[
        "match_location"
    ]
)

## 7.15 Execute Case 2 Candidate Matching

In [ ]:
gc.collect()
case2_memory_before = (
    get_memory_usage_mb()
)
case2_start_time = (
    time.perf_counter()
)
case2_candidates_df = (
    case2_source_df[
        [
            "clean_jobKey",
            "base_key",
            "clean_company",
            "clean_title",
            "clean_location",
            "clean_posted_date"
        ]
    ]
    .merge(
        case2_target_df[
            [
                "base_key",
                "clean_posted_date"
            ]
        ],
        on="base_key",
        how="inner",
        suffixes=(
            "_source",
            "_target"
        )
    )
)
print(
    "Case 2 candidate source-target pairs:",
    len(case2_candidates_df)
)

Case 2 candidate source-target pairs: 0


## 7.16 Calculate Date Similarity

In [ ]:
case2_candidates_df[
    "date_similarity_pct"
    ] = (
        case2_candidates_df .apply(
            lambda row:
            calculate_date_similarity(
                row[ "clean_posted_date_source" ],
                row[ "clean_posted_date_target" ] ),
            axis=1 ) )

## 7.17 Inspect Date Similarity Distribution

In [ ]:
display(
    case2_candidates_df[ "date_similarity_pct" ]
    .describe()
    .to_frame( "date_similarity_pct" ) )

,date_similarity_pct
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


## 7.18 Evaluate Date Similarity Thresholds

Rather than choosing an arbitrary threshold without evidence, test several thresholds.

In [ ]:
DATE_SIMILARITY_THRESHOLDS = [
    100,
    75,
    50,
    25
]

### 7.18.1 Calculate Source Matches at Each Threshold

In [ ]:
case2_threshold_results = []
for threshold in DATE_SIMILARITY_THRESHOLDS:

    matched_source_keys = (
        case2_candidates_df.loc[
            case2_candidates_df[
                "date_similarity_pct"
            ] >= threshold,
            "clean_jobKey"
        ]
        .dropna()
        .unique()
    )

    match_count = (
        case2_source_df[
            "clean_jobKey"
        ]
        .isin(
            matched_source_keys
        )
        .sum()
    )

    case2_threshold_results.append({
        "threshold_pct":
            threshold,

        "matched_source_records":
            int(
                match_count
            ),

        "match_rate_pct":
            round(
                match_count
                / len(
                    case2_source_df
                )
                * 100,
                2
            )
    })
case2_threshold_df = pd.DataFrame(
    case2_threshold_results
)
display(
    case2_threshold_df
)

,threshold_pct,matched_source_records,match_rate_pct
0,100,0,0.0
1,75,0,0.0
2,50,0,0.0
3,25,0,0.0


## 7.19 Select the Working Case 2 Threshold

The final threshold should ultimately be supported by manual validation.

For the experiment, define the working threshold explicitly so it can easily be changed.

At 50%, source and target posting dates may differ by up to approximately 24 hours.

In [ ]:
DATE_SIMILARITY_THRESHOLD = 50

## 7.20 Generate Final Case 2 Duplicate Decisions

A source record is a Case 2 duplicate if any target candidate reaches the configured date-similarity threshold.

In [ ]:
case2_matched_jobkeys = (
    case2_candidates_df.loc[
        case2_candidates_df[
            "date_similarity_pct"
        ]
        >= DATE_SIMILARITY_THRESHOLD,
        "clean_jobKey"
    ]
    .dropna()
    .unique()
)
case2_source_df[
    "is_duplicate"
] = (
    case2_source_df[
        "clean_jobKey"
    ]
    .isin(
        case2_matched_jobkeys
    )
)

## 7.21 Capture Case 2 Runtime and Memory

In [ ]:
case2_execution_time = (
    time.perf_counter()
    - case2_start_time
)
case2_memory_after = (
    get_memory_usage_mb()
)

## 7.22 Case 2 Results

In [ ]:
case2_duplicate_count = (
    case2_source_df[
        "is_duplicate"
    ]
    .sum()
)
case2_duplicate_rate = (
    case2_duplicate_count
    / len(case2_source_df)
    * 100
)
print(
    "Case 2 duplicates identified:",
    case2_duplicate_count
)

print(
    f"Case 2 duplicate rate: "
    f"{case2_duplicate_rate:.2f}%"
)

print(
    f"Date similarity threshold: "
    f"{DATE_SIMILARITY_THRESHOLD}%"
)

print(
    f"Case 2 execution time: "
    f"{case2_execution_time:.6f} seconds"
)

Case 2 duplicates identified: 0
Case 2 duplicate rate: 0.00%
Date similarity threshold: 50%
Case 2 execution time: 0.080028 seconds


## 7.23 Inspect Highest Date Similarity Per Source Candidate

In [ ]:
case2_best_matches_df = (
    case2_candidates_df
    .groupby(
        "clean_jobKey",
        dropna=False
    )[
        "date_similarity_pct"
    ]
    .max()
    .reset_index()
    .rename(
        columns={
            "date_similarity_pct":
                "best_date_similarity_pct"
        }
    )
)
display(
    case2_best_matches_df
    .sort_values(
        "best_date_similarity_pct",
        ascending=False
    )
    .head(50)
)

,clean_jobKey,best_date_similarity_pct


## 7.24 Case 3 — Company + Job Title + Location + descriptionTextHash

Case 3 evaluates content similarity using a generated deterministic hash.

descriptionTextHash does not exist in the source or target dataset before this experiment.

It is generated here from the cleaned job description using exactly the same hashing procedure for both datasets.

## 7.25 Define the Description Hash Function

SHA-256 is used to generate a stable deterministic hash.

In [ ]:
def create_description_text_hash( description ):
  if pd.isna(description):
    return None

  description = str( description ).strip()

  if not description:
    return None

  return hashlib.sha256(
      description.encode( "utf-8" )
      ).hexdigest()

## 7.26 Create Case 3 Working Copies

In [ ]:
case3_source_df = (
    dedup_test_source_df
    .copy()
)
case3_target_df = (
    dedup_target_df
    .copy()
)

## 7.27 Generate Source descriptionTextHash

In [ ]:
case3_hash_start_time = (
    time.perf_counter()
)
case3_source_df[
    "descriptionTextHash"
] = (
    case3_source_df[
        "clean_description"
    ]
    .apply(
        create_description_text_hash
    )
)

## 7.28 Generate Target descriptionTextHash

In [ ]:
case3_target_df[
    "descriptionTextHash"
] = (
    case3_target_df[
        "clean_description"
    ]
    .apply(
        create_description_text_hash
    )
)
case3_hash_generation_time = (
    time.perf_counter()
    - case3_hash_start_time
)

## 7.29 Inspect Generated Hash Coverage

In [ ]:
case3_hash_coverage_df = pd.DataFrame({
    "Dataset": [
        "Source",
        "Target"
    ],

    "Rows": [
        len(case3_source_df),
        len(case3_target_df)
    ],

    "Hash Available": [
        case3_source_df[
            "descriptionTextHash"
        ].notna().sum(),

        case3_target_df[
            "descriptionTextHash"
        ].notna().sum()
    ],

    "Hash Missing": [
        case3_source_df[
            "descriptionTextHash"
        ].isna().sum(),

        case3_target_df[
            "descriptionTextHash"
        ].isna().sum()
    ]
})
display(
    case3_hash_coverage_df
)

,Dataset,Rows,Hash Available,Hash Missing
0,Source,20,20,0
1,Target,370,370,0


## 7.30 Build the Case 3 Source Key

Required fields are:

Company + Title + Location + descriptionTextHash

In [ ]:
case3_source_valid = (
    case3_source_df[
        [
            "match_company",
            "match_title",
            "match_location",
            "descriptionTextHash"
        ]
    ]
    .notna()
    .all(axis=1)
)
case3_source_df[
    "dedup_key"
] = None
case3_source_df.loc[
    case3_source_valid,
    "dedup_key"
] = (
    case3_source_df.loc[
        case3_source_valid,
        [
            "match_company",
            "match_title",
            "match_location",
            "descriptionTextHash"
        ]
    ]
    .astype(str)
    .agg(
        "|".join,
        axis=1
    )
)

## 7.31 Build the Case 3 Target Key

In [ ]:
case3_target_valid = (
    case3_target_df[
        [
            "match_company",
            "match_title",
            "match_location",
            "descriptionTextHash"
        ]
    ]
    .notna()
    .all(axis=1)
)
case3_target_df[
    "dedup_key"
] = None
case3_target_df.loc[
    case3_target_valid,
    "dedup_key"
] = (
    case3_target_df.loc[
        case3_target_valid,
        [
            "match_company",
            "match_title",
            "match_location",
            "descriptionTextHash"
        ]
    ]
    .astype(str)
    .agg(
        "|".join,
        axis=1
    )
)

## 7.32 Execute Case 3

In [ ]:
gc.collect()
case3_memory_before = (
    get_memory_usage_mb()
)
case3_start_time = (
    time.perf_counter()
)
case3_target_keys = set(
    case3_target_df[
        "dedup_key"
    ]
    .dropna()
)
case3_source_df[
    "is_duplicate"
] = (
    case3_source_df[
        "dedup_key"
    ]
    .notna()
    &
    case3_source_df[
        "dedup_key"
    ]
    .isin(
        case3_target_keys
    )
)
case3_matching_time = (
    time.perf_counter()
    - case3_start_time
)
case3_execution_time = (
    case3_hash_generation_time
    + case3_matching_time
)
case3_memory_after = (
    get_memory_usage_mb()
)

## 7.33 Case 3 Results

In [ ]:
case3_duplicate_count = (
    case3_source_df[
        "is_duplicate"
    ]
    .sum()
)
case3_duplicate_rate = (
    case3_duplicate_count
    / len(case3_source_df)
    * 100
)
print(
    "Case 3 duplicates identified:",
    case3_duplicate_count
)

print(
    f"Case 3 duplicate rate: "
    f"{case3_duplicate_rate:.2f}%"
)

print(
    f"Case 3 total execution time: "
    f"{case3_execution_time:.6f} seconds"
)

Case 3 duplicates identified: 0
Case 3 duplicate rate: 0.00%
Case 3 total execution time: 0.022554 seconds


## 7.34 Inspect Case 3 Matches

In [ ]:
display(
    case3_source_df.loc[
        case3_source_df[ "is_duplicate" ], [
            "clean_jobKey", "clean_company",
            "clean_title", "clean_location",
            "clean_posted_date", "descriptionTextHash" ]
        ].head(50) )

,clean_jobKey,clean_company,clean_title,clean_location,clean_posted_date,descriptionTextHash


## 7.35 Calculate Common Efficiency Metrics

Create a helper function to calculate throughput and average processing time.

In [ ]:
def calculate_dedup_efficiency( record_count, execution_time ):
  if execution_time <= 0:
    return {
        "avg_time_per_record_seconds": 0,
        "records_per_second": np.nan }
  return {
      "avg_time_per_record_seconds": execution_time / record_count,
      "records_per_second": record_count / execution_time
      }

## 7.36 Calculate Case 1 Efficiency

In [ ]:
case1_efficiency = (
    calculate_dedup_efficiency(
        len( case1_source_df ),
        case1_execution_time ) )

## 7.37 Calculate Case 2 Efficiency

In [ ]:
case2_efficiency = (
    calculate_dedup_efficiency(
        len( case2_source_df ),
        case2_execution_time ) )

## 7.38 Calculate Case 3 Efficiency

In [ ]:
case3_efficiency = (
    calculate_dedup_efficiency(
        len( case3_source_df ),
        case3_execution_time ) )

## 7.39 Build Initial Deduplication Comparison Table

In [ ]:
dedup_metrics_df = pd.DataFrame([
    {
        "Case":
            "Case 1",

        "Rule":
            "Company + Title + Location",

        "Records Evaluated":
            len(
                case1_source_df
            ),

        "Duplicates Identified":
            int(
                case1_duplicate_count
            ),

        "Duplicate Rate %":
            round(
                case1_duplicate_rate,
                2
            ),

        "Execution Time (s)":
            case1_execution_time,

        "Avg Time / Record (s)":
            case1_efficiency[
                "avg_time_per_record_seconds"
            ],

        "Records / Second":
            case1_efficiency[
                "records_per_second"
            ],

        "Memory Delta (MB)":
            (
                case1_memory_after
                - case1_memory_before
            )
    },

    {
        "Case":
            "Case 2",

        "Rule":
            "Company + Title + Location + Date Similarity",

        "Records Evaluated":
            len(
                case2_source_df
            ),

        "Duplicates Identified":
            int(
                case2_duplicate_count
            ),

        "Duplicate Rate %":
            round(
                case2_duplicate_rate,
                2
            ),

        "Execution Time (s)":
            case2_execution_time,

        "Avg Time / Record (s)":
            case2_efficiency[
                "avg_time_per_record_seconds"
            ],

        "Records / Second":
            case2_efficiency[
                "records_per_second"
            ],

        "Memory Delta (MB)":
            (
                case2_memory_after
                - case2_memory_before
            )
    },

    {
        "Case":
            "Case 3",

        "Rule":
            "Company + Title + Location + Description Hash",

        "Records Evaluated":
            len(
                case3_source_df
            ),

        "Duplicates Identified":
            int(
                case3_duplicate_count
            ),

        "Duplicate Rate %":
            round(
                case3_duplicate_rate,
                2
            ),

        "Execution Time (s)":
            case3_execution_time,

        "Avg Time / Record (s)":
            case3_efficiency[
                "avg_time_per_record_seconds"
            ],

        "Records / Second":
            case3_efficiency[
                "records_per_second"
            ],

        "Memory Delta (MB)":
            (
                case3_memory_after
                - case3_memory_before
            )
    }
])
display(
    dedup_metrics_df
)

,Case,Rule,Records Evaluated,Duplicates Identified,Duplicate Rate %,Execution Time (s),Avg Time / Record (s),Records / Second,Memory Delta (MB)
0,Case 1,Company + Title + Location,20,0,0.0,0.002601,0.000130,7689.048719,0.0
1,Case 2,Company + Title + Location + Date Similarity,20,0,0.0,0.080028,0.004001,249.912371,0.0
2,Case 3,Company + Title + Location + Description Hash,20,0,0.0,0.022554,0.001128,886.757872,0.0


## 7.40 Compare Agreement Between Deduplication Cases

Create a single dataframe containing each case's decision for the same source records.

In [ ]:
dedup_comparison_df = (
    dedup_test_source_df[
        [
            "clean_jobKey",
            "clean_company",
            "clean_title",
            "clean_location",
            "clean_posted_date"
        ]
    ]
    .copy()
)
dedup_comparison_df[
    "case1_duplicate"
] = (
    case1_source_df[
        "is_duplicate"
    ].values
)
dedup_comparison_df[
    "case2_duplicate"
] = (
    case2_source_df[
        "is_duplicate"
    ].values
)
dedup_comparison_df[
    "case3_duplicate"
] = (
    case3_source_df[
        "is_duplicate"
    ].values
)

## 7.41 Inspect Decision Patterns

This helps identify records for which the different approaches disagree.

In [ ]:
dedup_decision_patterns = (
    dedup_comparison_df[
        [
            "case1_duplicate",
            "case2_duplicate",
            "case3_duplicate"
        ]
    ]
    .value_counts()
    .reset_index(
        name="record_count"
    )
)
display(
    dedup_decision_patterns
)

,case1_duplicate,case2_duplicate,case3_duplicate,record_count
0,False,False,False,20


## 7.42 Identify Disagreement Records

In [ ]:
dedup_comparison_df[
    "all_cases_agree"
] = (
    (
        dedup_comparison_df[
            "case1_duplicate"
        ]
        ==
        dedup_comparison_df[
            "case2_duplicate"
        ]
    )
    &
    (
        dedup_comparison_df[
            "case1_duplicate"
        ]
        ==
        dedup_comparison_df[
            "case3_duplicate"
        ]
    )
)
print(
    "Records where all cases agree:",
    dedup_comparison_df[
        "all_cases_agree"
    ].sum()
)

print(
    "Records where cases disagree:",
    (
        ~dedup_comparison_df[
            "all_cases_agree"
        ]
    ).sum()
)

Records where all cases agree: 20
Records where cases disagree: 0


## 7.43 Inspect Disagreements

In [ ]:
display(
    dedup_comparison_df.loc[
        ~dedup_comparison_df[ "all_cases_agree" ]
        ].head(50) )

,clean_jobKey,clean_company,clean_title,clean_location,clean_posted_date,case1_duplicate,case2_duplicate,case3_duplicate,all_cases_agree


## 7.44 Create Manual Validation Dataset

The number of records removed by a method does not establish whether that method is correct.

A manual validation sample is required to calculate:

* precision,
* recall,
* F1 score,
* accuracy,
* false positives,
* false negatives.

The validation sample should contain both:

* records where approaches agree;
* records where approaches disagree.

### 7.44.1 Create Disagreement Sample

In [ ]:
validation_disagreement_df = (
    dedup_comparison_df.loc[
        ~dedup_comparison_df[ "all_cases_agree" ]
        ] .copy() )

### 7.44.2 Create Agreement Sample

In [ ]:
DEDUP_VALIDATION_AGREEMENT_SAMPLE = min(
    50,
    dedup_comparison_df[
        "all_cases_agree"
    ].sum()
)
validation_agreement_df = (
    dedup_comparison_df.loc[
        dedup_comparison_df[
            "all_cases_agree"
        ]
    ]
    .sample(
        n=DEDUP_VALIDATION_AGREEMENT_SAMPLE,
        random_state=RANDOM_STATE
    )
    .copy()
)

### 7.44.3 Combine the Validation Sample

In [ ]:
dedup_validation_df = (
    pd.concat( [
        validation_disagreement_df,
        validation_agreement_df
        ], ignore_index=True ) )

## 7.45 Add Manual Ground Truth

The analyst should manually label each sampled source record as:

* True — duplicate already exists in the target;
* False — legitimate new job record.

In [ ]:
dedup_validation_df[
    "manual_is_duplicate"
] = pd.NA
display(
    dedup_validation_df
)

,clean_jobKey,clean_company,clean_title,clean_location,clean_posted_date,case1_duplicate,case2_duplicate,case3_duplicate,all_cases_agree,manual_is_duplicate
0,9c2916ac1bdaf7e2,Department of Education,Data Analyst,Melbourne City Centre VIC,2026-08-21 00:00:00+00:00,False,False,False,True,<NA>
1,1df3bf48f6eea6ca,Showtime Consulting,GCP Data Engineer,Canberra ACT,2026-08-10 00:00:00+00:00,False,False,False,True,<NA>
2,6f94f65fc6c95564,Monash University,"Group Manager - Network, Storage and Data Centres",Clayton VIC,2026-08-11 00:00:00+00:00,False,False,False,True,<NA>
3,1651d59366f4f3a1,CSIRO,Manager - Governance of Indigenous Data and In...,Brisbane QLD,2026-08-21 00:00:00+00:00,False,False,False,True,<NA>
4,8c4ce1fb43cadf86,Colliers,Database & Sales Support,Parramatta NSW,2026-08-19 00:00:00+00:00,False,False,False,True,<NA>
5,4a76befd023692bf,CBRE,Data & AI Governance Lead,Sydney NSW,2026-08-20 00:00:00+00:00,False,False,False,True,<NA>
6,2de17585f6301445,FinXL,Data Migration Lead,Melbourne City Centre VIC,2026-08-14 00:00:00+00:00,False,False,False,True,<NA>
7,28e7a32d55dffa4f,Wesfarmers,Data Scientist - Hybrid,Melbourne VIC,2026-08-21 00:00:00+00:00,False,False,False,True,<NA>
8,a63ce55a87fe2b7d,Levande,Finance Systems Analyst,Sydney NSW,2026-08-10 00:00:00+00:00,False,False,False,True,<NA>
9,9650a124388567d9,Heidi,Senior Application Database Engineer,Melbourne VIC,2026-08-10 00:00:00+00:00,False,False,False,True,<NA>


In [ ]:
manual_labels = {
    0: False,
    1: False,
    2: False,
    3: False,
    4: False,
    5: False,
    6: False,
    7: False,
    8: False,
    9: False,
    10: False,
    11: False,
    12: False,
    13: False,
    14: False,
    15: False,
    16: False,
    17: False,
    18: False,
    19: False
}

for index, label in manual_labels.items():
    dedup_validation_df.loc[
        index,
        "manual_is_duplicate"
    ] = label

**Inspect Record-Level Deduplication Decisions**

In [ ]:
dedup_decision_example_df = (
    dedup_validation_df[
        [
            "clean_company",
            "clean_title",
            "clean_location",
            "case1_duplicate",
            "case2_duplicate",
            "case3_duplicate",
            "manual_is_duplicate"
        ]
    ]
    .head(10)
)

display(
    dedup_decision_example_df.head()
)

,clean_company,clean_title,clean_location,case1_duplicate,case2_duplicate,case3_duplicate,manual_is_duplicate
0,Department of Education,Data Analyst,Melbourne City Centre VIC,False,False,False,False
1,Showtime Consulting,GCP Data Engineer,Canberra ACT,False,False,False,False
2,Monash University,"Group Manager - Network, Storage and Data Centres",Clayton VIC,False,False,False,False
3,CSIRO,Manager - Governance of Indigenous Data and In...,Brisbane QLD,False,False,False,False
4,Colliers,Database & Sales Support,Parramatta NSW,False,False,False,False


## 7.46 Define Deduplication Quality Metrics

In [ ]:
def calculate_binary_metrics( validation_df, prediction_column ):
  labelled = (
      validation_df[
          validation_df[ "manual_is_duplicate" ].notna()
          ] .copy() )
  if len(labelled) == 0:
    return {
        "accuracy": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
        "false_positives": np.nan,
        "false_negatives": np.nan }

  actual = ( labelled[ "manual_is_duplicate" ] .astype(bool) )

  predicted = ( labelled[ prediction_column ] .astype(bool) )

  true_positive = ( actual & predicted ).sum()

  true_negative = ( ~actual & ~predicted ).sum()

  false_positive = ( ~actual & predicted ).sum()

  false_negative = ( actual & ~predicted ).sum()

  accuracy = ( ( true_positive + true_negative ) / len(labelled) )

  precision = (
      true_positive / ( true_positive + false_positive )
      if ( true_positive + false_positive ) > 0
      else 0 )

  recall = (
      true_positive / ( true_positive + false_negative )
      if ( true_positive + false_negative ) > 0 else 0 )

  f1 = (
      2 * precision * recall / ( precision + recall )
      if ( precision + recall ) > 0 else 0 )

  return {
      "accuracy": accuracy * 100,
      "precision": precision * 100,
      "recall": recall * 100,
      "f1": f1 * 100,
      "false_positives": int( false_positive ),
      "false_negatives": int( false_negative ) }

## 7.47 Calculate Case 1 Quality

Run after manual labels have been completed.

In [ ]:
case1_quality = (
    calculate_binary_metrics(
        dedup_validation_df,
        "case1_duplicate" ) )

## 7.48 Calculate Case 2 Quality

In [ ]:
case2_quality = (
    calculate_binary_metrics(
        dedup_validation_df, "case2_duplicate" ) )

## 7.49 Calculate Case 3 Quality

In [ ]:
case3_quality = (
    calculate_binary_metrics(
        dedup_validation_df, "case3_duplicate" ) )

## 7.50 Display Quality Metrics

In [ ]:
dedup_quality_df = pd.DataFrame([
    {
        "Case":
            "Case 1",

        **case1_quality
    },

    {
        "Case":
            "Case 2",

        **case2_quality
    },

    {
        "Case":
            "Case 3",

        **case3_quality
    }
])
display(
    dedup_quality_df
)

,Case,accuracy,precision,recall,f1,false_positives,false_negatives
0,Case 1,100.0,0,0,0,0,0
1,Case 2,100.0,0,0,0,0,0
2,Case 3,100.0,0,0,0,0,0


## 7.51 Build the Final Deduplication Comparison

Combine performance and quality measures.

The final comparison therefore considers:

* number of duplicates identified;
* duplicate rate;
* execution time;
* average time per record;
* throughput;
* memory usage;
* accuracy;
* precision;
* recall;
* F1 score;
* false positives;
* false negatives.

In [ ]:
dedup_final_comparison_df = (
    dedup_metrics_df
    .merge(
        dedup_quality_df,
        on="Case",
        how="left"
    )
)
display(
    dedup_final_comparison_df
)

,Case,Rule,Records Evaluated,Duplicates Identified,Duplicate Rate %,Execution Time (s),Avg Time / Record (s),Records / Second,Memory Delta (MB),accuracy,precision,recall,f1,false_positives,false_negatives
0,Case 1,Company + Title + Location,20,0,0.0,0.002601,0.000130,7689.048719,0.0,100.0,0,0,0,0,0
1,Case 2,Company + Title + Location + Date Similarity,20,0,0.0,0.080028,0.004001,249.912371,0.0,100.0,0,0,0,0,0
2,Case 3,Company + Title + Location + Description Hash,20,0,0.0,0.022554,0.001128,886.757872,0.0,100.0,0,0,0,0,0


## 7.52 Compare Selectivity Between Cases

A useful additional metric is how restrictive each deduplication rule is.

In [ ]:
dedup_selectivity_df = pd.DataFrame({
    "Case": [
        "Case 1",
        "Case 2",
        "Case 3"
    ],

    "Duplicates Identified": [
        case1_duplicate_count,
        case2_duplicate_count,
        case3_duplicate_count
    ],

    "Records Retained": [
        len(
            case1_source_df
        )
        - case1_duplicate_count,

        len(
            case2_source_df
        )
        - case2_duplicate_count,

        len(
            case3_source_df
        )
        - case3_duplicate_count
    ]
})
display(
    dedup_selectivity_df
)

,Case,Duplicates Identified,Records Retained
0,Case 1,0,20
1,Case 2,0,20
2,Case 3,0,20


## 7.53 Store Deduplication Experiment Results

In [ ]:
DEDUP_EXPERIMENT_RESULTS = {
    "jobkey": {
        "records_before": jobkey_rows_before,
        "records_after": len( dedup_source_df ),
        "records_removed": ( jobkey_rows_before - len( dedup_source_df ) ),
        "execution_time_seconds": jobkey_execution_time },
    "case_1": {
        "duplicate_count": int( case1_duplicate_count ),
        "duplicate_rate_pct": case1_duplicate_rate,
        "execution_time_seconds": case1_execution_time,
        "memory_delta_mb": ( case1_memory_after - case1_memory_before ),
        **case1_quality },
    "case_2": {
        "duplicate_count": int( case2_duplicate_count ),
        "duplicate_rate_pct": case2_duplicate_rate,
        "date_similarity_threshold": DATE_SIMILARITY_THRESHOLD,
        "execution_time_seconds": case2_execution_time,
        "memory_delta_mb": ( case2_memory_after - case2_memory_before ),
        **case2_quality },
     "case_3": {
         "duplicate_count": int( case3_duplicate_count ),
         "duplicate_rate_pct": case3_duplicate_rate,
         "hash_generation_seconds": case3_hash_generation_time,
         "execution_time_seconds": case3_execution_time,
         "memory_delta_mb": ( case3_memory_after - case3_memory_before ),
         **case3_quality } }

## 7.54 Prepare Records Retained by Each Case

Create retained datasets for each approach so that the eventual selected method can be passed to AI enrichment.

### Case 1 Retained Records

In [ ]:
case1_retained_df = (
    case1_source_df.loc[
        ~case1_source_df[ "is_duplicate" ] ]
     .copy() .reset_index(drop=True) )

### Case 2 Retained Records

In [ ]:
case2_retained_df = (
    case2_source_df.loc[
        ~case2_source_df[ "is_duplicate" ] ]
    .copy() .reset_index(drop=True) )

### Case 3 Retained Records

In [ ]:
case3_retained_df = (
    case3_source_df.loc[
        ~case3_source_df[ "is_duplicate" ] ]
     .copy() .reset_index(drop=True) )

## 7.55 Validate Experiment Input Consistency

All cases must have started with the same number of source records.

In [ ]:
assert (
    len(
        case1_source_df
    )
    ==
    len(
        case2_source_df
    )
    ==
    len(
        case3_source_df
    )
), (
    "Deduplication test cases did not use "
    "the same source record set."
)
print(
    "✓ All three deduplication cases "
    "used the same source records."
)

✓ All three deduplication cases used the same source records.


## 7.56 Section 7 Outcome

Section 7 performs the complete deduplication experiment required by the Indeed pipeline.

The process first applies a source-level control using jobKey.

Duplicate non-null jobKey records are removed from daily_scrape_dev, while records with missing jobKey values remain eligible for downstream matching.

The resulting dataset is then used identically across all three experimental cases.

**Case 1 — Company + Job Title + Location**

This approach performs a direct deterministic comparison using normalised:

* company,
* job title,
* location.

It provides the broadest of the three cross-dataset matching rules and is computationally inexpensive.

**Case 2 — Company + Job Title + Location + Date Similarity**

Case 2 uses the same base fields but adds temporal evidence.

Posting-date similarity is represented as a percentage based on the project's 48-hour window:

* 0 hours difference  = 100%
* 24 hours difference = 50%
* 48 hours difference = 0%

Multiple thresholds are evaluated before a working threshold is selected.

This makes the date-matching rule explicit, measurable, and reproducible rather than relying on an undefined approximate match.

**Case 3 — Company + Job Title + Location + descriptionTextHash**

Case 3 creates descriptionTextHash during the experiment.

The hash is generated from the cleaned job description using SHA-256 for both:

* daily_scrape_dev,
* and vw_dca_jobboard.

The generated hash is then combined with company, title, and location.

This provides the strictest content-based duplicate definition of the three approaches.

**Experimental Evaluation**

All three approaches are compared using:

* records evaluated;
* duplicates identified;
* duplicate rate;
* records retained;
* execution time;
* average execution time per record;
* throughput;
* memory usage;
* accuracy;
* precision;
* recall;
* F1 score;
* false positives;
* false negatives.

A manually labelled validation dataset is used because the number of records detected as duplicates cannot itself determine which approach is correct.

The three approaches are not ranked solely by duplicate count.

The preferred production approach should be selected based on the strongest balance between:

Duplicate Detection Quality
        +
False-Positive Control
        +
False-Negative Control
        +
Processing Efficiency
        +
Operational Simplicity

The section creates separate retained datasets for all three cases but deliberately does not choose the final production method until the experiment metrics have been reviewed.

The next section will evaluate the four required AI-enrichment implementation approaches using a common benchmark dataset and compare their output quality, execution time, API usage, token consumption, resource requirements, and maintainability.

# 8. AI Enrichment Experiments

This section evaluates four implementation approaches for generating the AI-enriched target fields.

The enrichment fields are:

* top_tech_skills
* role_cat
* function
* industry
* exp_level
* pr_citizenship_req
* sec_clearance

The supplied prompt library also contains top_tech_skills_validation, which is a supporting validation prompt rather than an additional target field. The existing prompt definitions use gemini-2.5-flash-lite.
The project requires the AI enrichment pipeline to use purpose-built prompts, handle failures safely, and validate enrichment quality before records are staged.

The four test cases are:

**Case 1 — Combined Prompt Defined in Notebook**

One manually defined prompt requests all seven enrichment outputs in a single AI call.

**Case 2 — Separate Prompts Defined in Notebook**

Each enrichment task has its own manually defined prompt. top_tech_skills_validation is also executed before role_cat.

**Case 3 — Combined Prompt Built From JSON Prompt Library**

The existing JSON prompt definitions are loaded and combined dynamically into a single AI request.

**Case 4 — Separate Prompts Loaded From JSON Prompt Library**

The existing JSON prompt definitions are loaded individually and executed as separate AI requests.

All four cases use:

* the same model;
* the same benchmark records;
* the same cleaned source fields;
* the same output validation rules;
* and the same performance metrics.

## 8.1 Prepare the AI Benchmark Dataset

The AI experiment should be independent of which cross-dataset deduplication method is ultimately selected.

Use the common post-jobKey dataset from Section 7.

In [ ]:
ai_experiment_source_df = (
    dedup_test_source_df
    .copy(deep=True)
)
print(
    "Available AI experiment records:",
    len(ai_experiment_source_df)
)

Available AI experiment records: 20


## 8.2 Create a Reproducible Benchmark Sample

For small datasets, use all available records.

For larger datasets, use up to 50 records so that all four cases remain practical to run during development.

In [ ]:
AI_TEST_SAMPLE_SIZE = min(
    50,
    len(ai_experiment_source_df)
)
if (
    len(ai_experiment_source_df)
    <= AI_TEST_SAMPLE_SIZE
):

    ai_benchmark_df = (
        ai_experiment_source_df
        .copy()
        .reset_index(drop=True)
    )

else:

    ai_benchmark_df = (
        ai_experiment_source_df
        .sample(
            n=AI_TEST_SAMPLE_SIZE,
            random_state=RANDOM_STATE
        )
        .reset_index(drop=True)
    )
print(
    "AI benchmark records:",
    len(ai_benchmark_df)
)

AI benchmark records: 20


## 8.3 Define the Enrichment Fields

In [ ]:
AI_ENRICHMENT_FIELDS = [
    "top_tech_skills",
    "role_cat", "function",
    "industry", "exp_level",
    "pr_citizenship_req", "sec_clearance" ]

## 8.4 Prepare AI Input Fields

Create the canonical field names expected by the prompt definitions.

In [ ]:
ai_benchmark_df[
    "job_description"
] = (
    ai_benchmark_df[
        "clean_description"
    ]
)
ai_benchmark_df[
    "job_title"
] = (
    ai_benchmark_df[
        "clean_title"
    ]
)

### 8.4.1 Prepare Existing Industry Input

In [ ]:
if ( "clean_industry" in ai_benchmark_df.columns ):
  ai_benchmark_df[ "source_industry" ] = (
      ai_benchmark_df[ "clean_industry" ] )
else:
  ai_benchmark_df[ "source_industry" ] = None

## 8.5 Validate AI Inputs

In [ ]:
AI_REQUIRED_INPUTS = [
    "job_description",
    "job_title",
    "source_industry"
]
ai_input_check = pd.DataFrame({
    "field":
        AI_REQUIRED_INPUTS,

    "exists":
        [
            field
            in ai_benchmark_df.columns
            for field
            in AI_REQUIRED_INPUTS
        ],

    "missing_count":
        [
            ai_benchmark_df[
                field
            ].isna().sum()
            for field
            in AI_REQUIRED_INPUTS
        ]
})
display(
    ai_input_check
)

,field,exists,missing_count
0,job_description,True,0
1,job_title,True,0
2,source_industry,True,13


## 8.6 Define Approved Enum Values

The approved categorical outputs are based on the supplied prompt library.

In [ ]:
VALID_ROLE_CATEGORIES = [
    "Data Analyst",
    "Data Engineer"
]
VALID_EXPERIENCE_LEVELS = [
    "Entry-level",
    "Associate",
    "Senior",
    "Very senior",
    "Not specified"
]
VALID_PR_REQUIREMENTS = [
    "Permanent Residency",
    "Citizenship",
    "Valid work visa",
    "None mentioned"
]

## 8.7 Define Common Output Cleaning

In [ ]:
def clean_ai_output( value ):
  if value is None:
    return None

  if isinstance( value, bool ):
    return value

  value = str( value ).strip()

  if ( not value or value.lower()
  in { "none", "null", "nan" } ):
    return None

  return value

## 8.8 Define Multi-Value Normalisation

In [ ]:
def normalise_multi_value( value, allowed_values=None ):

  value = clean_ai_output( value )

  if value is None:
    return None

  parts = [
      item.strip() for item in str(value).split(",") if item.strip() ]

  if allowed_values is not None:
    cleaned_parts = []
    for item in parts:
      match = next( (
          allowed
          for allowed in allowed_values
          if item.lower() == allowed.lower() ), None )
      if match is not None: cleaned_parts.append( match )

      parts = cleaned_parts

  if not parts:
    return None

  return ", ".join( dict.fromkeys( parts ) )

## 8.9 Define Boolean Output Normalisation

In [ ]:
def normalise_boolean( value ):
  if isinstance( value, bool ):
    return value

  if value is None:
    return None

  value = str( value ).strip().lower()

  if value == "true":
    return True

  if value == "false":
    return False

  return None

## 8.10 Define Enrichment Output Validation

In [ ]:
def validate_enrichment_output( field, value ):

  if field == "role_cat":
    return normalise_multi_value( value, VALID_ROLE_CATEGORIES )

  if field == "exp_level":
    cleaned = clean_ai_output( value )

    if cleaned is None:
      return None

    for allowed in ( VALID_EXPERIENCE_LEVELS ):
      if ( cleaned.lower() == allowed.lower() ):
        return allowed

      return None

  if ( field == "pr_citizenship_req" ):
    return normalise_multi_value( value, VALID_PR_REQUIREMENTS )

  if ( field == "sec_clearance" ):
    return normalise_boolean( value )

  return clean_ai_output( value )

## 8.11 Define a Common Gemini Call Function

Every AI request should capture:

* response text;
* success/failure;
* execution time;
* input tokens;
* output tokens;
* total tokens;
* error message.

In [ ]:
def call_gemini( prompt, model=DEFAULT_AI_MODEL ):
  start_time = ( time.perf_counter() )

  try:
    response = (
        gemini_client
        .models
        .generate_content( model=model, contents=prompt ) )

    execution_time = ( time.perf_counter() - start_time )

    response_text = ( response.text.strip() if response.text else None )

    usage = getattr( response, "usage_metadata", None )

    input_tokens = (
        getattr(
            usage,
            "prompt_token_count", 0 )
        if usage else 0 )

    output_tokens = ( getattr(
        usage, "candidates_token_count", 0 )
        if usage else 0 )

    total_tokens = ( getattr(
        usage, "total_token_count",
        input_tokens + output_tokens )
        if usage else 0 )

    return {
        "text": response_text,
        "success": response_text is not None,
        "execution_time": execution_time,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens, "error": None }

  except Exception as exc:
    return {
        "text": None,
        "success": False,
        "execution_time": ( time.perf_counter() - start_time ),
        "input_tokens": 0,
        "output_tokens": 0,
        "total_tokens": 0,
        "error": str(exc) }

## 8.12 Define JSON Response Parsing

Combined-prompt cases return all enrichment fields in one JSON object.

In [ ]:
def parse_json_response( response_text ):
  if response_text is None:
    return None

  text = str( response_text ).strip()

  text = re.sub( r"^```(?:json)?", "",
                text, flags=re.IGNORECASE )
  text = re.sub( r"```$", "", text ).strip()

  try:
    return json.loads( text )
  except Exception:
    return None

## 8.13 Case 1 — Combined Prompt Defined in Notebook

**Case 1 performs all seven enrichment tasks in one AI call per record.**

The prompt is written directly in the notebook.

## 8.14 Define the Case 1 Combined Prompt

In [ ]:
CASE1_COMBINED_PROMPT = """
You are enriching an Australian job advertisement.

INPUT
Job title: {job_title}
Job description: {job_description}
Existing source industry: {source_industry}

TASKS
1. top_tech_skills
Extract only technical tools,
software platforms, programming languages, databases,
cloud platforms, frameworks, libraries or named technologies
explicitly mentioned in the advertisement. Do not include
soft skills, responsibilities, methodologies or generic concepts.
Return a comma-separated list or null.

2. role_cat
Determine whether the role is:
Data Analyst
Data Engineer
or both.
Return only approved values separated by a comma,
or null if neither can reasonably be identified.

3. function
Identify the business function supported by the job.
Do not return an industry and do not return Data Analytics or Data Engineering.
Return comma-separated business functions or null.

4. industry Identify the industry or business domain.
 Prefer the supplied source industry when it is consistent
 with the advertisement. Do not return business functions.
 Do not guess without reasonable evidence.
 Return comma-separated industries or null.

 5. exp_level Choose exactly one:
 Entry-level
 Associate
 Senior
 Very senior
 Not specified
 Definitions:
 Entry-level = 0 to 2 years
 Associate = 2 to 5 years
 Senior = 5 to 10 years
 Very senior = 10 or more years
 Use explicit experience requirements where available.

 6. pr_citizenship_req
 Choose only from:
 Permanent
 Residency
 Citizenship
 Valid work visa
 None mentioned
 Multiple explicitly stated requirements may be comma separated.
 Do not infer a requirement simply because the role is located in Australia.

 7. sec_clearance Return true only if the advertisement
 explicitly requires either:
 - security clearance, or
 - police/background check.
 Otherwise return false.

 OUTPUT Return valid JSON only using exactly these keys:
 {{
 "top_tech_skills": null,
 "role_cat": null,
 "function": null,
 "industry": null,
 "exp_level": null,
 "pr_citizenship_req": null,
 "sec_clearance": false
 }}

 Do not include explanations or markdown.
 """

## 8.15 Define Combined-Result Validation

In [ ]:
def validate_combined_result( parsed_result ):
  if not isinstance( parsed_result, dict ):
    return None

  validated = {}

  for field in ( AI_ENRICHMENT_FIELDS ):
    validated[field] = (
        validate_enrichment_output(
            field, parsed_result.get( field ) ) )
  return validated

## 8.16 Execute Case 1

In [ ]:
case1_enrichment_results = []
case1_ai_start = (
    time.perf_counter()
)
for index, row in (
    ai_benchmark_df.iterrows()
):

    prompt = (
        CASE1_COMBINED_PROMPT
        .format(
            job_title=(
                row["job_title"]
                or "None"
            ),
            job_description=(
                row[
                    "job_description"
                ]
                or "None"
            ),
            source_industry=(
                row[
                    "source_industry"
                ]
                or "None"
            )
        )
    )

    api_result = call_gemini(
        prompt
    )

    parsed = (
        parse_json_response(
            api_result[
                "text"
            ]
        )
    )

    validated = (
        validate_combined_result(
            parsed
        )
    )

    case1_enrichment_results.append({
        "record_index":
            index,

        "output":
            validated,

        **api_result
    })
case1_ai_total_time = (
    time.perf_counter()
    - case1_ai_start
)

## 8.17 Convert Case 1 Outputs to DataFrame

In [ ]:
case1_ai_output_df = (
    ai_benchmark_df
    .copy()
)
for field in (
    AI_ENRICHMENT_FIELDS
):

    case1_ai_output_df[
        field
    ] = [
        (
            result["output"].get(
                field
            )
            if isinstance(
                result["output"],
                dict
            )
            else None
        )
        for result
        in case1_enrichment_results
    ]

## 8.18 Define Case 2 Separate Notebook Prompts

**Case 2 executes individual prompts for each task.**

The structure follows the supplied prompt definitions, including the supporting top_tech_skills_validation step.

In [ ]:
CASE2_PROMPTS = {}

### 8.18.1 Technical Skills Prompt

In [ ]:
CASE2_PROMPTS[ "top_tech_skills" ] = """
INPUT:
Job description: {job_description}

INSTRUCTIONS:
Identify the top technical tools explicitly mentioned.
Only include technical tool names, software platforms,
programming languages, databases, cloud platforms,
frameworks, libraries or named technologies.
Do not include soft skills, responsibilities, methodologies
or generic concepts. Return a comma-separated list.
If none are explicitly mentioned, return exactly: None
"""

### 8.18.2 Technical Skills Validation Prompt

In [ ]:
CASE2_PROMPTS[ "top_tech_skills_validation" ] = """
INPUT:
Technical skills: {top_tech_skills}

INSTRUCTIONS:
Keep only actual technical tools, software platforms, programming
languages, databases, cloud platforms, frameworks, libraries or
named technologies. Remove soft skills, methodologies, responsibilities,
generic technical concepts and business terminology. Return the remaining
values as a comma-separated list. If none remain, return exactly: None
"""

### 8.18.3 Role Category Prompt

In [ ]:
CASE2_PROMPTS[ "role_cat" ] = """
INPUT:
Job description: {job_description}
Technical tools: {top_tech_skills}
Job title: {job_title}

INSTRUCTIONS:
Determine whether this role is Data Analyst or Data Engineer.
Choose only from:
Data Analyst
Data Engineer

If both apply, return both separated by a comma.

Do not provide explanations.
"""

### 8.18.4 Function Prompt

In [ ]:
CASE2_PROMPTS[ "function" ] = """
INPUT:
Job description: {job_description}

INSTRUCTIONS:
Identify the business function this job supports.
Return only function names. Do not return industries or domains.
Do not return Data Analytics or Data Engineering.
Multiple functions may be comma separated.
If the function cannot be reliably determined, return exactly: None
"""

### 8.18.5 Industry Prompt

In [ ]:
CASE2_PROMPTS[ "industry" ] = """
INPUT:
Job description: {job_description}

Existing source industry: {source_industry}

INSTRUCTIONS:
Identify the industry or business domain associated with the company or role.
Return only industry/domain names. Do not return business functions.
Prefer explicit source industry information when consistent with the advertisement.
Do not infer an industry without reasonable evidence.
If the industry cannot be reliably identified, return exactly: None
"""

### 8.18.6 Experience Level Prompt

In [ ]:
CASE2_PROMPTS[ "exp_level" ] = """
INPUT:
Job description: {job_description}
Job title: {job_title}

INSTRUCTIONS:
Choose exactly one:
Entry-level
Associate
Senior
Very senior
Not specified

Definitions:
Entry-level = 0 to 2 years
Associate = 2 to 5 years
Senior = 5 to 10 years
Very senior = 10 or more years

Use explicit years of experience where available.
Job-title seniority may also be used as evidence.
Do not guess when insufficient information exists.
Return only one option.

"""

### 8.18.7 Residency / Citizenship Prompt

In [ ]:
CASE2_PROMPTS[ "pr_citizenship_req" ] = """

INPUT:
Job description: {job_description}

INSTRUCTIONS:
Identify whether the advertisement explicitly states
Australian residency, citizenship or work-right requirements.
Choose only from:
Permanent Residency
Citizenship
Valid work visa
None mentioned
Multiple explicit requirements may be comma separated.
Do not infer requirements simply because the job is located in Australia.
If nothing is stated, return: None mentioned
"""

### 8.18.8 Security Clearance Prompt

In [ ]:
CASE2_PROMPTS[ "sec_clearance" ] = """
INPUT:
Job description: {job_description}

INSTRUCTIONS:
Determine whether the advertisement explicitly requires either:
- a security clearance, or
- a police/background check.
Return true only when one of these requirements is explicitly stated.
Otherwise return false. Return only true or false.
"""

## 8.19 Define Separate-Prompt Execution Function

In [ ]:
def run_separate_prompts(
    row,
    prompt_library,
    model=DEFAULT_AI_MODEL
):

    outputs = {}

    call_metrics = []

    base_values = {
        "job_description":
            row[
                "job_description"
            ]
            or "None",

        "job_title":
            row[
                "job_title"
            ]
            or "None",

        "source_industry":
            row[
                "source_industry"
            ]
            or "None"
    }

    skills_prompt = (
        prompt_library[
            "top_tech_skills"
        ]
        .format(
            **base_values
        )
    )

    skills_result = call_gemini(
        skills_prompt,
        model=model
    )

    call_metrics.append(
        skills_result
    )

    raw_skills = (
        validate_enrichment_output(
            "top_tech_skills",
            skills_result[
                "text"
            ]
        )
    )
    validation_prompt = (
        prompt_library[
            "top_tech_skills_validation"
        ]
        .format(
            top_tech_skills=(
                raw_skills
                or "None"
            )
        )
    )

    validation_result = (
        call_gemini(
            validation_prompt,
            model=model
        )
    )

    call_metrics.append(
        validation_result
    )

    validated_skills = (
        validate_enrichment_output(
            "top_tech_skills",
            validation_result[
                "text"
            ]
        )
    )

    outputs[
        "top_tech_skills"
    ] = validated_skills
    role_prompt = (
        prompt_library[
            "role_cat"
        ]
        .format(
            job_description=(
                base_values[
                    "job_description"
                ]
            ),
            job_title=(
                base_values[
                    "job_title"
                ]
            ),
            top_tech_skills=(
                validated_skills
                or "None"
            )
        )
    )

    role_result = call_gemini(
        role_prompt,
        model=model
    )

    call_metrics.append(
        role_result
    )

    outputs[
        "role_cat"
    ] = (
        validate_enrichment_output(
            "role_cat",
            role_result[
                "text"
            ]
        )
    )
    for field in [
        "function",
        "industry",
        "exp_level",
        "pr_citizenship_req",
        "sec_clearance"
    ]:

        prompt = (
            prompt_library[field]
            .format(
                **base_values
            )
        )

        result = call_gemini(
            prompt,
            model=model
        )

        call_metrics.append(
            result
        )

        outputs[field] = (
            validate_enrichment_output(
                field,
                result[
                    "text"
                ]
            )
        )

    return (
        outputs,
        call_metrics
    )

## 8.20 Execute Case 2

In [ ]:
case2_enrichment_results = []
case2_ai_start = (
    time.perf_counter()
)
for index, row in (
    ai_benchmark_df.iterrows()
):

    outputs, metrics = (
        run_separate_prompts(
            row,
            CASE2_PROMPTS
        )
    )

    case2_enrichment_results.append({
        "record_index":
            index,

        "output":
            outputs,

        "calls":
            metrics
    })
case2_ai_total_time = (
    time.perf_counter()
    - case2_ai_start
)

## 8.21 Create Case 2 Output DataFrame

In [ ]:
case2_ai_output_df = (
    ai_benchmark_df
    .copy()
)
for field in (
    AI_ENRICHMENT_FIELDS
):

    case2_ai_output_df[
        field
    ] = [
        result[
            "output"
        ].get(
            field
        )
        for result
        in case2_enrichment_results
    ]

## 8.22 Load the Existing JSON Prompt Library

Cases 3 and 4 must use the existing external prompt definitions rather than notebook-defined prompts.

The supplied file defines the enrichment tasks, required input fields, expected output type, expected outputs and model version.

In [ ]:
PROMPT_JSON_PATH = '/content/indeed_stage2_prompts.json'

with open(
    PROMPT_JSON_PATH,
    "r",
    encoding="utf-8"
) as file:

    JSON_PROMPT_LIBRARY = (
        json.load(
            file
        )
    )
print(
    JSON_PROMPT_LIBRARY.keys()
)

dict_keys(['top_tech_skills', 'top_tech_skills_validation', 'role_cat', 'function', 'industry', 'exp_level', 'pr_citizenship_req', 'sec_clearance'])


## 8.23 Validate Required JSON Prompt Definitions

In [ ]:
REQUIRED_JSON_PROMPTS = [
    "top_tech_skills",
    "top_tech_skills_validation",
    "role_cat",
    "function",
    "industry",
    "exp_level",
    "pr_citizenship_req",
    "sec_clearance"
]
missing_json_prompts = [
    prompt
    for prompt
    in REQUIRED_JSON_PROMPTS
    if prompt
    not in JSON_PROMPT_LIBRARY
]
assert (
    len(
        missing_json_prompts
    )
    == 0
), (
    "Missing JSON prompts: "
    f"{missing_json_prompts}"
)

## 8.24 Inspect Models Defined in JSON

In [ ]:
json_models = {
    JSON_PROMPT_LIBRARY[
        name
    ].get(
        "model"
    )
    for name
    in REQUIRED_JSON_PROMPTS
}
print(
    "Models defined in JSON:",
    json_models
)

Models defined in JSON: {'gemini-2.5-flash-lite'}


## 8.25 Convert the JSON Library to Prompt Strings

In [ ]:
JSON_PROMPT_STRINGS = {
    name: definition[ "prompt" ]
    for name, definition in
    JSON_PROMPT_LIBRARY.items() }

## 8.26 Case 3 — Combined Prompt From JSON

Case 3 uses the prompt definitions from the JSON file, but combines the tasks into one request.

This retains centralised prompt governance while reducing API calls.

## 8.27 Build a Combined Prompt From JSON Definitions

In [ ]:
def build_json_combined_prompt(
    row
):

    job_description = (
        row[
            "job_description"
        ]
        or "None"
    )

    job_title = (
        row[
            "job_title"
        ]
        or "None"
    )

    source_industry = (
        row[
            "source_industry"
        ]
        or "None"
    )

    task_sections = []
    for field in (
        AI_ENRICHMENT_FIELDS
    ):

        definition = (
            JSON_PROMPT_LIBRARY[
                field
            ]
        )

        prompt = (
            definition[
                "prompt"
            ]
        )

        replacements = {
            "job_description":
                job_description,

            "job_title":
                job_title,

            "source_industry":
                source_industry,

            "top_tech_skills":
                (
                    "Use the technical skills "
                    "identified in the "
                    "top_tech_skills task "
                    "in this same request."
                )
        }

        try:

            formatted_prompt = (
                prompt.format(
                    **replacements
                )
            )

        except KeyError:

            formatted_prompt = (
                prompt
            )

        task_sections.append(
            f"""
TASK: {field}

{formatted_prompt}
"""
        )
    combined_tasks = (
        "\n".join(
            task_sections
        )
    )

    final_prompt = f"""
Complete all of the following job-enrichment tasks.

Each task definition below was loaded from the
external JSON prompt library.

{combined_tasks}

Return one valid JSON object only.

Use exactly these keys:

{{
  "top_tech_skills": null,
  "role_cat": null,
  "function": null,
  "industry": null,
  "exp_level": null,
  "pr_citizenship_req": null,
  "sec_clearance": false
}}

Use null where a task cannot produce a reliable value.

Do not include markdown or explanations.
"""

    return final_prompt

## 8.28 Execute Case 3

In [ ]:
case3_enrichment_results = []
case3_ai_start = (
    time.perf_counter()
)
case3_model = (
    JSON_PROMPT_LIBRARY[
        "top_tech_skills"
    ][
        "model"
    ]
)
for index, row in (
    ai_benchmark_df.iterrows()
):

    prompt = (
        build_json_combined_prompt(
            row
        )
    )

    api_result = (
        call_gemini(
            prompt,
            model=case3_model
        )
    )

    parsed = (
        parse_json_response(
            api_result[
                "text"
            ]
        )
    )

    validated = (
        validate_combined_result(
            parsed
        )
    )

    case3_enrichment_results.append({
        "record_index":
            index,

        "output":
            validated,

        **api_result
    })
case3_ai_total_time = (
    time.perf_counter()
    - case3_ai_start
)

## 8.29 Create Case 3 Output DataFrame

In [ ]:
case3_ai_output_df = (
    ai_benchmark_df
    .copy()
)
for field in (
    AI_ENRICHMENT_FIELDS
):

    case3_ai_output_df[
        field
    ] = [
        (
            result["output"].get(
                field
            )
            if isinstance(
                result["output"],
                dict
            )
            else None
        )
        for result
        in case3_enrichment_results
    ]

## 8.30 Case 4 — Separate Prompts From JSON

Case 4 uses the external prompt library exactly as individual task definitions.

top_tech_skills is generated first, passed through top_tech_skills_validation, and the validated result is then supplied to role_cat, matching the dependency structure in the prompt library.

## 8.31 Execute Case 4

In [ ]:
case4_enrichment_results = []
case4_ai_start = (
    time.perf_counter()
)
for index, row in (
    ai_benchmark_df.iterrows()
):

    outputs, metrics = (
        run_separate_prompts(
            row,
            JSON_PROMPT_STRINGS,
            model=case3_model
        )
    )

    case4_enrichment_results.append({
        "record_index":
            index,

        "output":
            outputs,

        "calls":
            metrics
    })
case4_ai_total_time = (
    time.perf_counter()
    - case4_ai_start
)

## 8.32 Create Case 4 Output DataFrame

In [ ]:
case4_ai_output_df = (
    ai_benchmark_df
    .copy()
)
for field in (
    AI_ENRICHMENT_FIELDS
):

    case4_ai_output_df[
        field
    ] = [
        result[
            "output"
        ].get(
            field
        )
        for result
        in case4_enrichment_results
    ]

## 8.33 Define Metrics for Combined Cases

In [ ]:
def summarise_combined_case( results, total_time ):
  input_tokens = sum(
      item[ "input_tokens" ] for item in results )

  output_tokens = sum( item[ "output_tokens" ] for item in results )

  total_tokens = sum( item[ "total_tokens" ] for item in results )

  successful_calls = sum( item[ "success" ] for item in results )

  return {
      "api_calls": len(results),
      "successful_calls": successful_calls,
      "failed_calls": ( len(results) - successful_calls ),
      "execution_time_seconds": total_time,
      "input_tokens": input_tokens,
      "output_tokens": output_tokens,
      "total_tokens": total_tokens }

## 8.34 Define Metrics for Separate-Prompt Cases

In [ ]:
def summarise_separate_case( results, total_time ):
  all_calls = [
      call for record in results for call in record[ "calls" ] ]
  return {
      "api_calls": len( all_calls ),
      "successful_calls": sum( call[ "success" ] for call in all_calls ),
      "failed_calls": sum( not call[ "success" ] for call in all_calls ),
      "execution_time_seconds": total_time,
      "input_tokens": sum( call[ "input_tokens" ] for call in all_calls ),
      "output_tokens": sum( call[ "output_tokens" ] for call in all_calls ),
      "total_tokens": sum( call[ "total_tokens" ] for call in all_calls ) }

## 8.35 Calculate Metrics for All Four Cases

In [ ]:
case1_ai_metrics = (
    summarise_combined_case(
        case1_enrichment_results,
        case1_ai_total_time
    )
)
case2_ai_metrics = (
    summarise_separate_case(
        case2_enrichment_results,
        case2_ai_total_time
    )
)
case3_ai_metrics = (
    summarise_combined_case(
        case3_enrichment_results,
        case3_ai_total_time
    )
)
case4_ai_metrics = (
    summarise_separate_case(
        case4_enrichment_results,
        case4_ai_total_time
    )
)

## 8.36 Calculate Field Coverage

Coverage measures how often each enrichment field receives a non-null value.

It does not by itself prove semantic correctness.

In [ ]:
def calculate_enrichment_coverage(
    dataframe
):

    rows = []

    for field in (
        AI_ENRICHMENT_FIELDS
    ):

        populated = (
            dataframe[
                field
            ]
            .notna()
            .sum()
        )

        rows.append({
            "field":
                field,

            "populated":
                populated,

            "coverage_pct":
                round(
                    populated
                    / len(
                        dataframe
                    )
                    * 100,
                    2
                )
        })

    return pd.DataFrame(
        rows
    )

## 8.37 Calculate Coverage for Each Case

In [ ]:
case1_coverage_df = (
    calculate_enrichment_coverage(
        case1_ai_output_df
    )
)
case2_coverage_df = (
    calculate_enrichment_coverage(
        case2_ai_output_df
    )
)
case3_coverage_df = (
    calculate_enrichment_coverage(
        case3_ai_output_df
    )
)
case4_coverage_df = (
    calculate_enrichment_coverage(
        case4_ai_output_df
    )
)

## 8.38 Compare Field Coverage

In [ ]:
ai_coverage_comparison_df = (
    case1_coverage_df[
        [
            "field",
            "coverage_pct"
        ]
    ]
    .rename(
        columns={
            "coverage_pct":
                "Case 1"
        }
    )
)
for label, coverage_df in [
    (
        "Case 2",
        case2_coverage_df
    ),
    (
        "Case 3",
        case3_coverage_df
    ),
    (
        "Case 4",
        case4_coverage_df
    )
]:

    ai_coverage_comparison_df = (
        ai_coverage_comparison_df
        .merge(
            coverage_df[
                [
                    "field",
                    "coverage_pct"
                ]
            ]
            .rename(
                columns={
                    "coverage_pct":
                        label
                }
            ),
            on="field",
            how="left"
        )
    )
display(
    ai_coverage_comparison_df
)

,field,Case 1,Case 2,Case 3,Case 4
0,top_tech_skills,80.0,80.0,70.0,70.0
1,role_cat,80.0,100.0,85.0,75.0
2,function,100.0,100.0,90.0,100.0
3,industry,100.0,100.0,100.0,100.0
4,exp_level,5.0,10.0,5.0,5.0
5,pr_citizenship_req,85.0,95.0,100.0,100.0
6,sec_clearance,100.0,100.0,100.0,100.0


## 8.39 Calculate Valid Enum Output Rates

In [ ]:
def calculate_structured_validity(
    dataframe
):

    checks = {}

    checks[
        "role_cat"
    ] = (
        dataframe[
            "role_cat"
        ]
        .apply(
            lambda value:
                value is None
                or all(
                    item.strip()
                    in VALID_ROLE_CATEGORIES
                    for item
                    in value.split(",")
                )
        )
        .mean()
        * 100
    )

    checks[
        "exp_level"
    ] = (
        dataframe[
            "exp_level"
        ]
        .apply(
            lambda value:
                value is None
                or value
                in VALID_EXPERIENCE_LEVELS
        )
        .mean()
        * 100
    )

    checks[
        "pr_citizenship_req"
    ] = (
        dataframe[
            "pr_citizenship_req"
        ]
        .apply(
            lambda value:
                value is None
                or all(
                    item.strip()
                    in VALID_PR_REQUIREMENTS
                    for item
                    in value.split(",")
                )
        )
        .mean()
        * 100
    )

    checks[
        "sec_clearance"
    ] = (
        dataframe[
            "sec_clearance"
        ]
        .apply(
            lambda value:
                value is None
                or isinstance(
                    value,
                    bool
                )
        )
        .mean()
        * 100
    )

    return checks

## 8.40 Calculate Structured Validity for All Cases

In [ ]:
case1_validity = (
    calculate_structured_validity(
        case1_ai_output_df
    )
)
case2_validity = (
    calculate_structured_validity(
        case2_ai_output_df
    )
)
case3_validity = (
    calculate_structured_validity(
        case3_ai_output_df
    )
)
case4_validity = (
    calculate_structured_validity(
        case4_ai_output_df
    )
)

## 8.41 Estimate API Cost

As of August 2026, Google's published standard paid-tier pricing for gemini-2.5-flash-lite is USD $0.10 per million text input tokens and USD $0.40 per million output tokens. Pricing should be rechecked before production deployment because model pricing may change.

In [ ]:
GEMINI_INPUT_PRICE_PER_1M = 0.10
GEMINI_OUTPUT_PRICE_PER_1M = 0.40
def estimate_ai_cost(
    input_tokens,
    output_tokens
):

    input_cost = (
        input_tokens
        / 1_000_000
        * GEMINI_INPUT_PRICE_PER_1M
    )

    output_cost = (
        output_tokens
        / 1_000_000
        * GEMINI_OUTPUT_PRICE_PER_1M
    )

    return (
        input_cost
        + output_cost
    )

## 8.42 Build AI Performance Comparison

In [ ]:
AI_CASE_DETAILS = [
    (
        "Case 1",
        "Combined - Notebook",
        case1_ai_metrics,
        case1_validity
    ),
    (
        "Case 2",
        "Separate - Notebook",
        case2_ai_metrics,
        case2_validity
    ),
    (
        "Case 3",
        "Combined - JSON",
        case3_ai_metrics,
        case3_validity
    ),
    (
        "Case 4",
        "Separate - JSON",
        case4_ai_metrics,
        case4_validity
    )
]
ai_performance_rows = []
for (
    case,
    approach,
    metrics,
    validity
) in AI_CASE_DETAILS:

    ai_performance_rows.append({
        "Case":
            case,

        "Approach":
            approach,

        "Records":
            len(
                ai_benchmark_df
            ),

        "API Calls":
            metrics[
                "api_calls"
            ],

        "Failed Calls":
            metrics[
                "failed_calls"
            ],

        "Execution Time (s)":
            metrics[
                "execution_time_seconds"
            ],

        "Avg Time / Record (s)":
            (
                metrics[
                    "execution_time_seconds"
                ]
                / len(
                    ai_benchmark_df
                )
            ),

        "Input Tokens":
            metrics[
                "input_tokens"
            ],

        "Output Tokens":
            metrics[
                "output_tokens"
            ],

        "Total Tokens":
            metrics[
                "total_tokens"
            ],

        "Estimated Cost USD":
            estimate_ai_cost(
                metrics[
                    "input_tokens"
                ],
                metrics[
                    "output_tokens"
                ]
            ),

        "Structured Validity %":
            round(
                np.mean(
                    list(
                        validity.values()
                    )
                ),
                2
            )
    })
ai_performance_df = pd.DataFrame(
    ai_performance_rows
)
display(
    ai_performance_df
)

,Case,Approach,Records,API Calls,Failed Calls,Execution Time (s),Avg Time / Record (s),Input Tokens,Output Tokens,Total Tokens,Estimated Cost USD,Structured Validity %
0,Case 1,Combined - Notebook,20,20,0,9.139296,0.456965,26780,1928,28708,0.003449,100.0
1,Case 2,Separate - Notebook,20,160,0,39.665490,1.983274,133317,869,134186,0.013679,100.0
2,Case 3,Combined - JSON,20,20,0,8.588954,0.429448,136015,1803,137818,0.014323,100.0
3,Case 4,Separate - JSON,20,160,0,38.359521,1.917976,134227,760,134987,0.013727,100.0


## 8.43 Compare API Call Efficiency

In [ ]:
baseline_calls = (
    case2_ai_metrics[
        "api_calls"
    ]
)
ai_performance_df[
    "Call Reduction vs Separate Notebook %"
] = (
    (
        baseline_calls
        -
        ai_performance_df[
            "API Calls"
        ]
    )
    /
    baseline_calls
    * 100
).round(2)
display(
    ai_performance_df
)

,Case,Approach,Records,API Calls,Failed Calls,Execution Time (s),Avg Time / Record (s),Input Tokens,Output Tokens,Total Tokens,Estimated Cost USD,Structured Validity %,Call Reduction vs Separate Notebook %
0,Case 1,Combined - Notebook,20,20,0,9.139296,0.456965,26780,1928,28708,0.003449,100.0,87.5
1,Case 2,Separate - Notebook,20,160,0,39.665490,1.983274,133317,869,134186,0.013679,100.0,0.0
2,Case 3,Combined - JSON,20,20,0,8.588954,0.429448,136015,1803,137818,0.014323,100.0,87.5
3,Case 4,Separate - JSON,20,160,0,38.359521,1.917976,134227,760,134987,0.013727,100.0,0.0


## 8.44 Compare Predictions Across Cases

Create a common comparison dataframe.

In [ ]:
ai_output_comparison_df = (
    ai_benchmark_df[
        [
            "clean_jobKey",
            "job_title"
        ]
    ]
    .copy()
)
for field in (
    AI_ENRICHMENT_FIELDS
):

    ai_output_comparison_df[
        f"case1_{field}"
    ] = (
        case1_ai_output_df[
            field
        ]
    )

    ai_output_comparison_df[
        f"case2_{field}"
    ] = (
        case2_ai_output_df[
            field
        ]
    )

    ai_output_comparison_df[
        f"case3_{field}"
    ] = (
        case3_ai_output_df[
            field
        ]
    )

    ai_output_comparison_df[
        f"case4_{field}"
    ] = (
        case4_ai_output_df[
            field
        ]
    )

## 8.45 Calculate Four-Case Agreement

Agreement is useful for identifying unstable enrichment tasks, although it is not a substitute for ground-truth accuracy.

In [ ]:
ai_agreement_rows = []
for field in (
    AI_ENRICHMENT_FIELDS
):

    columns = [
        f"case1_{field}",
        f"case2_{field}",
        f"case3_{field}",
        f"case4_{field}"
    ]

    agreement_mask = (
        ai_output_comparison_df[
            columns
        ]
        .nunique(
            axis=1,
            dropna=False
        )
        == 1
    )

    ai_agreement_rows.append({
        "field":
            field,

        "agreement_pct":
            round(
                agreement_mask.mean()
                * 100,
                2
            ),

        "disagreement_records":
            int(
                (
                    ~agreement_mask
                ).sum()
            )
    })
ai_agreement_df = pd.DataFrame(
    ai_agreement_rows
)
display(
    ai_agreement_df
)

,field,agreement_pct,disagreement_records
0,top_tech_skills,35.0,13
1,role_cat,70.0,6
2,function,0.0,20
3,industry,5.0,19
4,exp_level,95.0,1
5,pr_citizenship_req,70.0,6
6,sec_clearance,100.0,0


## 8.46 Create Manual Validation Dataset

True enrichment quality requires human-reviewed ground truth.

If the benchmark contains 50 or fewer records, use all of them for validation.

For a larger benchmark, take a reproducible 30-record validation sample.

In [ ]:
if (
    len(ai_benchmark_df)
    <= 50
):

    ai_validation_df = (
        ai_output_comparison_df
        .copy()
    )

else:

    ai_validation_df = (
        ai_output_comparison_df
        .sample(
            n=30,
            random_state=RANDOM_STATE
        )
        .copy()
    )
print(
    "Manual AI validation records:",
    len(ai_validation_df)
)

Manual AI validation records: 20


## 8.47 Add Manual Ground-Truth Columns

The analyst enters the correct expected output for each enrichment field.

In [ ]:
display(
    source_eda_df[
        ["attributes", "occupation"]
    ]
)

,attributes,occupation
0,"[Power BI, Cover letter, Microsoft Excel, Full...","[Data & Database Occupations, Data Analysts, I..."
1,"[Cover letter, Full-time, Research, Australian...","[Data & Database Occupations, Information & Co..."
2,"[Data centre experience, Programme management,...","[Technical Project & Program Managers, Informa..."
3,"[Power BI, Cover letter, Permanent, Law, Data ...","[Data & Database Occupations, Information & Co..."
4,"[Data modelling, Permanent, Full-time, SQL, AW...","[Data & Database Occupations, Data Analysts, I..."
5,"[CISA, Data modelling, Full-time, Health insur...","[Data & Database Occupations, Information & Co..."
6,"[CRM software, Power BI, Permanent, Microsoft ...","[Data & Database Occupations, Information & Co..."
7,"[Cover letter, Full-time, Presentation skills,...","[Data & Database Occupations, Data Analysts, I..."
8,"[Sales support, Microsoft Excel, Full-time, Sa...","[Administrative Assistants & Receptionists, Sa..."
9,"[Full-time, Data management, No degree, Risk m...","[Data & Database Occupations, Information & Co..."


In [ ]:
for field in (
    AI_ENRICHMENT_FIELDS
):

    ai_validation_df[
        f"truth_{field}"
    ] = pd.NA
# display(
#     ai_validation_df [["case1_role_cat", "case2_role_cat", "case3_role_cat", "case4_role_cat" ]]
# )

# display(
#     ai_validation_df[["case1_", "case2_",
#                       "case3_", "case4_"]]
# )

# display(
#     ai_validation_df[["case1_top_tech_skills", "case2_top_tech_skills",
#                       "case3_top_tech_skills", "case4_top_tech_skills"]]
# )

# display(
#     ai_validation_df[["case1_function", "case2_function",
#                       "case3_function", "case4_function"]]
# )

# display(
#     ai_validation_df[["case1_industry", "case2_industry",
#                       "case3_industry", "case4_industry"]]
# )

# display(
#     ai_validation_df[["case1_exp_level", "case2_exp_level",
#                       "case3_exp_level", "case4_exp_level"]]
# )

# display(
#     ai_validation_df[["case1_pr_citizenship_req", "case2_pr_citizenship_req",
#                       "case3_pr_citizenship_req", "case4_pr_citizenship_req"]]
# )

# display(
#     ai_validation_df[["case1_sec_clearance", "case2_sec_clearance",
#                       "case3_sec_clearance", "case4_sec_clearance"]]
# )

# display(ai_validation_df)

View the manual validation dataframe

In [ ]:
SOURCE_COLUMNS = [
    "job_title",
    "job_description",
    "source_industry",
]

ai_manual_truth_df = (
    ai_benchmark_df[
        SOURCE_COLUMNS
    ]
    .copy()
)

display(
    ai_manual_truth_df.iloc[0:1]
)

,job_title,job_description,source_industry
0,Data Analyst,This position commences 1 January 2027 Are you...,None


In [ ]:
MANUAL_TRUTH_COLUMNS = [
    "truth_top_tech_skills",
    "truth_role_cat",
    "truth_function",
    "truth_industry",
    "truth_exp_level",
    "truth_pr_citizenship_req",
    "truth_sec_clearance"
]

manual_truth_labels = {
    0: {
        "truth_top_tech_skills": "Excel, Power BI",
        "truth_role_cat": "Data Analyst",
        "truth_function": "Reporting, Performance Measurement, Evaluation",
        "truth_industry": "Education",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "None mentioned",
        "truth_sec_clearance": False
    },

    1: {
        "truth_top_tech_skills": "None",
        "truth_role_cat": "Data Analyst",
        "truth_function": "Indigenous Data Governance, Intellectual Property Management, Policy Development, Research Strategy, Stakeholder Engagement",
        "truth_industry": "Government, Research",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "Citizenship",
        "truth_sec_clearance": True
    },

    2: {
        "truth_top_tech_skills": "None",
        "truth_role_cat": "Data Engineer",
        "truth_function": "Infrastructure Management, Risk Management, Program Management, Operations Management",
        "truth_industry": "CLOUD COMPUTING DATA CENTERS INFRASTRUCTURE SERVICES",
        "truth_exp_level": "Associate",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": False
    },

    3: {
        "truth_top_tech_skills": "Python, SQL, Snowflake, Google Agent Platform, Vertex AI, PowerBI, Codex",
        "truth_role_cat": "Data Analyst",
        "truth_function": "AI, Machine Learning, Business Intelligence",
        "truth_industry": "Retail",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "",
        "truth_sec_clearance": False
    },

    4: {
        "truth_top_tech_skills": "SQL, AWS Connect, AWS Athena",
        "truth_role_cat": "Data Analyst",
        "truth_function": "Customer Engagement, Reporting, Business Intelligence",
        "truth_industry": "Banking",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": False
    },

    5: {
        "truth_top_tech_skills": "None",
        "truth_role_cat": "Data Engineer",
        "truth_function": "Data Governance, AI Governance, Risk Management",
        "truth_industry": "Commercial Real Estate",
        "truth_exp_level": "Very Senior",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": True
    },

    6: {
        "truth_top_tech_skills": "Microsoft Dynamics 365, Dataverse, SQL, Excel, Power BI",
        "truth_role_cat": "Data Analyst, Data Engineer",
        "truth_function": "Sales Operations, Revenue Operations, Product Operations",
        "truth_industry": "Advertising, Media",
        "truth_exp_level": "Senior",
        "truth_pr_citizenship_req": "Permanent Residency, Citizenship",
        "truth_sec_clearance": True
    },

    7: {
        "truth_top_tech_skills": "None",
        "truth_role_cat": "Data Analyst",
        "truth_function": "Analysis, Business Improvement, Strategy, Risk Management, Regulatory Compliance",
        "truth_industry": "GOVERNMENT REGULATION",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": False
    },

    8: {
        "truth_top_tech_skills": "Microsoft Excel, Microsoft Office Suite",
        "truth_role_cat": "Data Analyst",
        "truth_function": "Sales Support, Database Management, Business Development",
        "truth_industry": "Commercial Real Estate",
        "truth_exp_level": "Entry-level",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": False
    },

    9: {
        "truth_top_tech_skills": "None",
        "truth_role_cat": "Data Analyst",
        "truth_function": "Risk Management, Business Process Improvement, Data Governance",
        "truth_industry": "Banking",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": False
    },

    10: {
        "truth_top_tech_skills": "AWS, Linux, TCP/ICP",
        "truth_role_cat": "Data Engineer",
        "truth_function": "Infrastructure Operations,IT Operations,Cloud Operations",
        "truth_industry": "Cloud Computing",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": False
    },

    11: {
        "truth_top_tech_skills": "Kraken, APIs",
        "truth_role_cat": "Data Engineer",
        "truth_function": "Data Migration, Project Management",
        "truth_industry": "BANKS_AND_FINANCIAL_SERVICES",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": False
    },

    12: {
        "truth_top_tech_skills": "SQL Server, PostgreSQL, Azure, Ansible, GitHub Actions, PowerShell, Python, Bash, Datadog, NoSQL",
        "truth_role_cat": "Data Engineer",
        "truth_function": "Database Administration, Information Technology",
        "truth_industry": "ERP, Business Services",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": False
    },

    13: {
        "truth_top_tech_skills": "Python, R, SQL, MS Fabric, Databricks, Power BI, Tableau",
        "truth_role_cat": "Data Analyst",
        "truth_function": "Pricing, Actuarial, Business Operations",
        "truth_industry": "Insurance",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": False
    },

    14: {
        "truth_top_tech_skills": "SQL, Oracle Utilities CC&B/C1, Databricks",
        "truth_role_cat": "Data Analyst",
        "truth_function": "Business Intelligence, Data Governance, Data Quality Management",
        "truth_industry": "ENERGY_AND_UTILITIES",
        "truth_exp_level": "Senior",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": False
    },

    15: {
        "truth_top_tech_skills": "None",
        "truth_role_cat": "Data Engineer",
        "truth_function": "Infrastructure Management, IT Operations",
        "truth_industry": "Education and Schools",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": True
    },

    16: {
        "truth_top_tech_skills": "MongoDB Atlas, AWS, Azure, Terraform, Python, Amazon RDS, SQL, OIDC",
        "truth_role_cat": "Data Engineer",
        "truth_function": "Database Administration,Infrastructure Management,Platform Engineering",
        "truth_industry": "Healthcare",
        "truth_exp_level": "Senior",
        "truth_pr_citizenship_req": "None",
        "truth_sec_clearance": False
    },

    17: {
        "truth_top_tech_skills": "Google Cloud Platform, Dataflow, Apache Beam, BigQuery, Kubeflow, Python, DevOps, MLOps, GCP Model Registry, Django, Cloud Run, Cloud Functions, Cloud Build, Secret Manager, Cloud KMS, R Shiny",
        "truth_role_cat": "Data Engineer",
        "truth_function": "Cloud Computing, Machine Learning Engineering, Software Development, DevOps",
        "truth_industry": "Consulting",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "Permanent Residency, Citizenship, Valid work visa",
        "truth_sec_clearance": True
    },

    18: {
        "truth_top_tech_skills": "Microsoft Dynamics 365 Finance & Operations, Power BI, SQL, Power Query, DAX, Microsoft Fabric, Azure data services, Power Automate, Power Apps",
        "truth_role_cat": "Data Analyst",
        "truth_function": "Finance, Reporting, Business Systems Analysis",
        "truth_industry": "Aged Care",
        "truth_exp_level": "Associate",
        "truth_pr_citizenship_req": "Permanent Residency",
        "truth_sec_clearance": False
    },

    19: {
        "truth_top_tech_skills": "Excel, ArcGIS",
        "truth_role_cat": "Data Analyst",
        "truth_function": "Spatial Data Management, Land Information Management, Planning Support",
        "truth_industry": "Government",
        "truth_exp_level": "None",
        "truth_pr_citizenship_req": "Citizenship, Permanent Residency, Valid work visa",
        "truth_sec_clearance": True
    },

}

for row_index, labels in manual_truth_labels.items():

    for column, value in labels.items():

        ai_validation_df.loc[
            row_index,
            column
        ] = value

ai_manual_truth_validation_df = (
    ai_validation_df[
        MANUAL_TRUTH_COLUMNS
    ]
    .copy()
)

display(
    ai_manual_truth_validation_df
)

,truth_top_tech_skills,truth_role_cat,truth_function,truth_industry,truth_exp_level,truth_pr_citizenship_req,truth_sec_clearance
0,"Excel, Power BI",Data Analyst,"Reporting, Performance Measurement, Evaluation",Education,None,None mentioned,False
1,None,Data Analyst,"Indigenous Data Governance, Intellectual Prope...","Government, Research",None,Citizenship,True
2,None,Data Engineer,"Infrastructure Management, Risk Management, Pr...",CLOUD COMPUTING DATA CENTERS INFRASTRUCTURE SE...,Associate,None,False
3,"Python, SQL, Snowflake, Google Agent Platform,...",Data Analyst,"AI, Machine Learning, Business Intelligence",Retail,None,,False
4,"SQL, AWS Connect, AWS Athena",Data Analyst,"Customer Engagement, Reporting, Business Intel...",Banking,None,None,False
5,None,Data Engineer,"Data Governance, AI Governance, Risk Management",Commercial Real Estate,Very Senior,None,True
6,"Microsoft Dynamics 365, Dataverse, SQL, Excel,...","Data Analyst, Data Engineer","Sales Operations, Revenue Operations, Product ...","Advertising, Media",Senior,"Permanent Residency, Citizenship",True
7,None,Data Analyst,"Analysis, Business Improvement, Strategy, Risk...",GOVERNMENT REGULATION,None,None,False
8,"Microsoft Excel, Microsoft Office Suite",Data Analyst,"Sales Support, Database Management, Business D...",Commercial Real Estate,Entry-level,None,False
9,None,Data Analyst,"Risk Management, Business Process Improvement,...",Banking,None,None,False


## 8.48 Define Ground-Truth Comparison Normalisation

For comma-separated fields, order should not affect correctness.

In [ ]:
MULTI_VALUE_AI_FIELDS = {
    "top_tech_skills",
    "role_cat",
    "function",
    "industry",
    "pr_citizenship_req"
}
def values_match(
    prediction,
    truth,
    field
):

    if pd.isna(truth):
        return np.nan

    if (
        field
        == "sec_clearance"
    ):

        return (
            normalise_boolean(
                prediction
            )
            ==
            normalise_boolean(
                truth
            )
        )

    if (
        field
        in MULTI_VALUE_AI_FIELDS
    ):

        def to_set(
            value
        ):

            cleaned = (
                clean_ai_output(
                    value
                )
            )

            if cleaned is None:
                return set()

            return {
                item.strip().lower()
                for item
                in cleaned.split(",")
                if item.strip()
            }

        return (
            to_set(
                prediction
            )
            ==
            to_set(
                truth
            )
        )

    return (
        str(
            prediction
        ).strip().lower()
        ==
        str(
            truth
        ).strip().lower()
    )

## 8.49 Calculate Manual Accuracy

In [ ]:
def calculate_ai_case_accuracy(
    validation_df,
    case_number
):

    field_results = []

    for field in (
        AI_ENRICHMENT_FIELDS
    ):

        prediction_column = (
            f"case{case_number}_{field}"
        )

        truth_column = (
            f"truth_{field}"
        )

        labelled = (
            validation_df[
                validation_df[
                    truth_column
                ].notna()
            ]
            .copy()
        )

        if len(
            labelled
        ) == 0:

            accuracy = np.nan
            labelled_count = 0

        else:

            matches = (
                labelled.apply(
                    lambda row:
                        values_match(
                            row[
                                prediction_column
                            ],
                            row[
                                truth_column
                            ],
                            field
                        ),
                    axis=1
                )
            )

            accuracy = (
                matches.mean()
                * 100
            )

            labelled_count = (
                len(
                    labelled
                )
            )

        field_results.append({
            "field":
                field,

            "labelled_records":
                labelled_count,

            "accuracy_pct":
                accuracy
        })

    return pd.DataFrame(
        field_results
    )

## 8.50 Calculate Accuracy for All Cases

Run after the manual truth columns have been populated.

In [ ]:
case1_accuracy_df = (
    calculate_ai_case_accuracy(
        ai_validation_df,
        1
    )
)
case2_accuracy_df = (
    calculate_ai_case_accuracy(
        ai_validation_df,
        2
    )
)
case3_accuracy_df = (
    calculate_ai_case_accuracy(
        ai_validation_df,
        3
    )
)
case4_accuracy_df = (
    calculate_ai_case_accuracy(
        ai_validation_df,
        4
    )
)

## 8.51 Compare Field-Level Accuracy

In [ ]:
ai_accuracy_comparison_df = (
    case1_accuracy_df[
        [
            "field",
            "accuracy_pct"
        ]
    ]
    .rename(
        columns={
            "accuracy_pct":
                "Case 1"
        }
    )
)
for label, dataframe in [
    (
        "Case 2",
        case2_accuracy_df
    ),
    (
        "Case 3",
        case3_accuracy_df
    ),
    (
        "Case 4",
        case4_accuracy_df
    )
]:

    ai_accuracy_comparison_df = (
        ai_accuracy_comparison_df
        .merge(
            dataframe[
                [
                    "field",
                    "accuracy_pct"
                ]
            ]
            .rename(
                columns={
                    "accuracy_pct":
                        label
                }
            ),
            on="field",
            how="left"
        )
    )
display(
    ai_accuracy_comparison_df
)

,field,Case 1,Case 2,Case 3,Case 4
0,top_tech_skills,65.0,55.0,60.0,60.0
1,role_cat,70.0,85.0,80.0,65.0
2,function,0.0,0.0,0.0,0.0
3,industry,25.0,15.0,40.0,40.0
4,exp_level,70.0,65.0,70.0,70.0
5,pr_citizenship_req,25.0,25.0,30.0,20.0
6,sec_clearance,100.0,100.0,100.0,100.0


## 8.52 Calculate Overall Accuracy

In [ ]:
overall_ai_accuracy = {
    "Case 1":
        case1_accuracy_df[
            "accuracy_pct"
        ].mean(),

    "Case 2":
        case2_accuracy_df[
            "accuracy_pct"
        ].mean(),

    "Case 3":
        case3_accuracy_df[
            "accuracy_pct"
        ].mean(),

    "Case 4":
        case4_accuracy_df[
            "accuracy_pct"
        ].mean()
}
overall_ai_accuracy_df = (
    pd.DataFrame(
        overall_ai_accuracy.items(),
        columns=[
            "Case",
            "Overall Accuracy %"
        ]
    )
)
display(
    overall_ai_accuracy_df
)

,Case,Overall Accuracy %
0,Case 1,50.714286
1,Case 2,49.285714
2,Case 3,54.285714
3,Case 4,50.714286


## 8.53 Add Accuracy to the Performance Comparison

In [ ]:
ai_final_comparison_df = (
    ai_performance_df
    .merge(
        overall_ai_accuracy_df,
        on="Case",
        how="left"
    )
)
display(
    ai_final_comparison_df
)

,Case,Approach,Records,API Calls,Failed Calls,Execution Time (s),Avg Time / Record (s),Input Tokens,Output Tokens,Total Tokens,Estimated Cost USD,Structured Validity %,Call Reduction vs Separate Notebook %,Overall Accuracy %
0,Case 1,Combined - Notebook,20,20,0,9.139296,0.456965,26780,1928,28708,0.003449,100.0,87.5,50.714286
1,Case 2,Separate - Notebook,20,160,0,39.665490,1.983274,133317,869,134186,0.013679,100.0,0.0,49.285714
2,Case 3,Combined - JSON,20,20,0,8.588954,0.429448,136015,1803,137818,0.014323,100.0,87.5,54.285714
3,Case 4,Separate - JSON,20,160,0,38.359521,1.917976,134227,760,134987,0.013727,100.0,0.0,50.714286


## 8.54 Add Maintainability Characteristics

The four approaches also differ operationally.

These are architectural characteristics rather than measured accuracy scores and should therefore be considered separately from empirical quality metrics.

In [ ]:
ai_maintainability_df = pd.DataFrame([
    {
        "Case":
            "Case 1",

        "Prompt Source":
            "Notebook",

        "Prompt Structure":
            "Combined",

        "Central Prompt Governance":
            "Low",

        "Task-Level Debugging":
            "Lower",

        "API Request Efficiency":
            "High"
    },

    {
        "Case":
            "Case 2",

        "Prompt Source":
            "Notebook",

        "Prompt Structure":
            "Separate",

        "Central Prompt Governance":
            "Low",

        "Task-Level Debugging":
            "High",

        "API Request Efficiency":
            "Low"
    },

    {
        "Case":
            "Case 3",

        "Prompt Source":
            "JSON",

        "Prompt Structure":
            "Combined",

        "Central Prompt Governance":
            "High",

        "Task-Level Debugging":
            "Lower",

        "API Request Efficiency":
            "High"
    },

    {
        "Case":
            "Case 4",

        "Prompt Source":
            "JSON",

        "Prompt Structure":
            "Separate",

        "Central Prompt Governance":
            "High",

        "Task-Level Debugging":
            "High",

        "API Request Efficiency":
            "Low"
    }
])
display(
    ai_maintainability_df
)

,Case,Prompt Source,Prompt Structure,Central Prompt Governance,Task-Level Debugging,API Request Efficiency
0,Case 1,Notebook,Combined,Low,Lower,High
1,Case 2,Notebook,Separate,Low,High,Low
2,Case 3,JSON,Combined,High,Lower,High
3,Case 4,JSON,Separate,High,High,Low


## 8.55 Store AI Experiment Results

In [ ]:
AI_EXPERIMENT_RESULTS = {
    "case_1":
        {
            **case1_ai_metrics,

            "overall_accuracy_pct":
                overall_ai_accuracy[
                    "Case 1"
                ]
        },

    "case_2":
        {
            **case2_ai_metrics,

            "overall_accuracy_pct":
                overall_ai_accuracy[
                    "Case 2"
                ]
        },

    "case_3":
        {
            **case3_ai_metrics,

            "overall_accuracy_pct":
                overall_ai_accuracy[
                    "Case 3"
                ]
        },

    "case_4":
        {
            **case4_ai_metrics,

            "overall_accuracy_pct":
                overall_ai_accuracy[
                    "Case 4"
                ]
        }
}

display(AI_EXPERIMENT_RESULTS)

{'case_1': {'api_calls': 20,
  'successful_calls': 20,
  'failed_calls': 0,
  'execution_time_seconds': 9.139296473000286,
  'input_tokens': 26780,
  'output_tokens': 1928,
  'total_tokens': 28708,
  'overall_accuracy_pct': np.float64(50.714285714285715)},
 'case_2': {'api_calls': 160,
  'successful_calls': 160,
  'failed_calls': 0,
  'execution_time_seconds': 39.665489781998986,
  'input_tokens': 133317,
  'output_tokens': 869,
  'total_tokens': 134186,
  'overall_accuracy_pct': np.float64(49.285714285714285)},
 'case_3': {'api_calls': 20,
  'successful_calls': 20,
  'failed_calls': 0,
  'execution_time_seconds': 8.58895400199981,
  'input_tokens': 136015,
  'output_tokens': 1803,
  'total_tokens': 137818,
  'overall_accuracy_pct': np.float64(54.285714285714285)},
 'case_4': {'api_calls': 160,
  'successful_calls': 160,
  'failed_calls': 0,
  'execution_time_seconds': 38.359521222000694,
  'input_tokens': 134227,
  'output_tokens': 760,
  'total_tokens': 134987,
  'overall_accuracy_pc

## 8.56 Section 8 Outcome

Section 8 evaluates four alternative implementations for the AI-enrichment component of the Indeed pipeline.

All four approaches operate on the same cleaned benchmark records and produce the same seven target enrichment fields.

**Case 1 — Combined Notebook Prompt**

All enrichment tasks are included in one manually maintained notebook prompt.

The main advantages are:

* one API call per record;
* lower network overhead;
* lower request count;
* and simple batch-style processing.

The main limitation is that all enrichment tasks are coupled into one prompt, making individual prompt maintenance and debugging more difficult.

**Case 2 — Separate Notebook Prompts**

Each enrichment task is executed separately.

This makes individual task behaviour easier to inspect and modify but substantially increases the number of API requests.

The top_tech_skills_validation prompt adds an additional call because the extracted tools are validated before being supplied to the role_cat task.

**Case 3 — Combined Prompt From JSON**

Case 3 combines the efficiency of a single request with the externally governed prompt definitions.

The enrichment task definitions are obtained from the supplied JSON prompt library rather than being duplicated as the main source of truth inside the notebook.

This approach potentially provides:

* low API-call volume;
* centralised prompt versioning;
* easier prompt governance;
* and reduced duplication between code and prompt configuration.

**Case 4 — Separate Prompts From JSON**

Case 4 executes the supplied JSON prompt definitions independently.

This most closely preserves the structure of the existing prompt library, including the dependency:

top_tech_skills
        ↓
top_tech_skills_validation
        ↓
role_cat

It provides strong task-level traceability and prompt governance but produces the largest API-call volume.

Evaluation Metrics

The four approaches are compared using:

* API-call count;
* failed calls;
* total execution time;
* average execution time per record;
* input-token usage;
* output-token usage;
* total-token usage;
* estimated API cost;
* enrichment-field coverage;
* categorical output validity;
* cross-case agreement;
* manually validated field-level accuracy;
* overall accuracy;
* prompt maintainability;
* and debugging characteristics.

The Gemini pricing variables provide an approximate cost comparison based on the current standard paid-tier rates for gemini-2.5-flash-lite; production pricing should be revalidated when the final pipeline is deployed.

Quality Interpretation

High enrichment coverage does not automatically indicate high accuracy.

Similarly, agreement between the four AI approaches does not establish ground truth.

A manually reviewed validation dataset is therefore created so that each approach can be evaluated against the actual job advertisement.

For datasets containing 50 or fewer benchmark records, the entire benchmark is used for manual validation. This avoids unnecessary sampling when the dataset is already small.

Production Selection

Section 8 deliberately does not select a winning AI approach.

The final choice should consider:

Output Accuracy
      +
Structured Output Reliability
      +
Field Coverage
      +
Execution Time
      +
API Calls
      +
Token Usage
      +
Estimated Cost
      +
Prompt Maintainability

The next section will consolidate the results from:

Section 6 — location-type experiments;
Section 7 — deduplication experiments;
Section 8 — AI-enrichment experiments;

and select the recommended method for each production pipeline stage.

# 9. Experimental Results and Method Selection

This section consolidates the results from the three experimental components:

Section 6 — Location Type Determination
Section 7 — Deduplication
Section 8 — AI Enrichment

The purpose is to compare the competing approaches using a consistent decision framework and select the preferred production method for each stage.

The recommendation should not be based on a single metric.

Instead, each method is evaluated using a combination of:

output quality;
execution time;
resource usage;
API usage;
token consumption;
estimated cost;
failure rate;
operational complexity;
and maintainability.

## 9.1 Define the Evaluation Dimensions

Use the same broad dimensions across all experiments.

In [ ]:
EVALUATION_DIMENSIONS = [
    "Quality",
    "Coverage",
    "Execution Efficiency",
    "Resource Efficiency",
    "Operational Reliability",
    "Maintainability"
]

## 9.2 Define Decision Weights

Quality should carry the greatest weight because an efficient method is not useful if it produces incorrect output.

The following weights provide a balanced production-oriented framework.

In [ ]:
DECISION_WEIGHTS = {
    "quality": 0.40,
    "coverage": 0.15,
    "execution_efficiency": 0.15,
    "resource_efficiency": 0.10,
    "reliability": 0.10,
    "maintainability": 0.10
}
print(
    "Total decision weight:",
    sum(
        DECISION_WEIGHTS.values()
    )
)

Total decision weight: 1.0


## 9.3 Validate Decision Weights

In [ ]:
assert np.isclose(
    sum(
        DECISION_WEIGHTS.values()
    ),
    1.0
), (
    "Decision weights must sum to 1."
)
print(
    "✓ Decision weights validated."
)

✓ Decision weights validated.


## 9.4 Define a Min-Max Normalisation Function

Different metrics use different scales.

Convert them to a common 0–100 scale before weighting.

In [ ]:
def min_max_score(
    series,
    higher_is_better=True
):

    series = pd.to_numeric(
        series,
        errors="coerce"
    )

    valid = series.dropna()

    if len(valid) == 0:
        return pd.Series(
            np.nan,
            index=series.index
        )

    minimum = valid.min()
    maximum = valid.max()

    if maximum == minimum:

        score = pd.Series(
            100.0,
            index=series.index
        )

    else:

        score = (
            (
                series
                - minimum
            )
            /
            (
                maximum
                - minimum
            )
            * 100
        )

    if not higher_is_better:
        score = 100 - score

    return score

## 9.5 Location Experiment Decision

The location experiment compares:

**Case 1 — AI Only**
**Case 2 — GCCSA + AI Fallback**

Use the metrics generated in Section 6.

In [ ]:
location_selection_df = (
    location_metrics_df
    .copy()
)
display(
    location_selection_df
)

,Case,Approach,Records,Valid Output %,AI Calls,AI Fallback %,Execution Time (s),Avg Time / Record (s),Input Tokens,Output Tokens,Total Tokens,Accuracy %
0,Case 1,AI Only,20,100.0,20,100.0,5.277819,0.263891,2561,20,2581,100.0
1,Case 2,GCCSA + AI Fallback,20,100.0,1,5.0,0.373134,0.018657,127,1,128,100.0


## 9.6 Add Location Quality Score

In [ ]:
location_selection_df[ "Quality Score" ] = (
    location_selection_df[ "Accuracy %" ] )

## 9.7 Add Location Coverage Score

In [ ]:
location_selection_df[
    "Coverage Score"
] = (
    location_selection_df[
        "Valid Output %"
    ]
)

## 9.8 Add Location Execution Efficiency Score

Lower execution time is better.

In [ ]:
location_selection_df[
    "Execution Efficiency Score"
] = (
    min_max_score(
        location_selection_df[
            "Execution Time (s)"
        ],
        higher_is_better=False
    )
)

## 9.9 Add Location Resource Efficiency Score

Token usage is used as the primary resource metric because it directly influences API cost.

In [ ]:
location_selection_df[
    "Resource Efficiency Score"
] = (
    min_max_score(
        location_selection_df[
            "Total Tokens"
        ],
        higher_is_better=False
    )
)

## 9.10 Add Location Reliability Score

Use valid output rate as the reliability measure.

In [ ]:
location_selection_df[
    "Reliability Score"
] = (
    location_selection_df[
        "Valid Output %"
    ]
)

## 9.11 Add Location Maintainability Score

Maintainability is partly architectural and must therefore be explicitly documented rather than inferred from runtime metrics.

The hybrid method introduces a shapefile dependency, while AI-only is simpler operationally.

However, the hybrid method substantially reduces external AI dependency when deterministic coverage is high.

Use a transparent maintainability score.

In [ ]:
LOCATION_MAINTAINABILITY = {
    "AI Only": 90,
    "GCCSA + AI Fallback": 80
}
location_selection_df[
    "Maintainability Score"
] = (
    location_selection_df[
        "Approach"
    ]
    .map(
        LOCATION_MAINTAINABILITY
    )
)

## 9.12 Calculate Weighted Location Score

In [ ]:
location_selection_df[
    "Weighted Score"
] = (
    location_selection_df[
        "Quality Score"
    ]
    * DECISION_WEIGHTS[
        "quality"
    ]
    +
    location_selection_df[
        "Coverage Score"
    ]
    * DECISION_WEIGHTS[
        "coverage"
    ]
    +
    location_selection_df[
        "Execution Efficiency Score"
    ]
    * DECISION_WEIGHTS[
        "execution_efficiency"
    ]
    +
    location_selection_df[
        "Resource Efficiency Score"
    ]
    * DECISION_WEIGHTS[
        "resource_efficiency"
    ]
    +
    location_selection_df[
        "Reliability Score"
    ]
    * DECISION_WEIGHTS[
        "reliability"
    ]
    +
    location_selection_df[
        "Maintainability Score"
    ]
    * DECISION_WEIGHTS[
        "maintainability"
    ]
)
display(
    location_selection_df
)

,Case,Approach,Records,Valid Output %,AI Calls,AI Fallback %,Execution Time (s),Avg Time / Record (s),Input Tokens,Output Tokens,Total Tokens,Accuracy %,Quality Score,Coverage Score,Execution Efficiency Score,Resource Efficiency Score,Reliability Score,Maintainability Score,Weighted Score
0,Case 1,AI Only,20,100.0,20,100.0,5.277819,0.263891,2561,20,2581,100.0,100.0,100.0,0.0,0.0,100.0,90,74.0
1,Case 2,GCCSA + AI Fallback,20,100.0,1,5.0,0.373134,0.018657,127,1,128,100.0,100.0,100.0,100.0,100.0,100.0,80,98.0


## 9.13 Identify the Recommended Location Method

In [ ]:
if (
    location_selection_df[
        "Quality Score"
    ]
    .notna()
    .all()
):

    recommended_location_row = (
        location_selection_df
        .sort_values(
            "Weighted Score",
            ascending=False
        )
        .iloc[0]
    )

    RECOMMENDED_LOCATION_CASE = (
        recommended_location_row[
            "Case"
        ]
    )

    RECOMMENDED_LOCATION_METHOD = (
        recommended_location_row[
            "Approach"
        ]
    )

else:

    RECOMMENDED_LOCATION_CASE = None
    RECOMMENDED_LOCATION_METHOD = None
print(
    "Recommended location case:",
    RECOMMENDED_LOCATION_CASE
)

print(
    "Recommended location method:",
    RECOMMENDED_LOCATION_METHOD
)

Recommended location case: Case 2
Recommended location method: GCCSA + AI Fallback


## 9.14 Deduplication Experiment Decision

The deduplication experiment compares:

Case 1 — Company + Title + Location
Case 2 — Company + Title + Location + Date Similarity
Case 3 — Company + Title + Location + Description Hash

In [ ]:
dedup_selection_df = (
    dedup_final_comparison_df
    .copy()
)
display(
    dedup_selection_df
)

,Case,Rule,Records Evaluated,Duplicates Identified,Duplicate Rate %,Execution Time (s),Avg Time / Record (s),Records / Second,Memory Delta (MB),accuracy,precision,recall,f1,false_positives,false_negatives
0,Case 1,Company + Title + Location,20,0,0.0,0.002601,0.000130,7689.048719,0.0,100.0,0,0,0,0,0
1,Case 2,Company + Title + Location + Date Similarity,20,0,0.0,0.080028,0.004001,249.912371,0.0,100.0,0,0,0,0,0
2,Case 3,Company + Title + Location + Description Hash,20,0,0.0,0.022554,0.001128,886.757872,0.0,100.0,0,0,0,0,0


## 9.15 Add Deduplication Quality Score

F1 score is used as the principal quality measure because deduplication must balance:

* false-positive removal;
* and missed duplicates.

In [ ]:
dedup_selection_df[
    "Quality Score"
] = (
    dedup_selection_df[
        "f1"
    ]
)

## 9.16 Add Deduplication Coverage Score

Coverage for deduplication is interpreted as the proportion of records that could be evaluated using the method.

For the current experiment, all three cases use the same post-jobKey dataset.

Therefore each receives full evaluation coverage unless a method lacks required fields.

In [ ]:
dedup_selection_df[ "Coverage Score" ] = 100.0

## 9.17 Add Deduplication Execution Efficiency Score

In [ ]:
dedup_selection_df[
    "Execution Efficiency Score"
] = (
    min_max_score(
        dedup_selection_df[
            "Execution Time (s)"
        ],
        higher_is_better=False
    )
)

## 9.18 Add Deduplication Resource Efficiency Score

Lower memory growth is preferred.

In [ ]:
dedup_selection_df[
    "Resource Efficiency Score"
] = (
    min_max_score(
        dedup_selection_df[
            "Memory Delta (MB)"
        ],
        higher_is_better=False
    )
)

## 9.19 Add Deduplication Reliability Score

Precision is particularly important because false positives would cause legitimate new jobs to be removed.

Use precision as the operational reliability score.

In [ ]:
dedup_selection_df[ "Reliability Score" ] = (
    dedup_selection_df[ "precision" ] )

## 9.20 Define Deduplication Maintainability Scores

In [ ]:
DEDUP_MAINTAINABILITY = {
    "Case 1": 95,
    "Case 2": 80,
    "Case 3": 85
    }

Interpretation:

* Case 1 is the simplest to understand and maintain.
* Case 2 adds date-similarity threshold logic.
* Case 3 adds deterministic hash generation but remains straightforward.

In [ ]:
dedup_selection_df[
    "Maintainability Score"
] = (
    dedup_selection_df[
        "Case"
    ]
    .map(
        DEDUP_MAINTAINABILITY
    )
)

## 9.21 Calculate Weighted Deduplication Score

In [ ]:
dedup_selection_df[
    "Weighted Score"
] = (
    dedup_selection_df[
        "Quality Score"
    ]
    * DECISION_WEIGHTS[
        "quality"
    ]
    +
    dedup_selection_df[
        "Coverage Score"
    ]
    * DECISION_WEIGHTS[
        "coverage"
    ]
    +
    dedup_selection_df[
        "Execution Efficiency Score"
    ]
    * DECISION_WEIGHTS[
        "execution_efficiency"
    ]
    +
    dedup_selection_df[
        "Resource Efficiency Score"
    ]
    * DECISION_WEIGHTS[
        "resource_efficiency"
    ]
    +
    dedup_selection_df[
        "Reliability Score"
    ]
    * DECISION_WEIGHTS[
        "reliability"
    ]
    +
    dedup_selection_df[
        "Maintainability Score"
    ]
    * DECISION_WEIGHTS[
        "maintainability"
    ]
)
display(
    dedup_selection_df
)

,Case,Rule,Records Evaluated,Duplicates Identified,Duplicate Rate %,Execution Time (s),Avg Time / Record (s),Records / Second,Memory Delta (MB),accuracy,...,f1,false_positives,false_negatives,Quality Score,Coverage Score,Execution Efficiency Score,Resource Efficiency Score,Reliability Score,Maintainability Score,Weighted Score
0,Case 1,Company + Title + Location,20,0,0.0,0.002601,0.000130,7689.048719,0.0,100.0,...,0,0,0,0,100.0,100.000000,0.0,0,95,39.500000
1,Case 2,Company + Title + Location + Date Similarity,20,0,0.0,0.080028,0.004001,249.912371,0.0,100.0,...,0,0,0,0,100.0,0.000000,0.0,0,80,23.000000
2,Case 3,Company + Title + Location + Description Hash,20,0,0.0,0.022554,0.001128,886.757872,0.0,100.0,...,0,0,0,0,100.0,74.229943,0.0,0,85,34.634491


## 9.22 Identify the Recommended Deduplication Method

In [ ]:
if (
    dedup_selection_df[
        "Quality Score"
    ]
    .notna()
    .all()
):

    recommended_dedup_row = (
        dedup_selection_df
        .sort_values(
            "Weighted Score",
            ascending=False
        )
        .iloc[0]
    )

    RECOMMENDED_DEDUP_CASE = (
        recommended_dedup_row[
            "Case"
        ]
    )

    RECOMMENDED_DEDUP_METHOD = (
        recommended_dedup_row[
            "Rule"
        ]
    )

else:

    RECOMMENDED_DEDUP_CASE = None
    RECOMMENDED_DEDUP_METHOD = None
print(
    "Recommended deduplication case:",
    RECOMMENDED_DEDUP_CASE
)

print(
    "Recommended deduplication method:",
    RECOMMENDED_DEDUP_METHOD
)

Recommended deduplication case: Case 1
Recommended deduplication method: Company + Title + Location


## 9.23 AI-Enrichment Experiment Decision

The AI experiment compares:

* Case 1 — Combined Notebook Prompt
* Case 2 — Separate Notebook Prompts
* Case 3 — Combined JSON Prompt
* Case 4 — Separate JSON Prompts

In [ ]:
ai_selection_df = (
    ai_final_comparison_df
    .copy()
)
display(
    ai_selection_df
)

,Case,Approach,Records,API Calls,Failed Calls,Execution Time (s),Avg Time / Record (s),Input Tokens,Output Tokens,Total Tokens,Estimated Cost USD,Structured Validity %,Call Reduction vs Separate Notebook %,Overall Accuracy %
0,Case 1,Combined - Notebook,20,20,0,9.139296,0.456965,26780,1928,28708,0.003449,100.0,87.5,50.714286
1,Case 2,Separate - Notebook,20,160,0,39.665490,1.983274,133317,869,134186,0.013679,100.0,0.0,49.285714
2,Case 3,Combined - JSON,20,20,0,8.588954,0.429448,136015,1803,137818,0.014323,100.0,87.5,54.285714
3,Case 4,Separate - JSON,20,160,0,38.359521,1.917976,134227,760,134987,0.013727,100.0,0.0,50.714286


## 9.24 Add AI Quality Score

Use manually validated overall accuracy.

In [ ]:
ai_selection_df[
    "Quality Score"
] = (
    ai_selection_df[
        "Overall Accuracy %"
    ]
)

## 9.25 Add AI Coverage Score

Use the average enrichment-field coverage across all seven enrichment fields.

First calculate case-level averages.

In [ ]:
AI_COVERAGE_TABLES = {
    "Case 1":
        case1_coverage_df,

    "Case 2":
        case2_coverage_df,

    "Case 3":
        case3_coverage_df,

    "Case 4":
        case4_coverage_df
}
average_ai_coverage = {
    case:
        dataframe[
            "coverage_pct"
        ].mean()
    for case, dataframe
    in AI_COVERAGE_TABLES.items()
}
ai_selection_df[
    "Coverage Score"
] = (
    ai_selection_df[
        "Case"
    ]
    .map(
        average_ai_coverage
    )
)

## 9.26 Add AI Execution Efficiency Score

In [ ]:
ai_selection_df[
    "Execution Efficiency Score"
] = (
    min_max_score(
        ai_selection_df[
            "Execution Time (s)"
        ],
        higher_is_better=False
    )
)

## 9.27 Add AI Resource Efficiency Score

Use estimated API cost because it incorporates both input and output token consumption.

In [ ]:
ai_selection_df[
    "Resource Efficiency Score"
] = (
    min_max_score(
        ai_selection_df[
            "Estimated Cost USD"
        ],
        higher_is_better=False
    )
)

## 9.28 Add AI Operational Reliability Score

Reliability should reward:

* valid structured outputs;
* and low call-failure rates.


Calculate the successful-call percentage.

In [ ]:
ai_selection_df[
    "Call Success %"
] = (
    (
        ai_selection_df[
            "API Calls"
        ]
        -
        ai_selection_df[
            "Failed Calls"
        ]
    )
    /
    ai_selection_df[
        "API Calls"
    ]
    * 100
)
ai_selection_df[
    "Operational Reliability Score"
] = (
    (
        ai_selection_df[
            "Structured Validity %"
        ]
        +
        ai_selection_df[
            "Call Success %"
        ]
    )
    / 2
)

## 9.29 Define AI Maintainability Scores

The prompt architecture from Section 8 provides the maintainability basis.

In [ ]:
AI_MAINTAINABILITY = {
    "Case 1": 70,
    "Case 2": 75,
    "Case 3": 95,
    "Case 4": 90
}

Interpretation:

* notebook-managed prompts require code changes when prompts change;
* JSON-managed prompts provide stronger central governance;
* separate prompts are easier to debug;
* combined prompts reduce API complexity.

In [ ]:
ai_selection_df[
    "Maintainability Score"
] = (
    ai_selection_df[
        "Case"
    ]
    .map(
        AI_MAINTAINABILITY
    )
)

## 9.30 Calculate Weighted AI Score

In [ ]:
ai_selection_df[
    "Weighted Score"
] = (
    ai_selection_df[
        "Quality Score"
    ]
    * DECISION_WEIGHTS[
        "quality"
    ]
    +
    ai_selection_df[
        "Coverage Score"
    ]
    * DECISION_WEIGHTS[
        "coverage"
    ]
    +
    ai_selection_df[
        "Execution Efficiency Score"
    ]
    * DECISION_WEIGHTS[
        "execution_efficiency"
    ]
    +
    ai_selection_df[
        "Resource Efficiency Score"
    ]
    * DECISION_WEIGHTS[
        "resource_efficiency"
    ]
    +
    ai_selection_df[
        "Operational Reliability Score"
    ]
    * DECISION_WEIGHTS[
        "reliability"
    ]
    +
    ai_selection_df[
        "Maintainability Score"
    ]
    * DECISION_WEIGHTS[
        "maintainability"
    ]
)
display(
    ai_selection_df
)

,Case,Approach,Records,API Calls,Failed Calls,Execution Time (s),Avg Time / Record (s),Input Tokens,Output Tokens,Total Tokens,...,Call Reduction vs Separate Notebook %,Overall Accuracy %,Quality Score,Coverage Score,Execution Efficiency Score,Resource Efficiency Score,Call Success %,Operational Reliability Score,Maintainability Score,Weighted Score
0,Case 1,Combined - Notebook,20,20,0,9.139296,0.456965,26780,1928,28708,...,87.5,50.714286,50.714286,78.571429,98.229074,100.000000,100.0,100.0,70,73.805790
1,Case 2,Separate - Notebook,20,160,0,39.665490,1.983274,133317,869,134186,...,0.0,49.285714,49.285714,83.571429,0.000000,5.917138,100.0,100.0,75,50.341714
2,Case 3,Combined - JSON,20,20,0,8.588954,0.429448,136015,1803,137818,...,87.5,54.285714,54.285714,78.571429,100.000000,0.000000,100.0,100.0,95,68.000000
3,Case 4,Separate - JSON,20,160,0,38.359521,1.917976,134227,760,134987,...,0.0,50.714286,50.714286,78.571429,4.202426,5.481216,100.0,100.0,90,52.249914


## 9.31 Identify the Recommended AI Method

In [ ]:
if (
    ai_selection_df[
        "Quality Score"
    ]
    .notna()
    .all()
):

    recommended_ai_row = (
        ai_selection_df
        .sort_values(
            "Weighted Score",
            ascending=False
        )
        .iloc[0]
    )

    RECOMMENDED_AI_CASE = (
        recommended_ai_row[
            "Case"
        ]
    )

    RECOMMENDED_AI_METHOD = (
        recommended_ai_row[
            "Approach"
        ]
    )

else:

    RECOMMENDED_AI_CASE = None
    RECOMMENDED_AI_METHOD = None
print(
    "Recommended AI case:",
    RECOMMENDED_AI_CASE
)

print(
    "Recommended AI method:",
    RECOMMENDED_AI_METHOD
)

Recommended AI case: Case 1
Recommended AI method: Combined - Notebook


## 9.32 Create Overall Recommendation Summary

In [ ]:
method_selection_summary_df = pd.DataFrame([
    {
        "Pipeline Stage":
            "Location Type",

        "Recommended Case":
            RECOMMENDED_LOCATION_CASE,

        "Recommended Method":
            RECOMMENDED_LOCATION_METHOD
    },

    {
        "Pipeline Stage":
            "Deduplication",

        "Recommended Case":
            RECOMMENDED_DEDUP_CASE,

        "Recommended Method":
            RECOMMENDED_DEDUP_METHOD
    },

    {
        "Pipeline Stage":
            "AI Enrichment",

        "Recommended Case":
            RECOMMENDED_AI_CASE,

        "Recommended Method":
            RECOMMENDED_AI_METHOD
    }
])
display(
    method_selection_summary_df
)

,Pipeline Stage,Recommended Case,Recommended Method
0,Location Type,Case 2,GCCSA + AI Fallback
1,Deduplication,Case 1,Company + Title + Location
2,AI Enrichment,Case 1,Combined - Notebook


## 9.33 Check Whether All Selections Are Complete

In [ ]:
selection_complete = all([
    RECOMMENDED_LOCATION_CASE
    is not None,

    RECOMMENDED_DEDUP_CASE
    is not None,

    RECOMMENDED_AI_CASE
    is not None
])
print(
    "All production methods selected:",
    selection_complete
)

All production methods selected: True


## 9.34 Prevent Premature Production Selection

If manual labels have not yet been entered, the notebook should clearly identify which experiment still lacks validated quality evidence.

In [ ]:
selection_readiness_df = pd.DataFrame({
    "Experiment": [
        "Location",
        "Deduplication",
        "AI Enrichment"
    ],

    "Manual Quality Available": [
        location_selection_df[
            "Quality Score"
        ]
        .notna()
        .all(),

        dedup_selection_df[
            "Quality Score"
        ]
        .notna()
        .all(),

        ai_selection_df[
            "Quality Score"
        ]
        .notna()
        .all()
    ]
})
display(
    selection_readiness_df
)

,Experiment,Manual Quality Available
0,Location,True
1,Deduplication,True
2,AI Enrichment,True


## 9.35 Create Ranked Location Results

In [ ]:
location_ranking_df = (
    location_selection_df
    .sort_values(
        "Weighted Score",
        ascending=False,
        na_position="last"
    )
    [
        [
            "Case",
            "Approach",
            "Quality Score",
            "Coverage Score",
            "Execution Efficiency Score",
            "Resource Efficiency Score",
            "Reliability Score",
            "Maintainability Score",
            "Weighted Score"
        ]
    ]
)
display(
    location_ranking_df
)

,Case,Approach,Quality Score,Coverage Score,Execution Efficiency Score,Resource Efficiency Score,Reliability Score,Maintainability Score,Weighted Score
1,Case 2,GCCSA + AI Fallback,100.0,100.0,100.0,100.0,100.0,80,98.0
0,Case 1,AI Only,100.0,100.0,0.0,0.0,100.0,90,74.0


## 9.36 Create Ranked Deduplication Results

In [ ]:
dedup_ranking_df = (
    dedup_selection_df
    .sort_values(
        "Weighted Score",
        ascending=False,
        na_position="last"
    )
    [
        [
            "Case",
            "Rule",
            "Quality Score",
            "Coverage Score",
            "Execution Efficiency Score",
            "Resource Efficiency Score",
            "Reliability Score",
            "Maintainability Score",
            "Weighted Score"
        ]
    ]
)
display(
    dedup_ranking_df
)

,Case,Rule,Quality Score,Coverage Score,Execution Efficiency Score,Resource Efficiency Score,Reliability Score,Maintainability Score,Weighted Score
0,Case 1,Company + Title + Location,0,100.0,100.000000,0.0,0,95,39.500000
2,Case 3,Company + Title + Location + Description Hash,0,100.0,74.229943,0.0,0,85,34.634491
1,Case 2,Company + Title + Location + Date Similarity,0,100.0,0.000000,0.0,0,80,23.000000


## 9.37 Create Ranked AI Results

In [ ]:
ai_ranking_df = (
    ai_selection_df
    .sort_values(
        "Weighted Score",
        ascending=False,
        na_position="last"
    )
    [
        [
            "Case",
            "Approach",
            "Quality Score",
            "Coverage Score",
            "Execution Efficiency Score",
            "Resource Efficiency Score",
            "Operational Reliability Score",
            "Maintainability Score",
            "Weighted Score"
        ]
    ]
)
display(
    ai_ranking_df
)


,Case,Approach,Quality Score,Coverage Score,Execution Efficiency Score,Resource Efficiency Score,Operational Reliability Score,Maintainability Score,Weighted Score
0,Case 1,Combined - Notebook,50.714286,78.571429,98.229074,100.000000,100.0,70,73.805790
2,Case 3,Combined - JSON,54.285714,78.571429,100.000000,0.000000,100.0,95,68.000000
3,Case 4,Separate - JSON,50.714286,78.571429,4.202426,5.481216,100.0,90,52.249914
1,Case 2,Separate - Notebook,49.285714,83.571429,0.000000,5.917138,100.0,75,50.341714


## 9.38 Store Final Method-Selection Results

In [ ]:
METHOD_SELECTION_RESULTS = {
    "location": {
        "case":
            RECOMMENDED_LOCATION_CASE,

        "method":
            RECOMMENDED_LOCATION_METHOD
    },

    "deduplication": {
        "case":
            RECOMMENDED_DEDUP_CASE,

        "method":
            RECOMMENDED_DEDUP_METHOD
    },

    "ai_enrichment": {
        "case":
            RECOMMENDED_AI_CASE,

        "method":
            RECOMMENDED_AI_METHOD
    },

    "selection_complete":
        selection_complete
}

## 9.39 Section 9 Outcome

Section 9 consolidates the experimental evidence generated in Sections 6–8 and converts it into a structured method-selection framework.

The three production decisions are evaluated independently.

**Location-Type Selection**

The AI-only and hybrid GCCSA + AI approaches are compared using:

* manually validated accuracy;
* valid classification coverage;
* execution time;
* token consumption;
* successful output rate;
* and maintainability.

The hybrid approach is not automatically preferred simply because it uses fewer AI calls.

Likewise, AI-only is not preferred simply because it is architecturally simpler.

The selected method must demonstrate an acceptable balance between classification quality and operational efficiency.

**Deduplication Selection**

The three deduplication approaches are evaluated using:

* F1 score;
* precision;
* recall;
* false positives;
* false negatives;
* execution time;
* memory usage;
* and maintainability.

Particular importance is placed on precision because an incorrect duplicate decision can cause a legitimate new job to be discarded.

The preferred method therefore needs to minimise both:

*False Positive:*
Legitimate job incorrectly removed

*False Negative:*
Existing duplicate incorrectly retained

while still remaining computationally practical.

**AI-Enrichment Selection**

The four AI architectures are evaluated using:

* manually validated accuracy;
* enrichment coverage;
* structured-output validity;
* API success rate;
* execution time;
* API-call count;
* token usage;
* estimated cost;
* and prompt maintainability.

The comparison explicitly separates:

Prompt Structure
Combined vs Separate

from

Prompt Source
Notebook vs JSON

This allows the final decision to distinguish between model-call efficiency and prompt-governance quality.

***Weighted Decision Framework***

A common weighted scoring framework is used:

Dimension	Weight
Quality	40%
Coverage	15%
Execution Efficiency	15%
Resource Efficiency	10%
Reliability	10%
Maintainability	10%

Quality receives the largest weighting because the objective is not merely to build the fastest pipeline, but to produce reliable data suitable for the production job board.

**Manual Validation Requirement**

The notebook deliberately prevents an automatic final recommendation when manually validated quality metrics are unavailable.

This is important because:

* agreement between algorithms is not ground truth;
* high output coverage does not prove accuracy;
* high duplicate counts do not prove good deduplication;
* and low API cost does not prove good enrichment.

The final production selections are therefore made only when sufficient validation evidence exists.

The selected approaches are stored in:

**METHOD_SELECTION_RESULTS**

with separate selections for:

* location
* deduplication
* ai_enrichment

These results will be consumed by the next section, which will construct the final end-to-end production pipeline using the selected methods.

# 10. Final End-to-End Production Pipeline

This section combines the completed notebook components into a single production-oriented workflow.

The final pipeline applies the following sequence:

daily_scrape_dev

        ↓

Cleaning and Standardisation

        ↓

Source-Level jobKey Deduplication

        ↓

Selected Location-Type Method

        ↓

Selected Cross-Dataset Deduplication Method

        ↓

Selected AI-Enrichment Method

        ↓

Final 20-Field Mapping

        ↓

Data-Quality Validation

        ↓

jobboard_staging

The pipeline does not write records directly to vw_dca_jobboard. Records are prepared for the staging and review layer first, consistent with the project requirement that transformed records be reviewed before production insertion.

## 10.1 Define Production Method Configuration

Use the recommendations produced in Section 9.

In [ ]:
PRODUCTION_CONFIG = {
    "location_case":
        RECOMMENDED_LOCATION_CASE,

    "dedup_case":
        RECOMMENDED_DEDUP_CASE,

    "ai_case":
        RECOMMENDED_AI_CASE
}
print(
    PRODUCTION_CONFIG
)

{'location_case': 'Case 2', 'dedup_case': 'Case 1', 'ai_case': 'Case 1'}


## 10.2 Validate Production Configuration

In [ ]:
def validate_production_config(
    config
):

    valid_location_cases = {
        "Case 1",
        "Case 2"
    }

    valid_dedup_cases = {
        "Case 1",
        "Case 2",
        "Case 3"
    }

    valid_ai_cases = {
        "Case 1",
        "Case 2",
        "Case 3",
        "Case 4"
    }

    errors = []

    if (
        config[
            "location_case"
        ]
        not in valid_location_cases
    ):

        errors.append(
            "A valid location method "
            "has not been selected."
        )

    if (
        config[
            "dedup_case"
        ]
        not in valid_dedup_cases
    ):

        errors.append(
            "A valid deduplication method "
            "has not been selected."
        )

    if (
        config[
            "ai_case"
        ]
        not in valid_ai_cases
    ):

        errors.append(
            "A valid AI enrichment method "
            "has not been selected."
        )

    return errors
production_config_errors = (
    validate_production_config(
        PRODUCTION_CONFIG
    )
)
if production_config_errors:

    for error in (
        production_config_errors
    ):
        print(
            "⚠",
            error
        )

else:

    print(
        "✓ Production configuration validated."
    )

✓ Production configuration validated.


## 10.3 Define Final Target Fields

The final output contains exactly 20 mapped target fields.

In [ ]:
FINAL_TARGET_FIELDS = [
    "id",
    "api_ini_dt_time",
    "api_name",
    "posted_date",
    "job_title",
    "company_name",
    "city",
    "state",
    "location",
    "job_description",
    "url",
    "source",
    "industry",
    "function",
    "top_tech_skills",
    "exp_level",
    "pr_citizenship_req",
    "sec_clearance",
    "location_type",
    "role_cat"
]
assert (
    len(
        FINAL_TARGET_FIELDS
    )
    == 20
)
print(
    "Final target fields:",
    len(
        FINAL_TARGET_FIELDS
    )
)

Final target fields: 20


## 10.4 Define Production Cleaning Function

In [ ]:
def prepare_source_for_pipeline(
    dataframe
):

    df = dataframe.copy(
        deep=True
    )

    df[
        "clean_jobKey"
    ] = (
        df[
            "jobKey"
        ]
        .apply(
            clean_text
        )
    )

    df[
        "clean_company"
    ] = (
        df[
            "companyName"
        ]
        .apply(
            clean_text
        )
    )

    df[
        "clean_title"
    ] = (
        df[
            "title"
        ]
        .apply(
            clean_text
        )
    )

    df[
        "clean_description"
    ] = (
        df[
            "descriptionText"
        ]
        .apply(
            clean_description
        )
    )

    df[
        "clean_url"
    ] = (
        df[
            "jobUrl"
        ]
        .apply(
            clean_text
        )
    )

    return df

## 10.5 Extract Production Location Fields

In [ ]:
STATE_ABBREVIATIONS = {
    "NSW": "NSW",
    "VIC": "VIC",
    "QLD": "QLD",
    "SA": "SA",
    "WA": "WA",
    "TAS": "TAS",
    "NT": "NT",
    "ACT": "ACT"
}

STATE_FULL_NAMES = {
    "NEW SOUTH WALES": "NSW",
    "VICTORIA": "VIC",
    "QUEENSLAND": "QLD",
    "SOUTH AUSTRALIA": "SA",
    "WESTERN AUSTRALIA": "WA",
    "TASMANIA": "TAS",
    "NORTHERN TERRITORY": "NT",
    "AUSTRALIAN CAPITAL TERRITORY": "ACT"
}

def extract_state_from_location(
    location
):

    if not isinstance(
        location,
        dict
    ):
        return None

    address = (
        location.get(
            "formattedAddressShort"
        )
        or location.get(
            "formattedAddressLong"
        )
    )

    if not address:
        return None

    address_upper = (
        str(address)
        .strip()
        .upper()
    )

    # Check state abbreviations first
    for abbreviation in (
        STATE_ABBREVIATIONS
    ):

        pattern = (
            rf"\b{re.escape(abbreviation)}\b"
        )

        if re.search(
            pattern,
            address_upper
        ):
            return (
                STATE_ABBREVIATIONS[
                    abbreviation
                ]
            )

    # Fall back to full state names
    for (
        full_name,
        abbreviation
    ) in STATE_FULL_NAMES.items():

        if (
            full_name
            in address_upper
        ):
            return abbreviation

    return None

In [ ]:
def add_location_components(
    dataframe
):

    df = dataframe.copy()

    df[
        "clean_location"
    ] = (
        df[
            "location"
        ]
        .apply(
            lambda x:
                extract_location_value(
                    x,
                    "formattedAddressShort"
                )
        )
        .apply(
            clean_text
        )
    )

    df[
        "clean_city"
    ] = (
        df[
            "location"
        ]
        .apply(
            lambda x:
                extract_location_value(
                    x,
                    "city"
                )
        )
        .apply(
            clean_text
        )
    )

    df[
        "clean_state"
    ] = (
        df[
            "location"
        ]
        .apply(
            extract_state_from_location
        )
        .apply(
            clean_text
        )
    )

    return df

## 10.6 Extract and Validate Coordinates

In [ ]:
def add_clean_coordinates(
    dataframe
):

    df = dataframe.copy()

    df[
        "latitude"
    ] = (
        dataframe[
            "location"
        ]
        .apply(
            lambda x:
                extract_location_value(
                    x,
                    "latitude"
                )
        )
    )

    df[
        "longitude"
    ] = (
        dataframe[
            "location"
        ]
        .apply(
            lambda x:
                extract_location_value(
                    x,
                    "longitude"
                )
        )
    )

    df[
        "latitude"
    ] = pd.to_numeric(
        df[
            "latitude"
        ],
        errors="coerce"
    )

    df[
        "longitude"
    ] = pd.to_numeric(
        df[
            "longitude"
        ],
        errors="coerce"
    )

    return df

## 10.7 Remove Invalid Coordinates

In [ ]:
def validate_coordinates(
    dataframe
):

    df = dataframe.copy()

    invalid_lat = (
        df[
            "latitude"
        ].notna()
        &
        ~df[
            "latitude"
        ].between(
            -90,
            90
        )
    )

    invalid_lon = (
        df[
            "longitude"
        ].notna()
        &
        ~df[
            "longitude"
        ].between(
            -180,
            180
        )
    )

    df.loc[
        invalid_lat,
        "latitude"
    ] = np.nan

    df.loc[
        invalid_lon,
        "longitude"
    ] = np.nan

    return df

## 10.8 Parse Production Posting Dates

In [ ]:
def add_clean_posted_date(
    dataframe
):

    df = dataframe.copy()

    df[
        "clean_posted_date"
    ] = pd.to_datetime(
        df[
            "datePublished"
        ],
        errors="coerce",
        utc=True
    )

    return df

## 10.9 Prepare Matching Fields

In [ ]:
def add_match_fields(
    dataframe
):

    df = dataframe.copy()

    df[
        "match_company"
    ] = (
        df[
            "clean_company"
        ]
        .apply(
            normalise_match_text
        )
    )

    df[
        "match_title"
    ] = (
        df[
            "clean_title"
        ]
        .apply(
            normalise_match_text
        )
    )

    df[
        "match_location"
    ] = (
        df[
            "clean_location"
        ]
        .apply(
            normalise_match_text
        )
    )

    return df

## 10.10 Define Complete Source Cleaning Pipeline

In [ ]:
def clean_source_pipeline(
    dataframe
):

    df = (
        prepare_source_for_pipeline(
            dataframe
        )
    )

    df = (
        add_location_components(
            df
        )
    )

    df = (
        add_clean_coordinates(
            df
        )
    )

    df = (
        validate_coordinates(
            df
        )
    )

    df = (
        add_clean_posted_date(
            df
        )
    )

    df = (
        add_match_fields(
            df
        )
    )

    return df

## 10.11 Run Production Source Cleaning

In [ ]:
production_df = (
    clean_source_pipeline(
        source_eda_df
    )
)
print(
    "Records after cleaning:",
    len(
        production_df
    )
)

Records after cleaning: 20


## 10.12 Remove Duplicate jobKey Records

This is the source-level deduplication step that precedes all cross-dataset comparisons.

In [ ]:
production_rows_before_jobkey = (
    len(
        production_df
    )
)
production_jobkey_duplicate_mask = (
    production_df[
        "clean_jobKey"
    ].notna()
    &
    production_df[
        "clean_jobKey"
    ].duplicated(
        keep="first"
    )
)
production_df = (
    production_df.loc[
        ~production_jobkey_duplicate_mask
    ]
    .copy()
    .reset_index(drop=True)
)
print(
    "jobKey duplicates removed:",
    production_rows_before_jobkey
    - len(
        production_df
    )
)

jobKey duplicates removed: 0


## 10.13 Define AI-Only Production Location Function

In [ ]:
def apply_ai_location(
    dataframe
):

    df = dataframe.copy()

    results = []

    for _, row in (
        df.iterrows()
    ):

        result = (
            classify_location_ai(
                location=(
                    row[
                        "clean_location"
                    ]
                ),
                city=(
                    row[
                        "clean_city"
                    ]
                ),
                state=(
                    row[
                        "clean_state"
                    ]
                )
            )
        )

        results.append(
            result[
                "prediction"
            ]
        )

    df[
        "location_type"
    ] = results

    return df

## 10.14 Define GCCSA Hybrid Production Location Function

In [ ]:
def apply_hybrid_location(
    dataframe
):

    df = dataframe.copy()

    df[
        "location_type"
    ] = None

    valid_coordinates = (
        df[
            "latitude"
        ].notna()
        &
        df[
            "longitude"
        ].notna()
    )

    coordinate_df = (
        df.loc[
            valid_coordinates
        ]
        .copy()
    )

    if len(
        coordinate_df
    ) > 0:

        points = (
            gpd.GeoDataFrame(
                coordinate_df,
                geometry=(
                    gpd.points_from_xy(
                        coordinate_df[
                            "longitude"
                        ],
                        coordinate_df[
                            "latitude"
                        ]
                    )
                ),
                crs="EPSG:4326"
            )
        )

        points = (
            points.to_crs(
                gccsa_gdf.crs
            )
        )
        joined = (
            gpd.sjoin(
                points,
                gccsa_gdf[
                    [
                        GCCSA_NAME_FIELD,
                        "geometry"
                    ]
                ],
                how="left",
                predicate="within"
            )
        )

        joined[
            "deterministic_location_type"
        ] = (
            joined[
                GCCSA_NAME_FIELD
            ]
            .apply(
                classify_gccsa_name
            )
        )

        for index, value in (
            joined[
                "deterministic_location_type"
            ]
            .items()
        ):

            if pd.notna(
                value
            ):

                df.loc[
                    index,
                    "location_type"
                ] = value
    fallback_mask = (
        df[
            "location_type"
        ]
        .isna()
    )

    for index, row in (
        df.loc[
            fallback_mask
        ]
        .iterrows()
    ):

        result = (
            classify_location_ai(
                location=(
                    row[
                        "clean_location"
                    ]
                ),
                city=(
                    row[
                        "clean_city"
                    ]
                ),
                state=(
                    row[
                        "clean_state"
                    ]
                )
            )
        )

        df.loc[
            index,
            "location_type"
        ] = (
            result[
                "prediction"
            ]
        )

    return df

## 10.15 Apply the Selected Location Method

In [ ]:
if (
    PRODUCTION_CONFIG[
        "location_case"
    ]
    == "Case 1"
):

    production_df = (
        apply_ai_location(
            production_df
        )
    )

elif (
    PRODUCTION_CONFIG[
        "location_case"
    ]
    == "Case 2"
):

    production_df = (
        apply_hybrid_location(
            production_df
        )
    )

else:

    raise ValueError(
        "No valid production "
        "location method selected."
    )

## 10.16 Validate Location Outputs

In [ ]:
invalid_location_output_mask = (
    ~production_df[
        "location_type"
    ]
    .isin(
        VALID_LOCATION_TYPES
    )
)
print(
    "Unresolved / invalid location types:",
    invalid_location_output_mask.sum()
)

Unresolved / invalid location types: 0


## 10.17 Prepare the Clean 48-Hour Target Dataset
Use the cleaned comparison dataset generated in Section 5.

In [ ]:
lookback_start = (
    source_modified_time
    - timedelta(hours=48)
)
print(
    "Pipeline anchor time :",
    source_modified_time
)

print(
    "48-hour lookback start :",
    lookback_start
)

Pipeline anchor time : 2026-08-24 05:04:38.589000+00:00
48-hour lookback start : 2026-08-22 05:04:38.589000+00:00


In [ ]:
target_query = f"""
SELECT *
FROM `{TARGET_TABLE}`
WHERE api_ini_dt_time BETWEEN @lookback_start
                          AND @anchor_time
"""
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter(
            "lookback_start",
            "TIMESTAMP",
            lookback_start
        ),
        bigquery.ScalarQueryParameter(
            "anchor_time",
            "TIMESTAMP",
            source_modified_time
        )
    ]
)
production_target_raw_df = (
    bq_client
    .query(
        target_query,
        job_config=job_config
    )
    .to_dataframe()
)
print(
    f"Target records loaded: "
    f"{len(production_target_raw_df):,}"
)

print(
    f"Target columns loaded: "
    f"{len(production_target_raw_df.columns)}"
)

production_target_df = ( production_target_raw_df .copy(deep=True) )

Target records loaded: 370
Target columns loaded: 20


### 10.17.1 Define Production Target Cleaning Pipeline

The target records retrieved from vw_dca_jobboard should be cleaned using the same standardisation logic used for the incoming source records.

The production target-cleaning function performs:

company-name cleaning;
job-title cleaning;
location cleaning;
job-description cleaning;
posting-date parsing;
company matching normalisation;
title matching normalisation;
location matching normalisation.

In [ ]:
def clean_target_pipeline(
    dataframe
):

    df = dataframe.copy(
        deep=True
    )

    # -------------------------
    # Clean company
    # -------------------------

    df[
        "clean_company"
    ] = (
        df[
            "company_name"
        ]
        .apply(
            clean_text
        )
    )

    # -------------------------
    # Clean job title
    # -------------------------

    df[
        "clean_title"
    ] = (
        df[
            "job_title"
        ]
        .apply(
            clean_text
        )
    )

    # -------------------------
    # Clean location
    # -------------------------

    df[
        "clean_location"
    ] = (
        df[
            "location"
        ]
        .apply(
            clean_text
        )
    )

    # -------------------------
    # Clean job description
    # -------------------------

    df[
        "clean_description"
    ] = (
        df[
            "job_description"
        ]
        .apply(
            clean_description
        )
    )

    # -------------------------
    # Parse posting date
    # -------------------------

    df[
        "clean_posted_date"
    ] = pd.to_datetime(
        df[
            "posted_date"
        ],
        errors="coerce",
        utc=True
    )

    # -------------------------
    # Matching company
    # -------------------------

    df[
        "match_company"
    ] = (
        df[
            "clean_company"
        ]
        .apply(
            normalise_match_text
        )
    )

    # -------------------------
    # Matching title
    # -------------------------

    df[
        "match_title"
    ] = (
        df[
            "clean_title"
        ]
        .apply(
            normalise_match_text
        )
    )

    # -------------------------
    # Matching location
    # -------------------------

    df[
        "match_location"
    ] = (
        df[
            "clean_location"
        ]
        .apply(
            normalise_match_text
        )
    )

    return df

### 10.17.2 Clean the Production Target Dataset

In [ ]:
production_target_df = ( clean_target_pipeline( production_target_raw_df ) )

### 10.17.3 Inspect the Cleaned Target Dataset

In [ ]:
display(
    production_target_df[
        [
            "id",
            "clean_company",
            "clean_title",
            "clean_location",
            "clean_posted_date",
            "clean_description",
            "match_company",
            "match_title",
            "match_location"
        ]
    ].head()
)

,id,clean_company,clean_title,clean_location,clean_posted_date,clean_description,match_company,match_title,match_location
0,9f7630e6-8a1e-46f2-bb0e-6112c628bdc9,Country Road Group Pty Ltd,Business Analyst (Data & Analytics) - Country ...,"{'area': 'CBD & Inner Suburbs', 'displayLocati...",2026-08-21 00:00:00+00:00,"As the Business Analyst for Data & Analytics, ...",country road group pty ltd,business analyst (data & analytics) - country ...,"{'area': 'cbd & inner suburbs', 'displaylocati..."
1,988a980a-2b65-4727-bff7-f932cadfe3b6,Victoria Legal Aid,Data Integrity Lead,"{'area': 'CBD & Inner Suburbs', 'displayLocati...",2026-08-21 00:00:00+00:00,The Data Integrity Lead ensures Victoria Legal...,victoria legal aid,data integrity lead,"{'area': 'cbd & inner suburbs', 'displaylocati..."
2,ffca8643-f604-4bdb-9300-2b11f08efbde,Jellyfish,Digital Analytics Specialist,"{'area': 'CBD & Inner Suburbs', 'displayLocati...",2026-08-21 00:00:00+00:00,The Analytics Executive acts as an implementat...,jellyfish,digital analytics specialist,"{'area': 'cbd & inner suburbs', 'displaylocati..."
3,ba01df52-6e17-460a-946e-7207c64146e4,Google,"Customer Engineering Manager, Data Analytics, ...","{'area': 'CBD, Inner West & Eastern Suburbs', ...",2026-08-21 00:00:00+00:00,Empower businesses to digitally transform with...,google,"customer engineering manager, data analytics, ...","{'area': 'cbd, inner west & eastern suburbs', ..."
4,fdc5a556-e766-4019-b7a0-d32584319425,Cleared Recruitment,Senior Data Integration Engineer,"{'area': None, 'displayLocation': 'Canberra AC...",2026-08-22 00:00:00+00:00,Join a global Defence technology leader delive...,cleared recruitment,senior data integration engineer,"{'area': none, 'displaylocation': 'canberra ac..."


### 10.17.4 Validate Target Cleaning

In [ ]:
PRODUCTION_TARGET_CLEAN_FIELDS = [
    "clean_company",
    "clean_title",
    "clean_location",
    "clean_posted_date",
    "clean_description",
    "match_company",
    "match_title",
    "match_location"
]
production_target_quality_df = pd.DataFrame({
    "field":
        PRODUCTION_TARGET_CLEAN_FIELDS,

    "missing_count":
        [
            production_target_df[
                field
            ]
            .isna()
            .sum()
            for field
            in PRODUCTION_TARGET_CLEAN_FIELDS
        ],

    "missing_pct":
        [
            round(
                production_target_df[
                    field
                ]
                .isna()
                .mean()
                * 100,
                2
            )
            if len(
                production_target_df
            ) > 0
            else 0
            for field
            in PRODUCTION_TARGET_CLEAN_FIELDS
        ]
})
display(
    production_target_quality_df
)

,field,missing_count,missing_pct
0,clean_company,2,0.54
1,clean_title,0,0.00
2,clean_location,0,0.00
3,clean_posted_date,0,0.00
4,clean_description,0,0.00
5,match_company,2,0.54
6,match_title,0,0.00
7,match_location,0,0.00


## 10.18 Deduplication

### 10.18.1 Define Deduplication Case 1

In [ ]:
def production_dedup_case1(
    source_df,
    target_df
):

    source = (
        source_df.copy()
    )

    target = (
        target_df.copy()
    )

    fields = [
        "match_company",
        "match_title",
        "match_location"
    ]

    source[
        "_dedup_key"
    ] = (
        source[
            fields
        ]
        .fillna('')
        .astype("string")
        .agg(
            "|".join,
            axis=1
        )
    )

    target[
        "_dedup_key"
    ] = (
        target[
            fields
        ]
        .fillna('')
        .astype("string")
        .agg(
            "|".join,
            axis=1
        )
    )

    target_keys = set(
        target[
            "_dedup_key"
        ]
        .dropna()
    )

    source[
        "is_duplicate"
    ] = (
        source[
            "_dedup_key"
        ]
        .isin(
            target_keys
        )
    )

    return source

### 10.18.2 Define Deduplication Case 2

In [ ]:
def production_dedup_case2(
    source_df,
    target_df,
    threshold=
        DATE_SIMILARITY_THRESHOLD
):

    source = (
        source_df.copy()
    )

    target = (
        target_df.copy()
    )

    source[
        "_base_key"
    ] = (
        source[
            "match_company"
        ]
        + "|"
        + source[
            "match_title"
        ]
        + "|"
        + source[
            "match_location"
        ]
    )

    target[
        "_base_key"
    ] = (
        target[
            "match_company"
        ]
        + "|"
        + target[
            "match_title"
        ]
        + "|"
        + target[
            "match_location"
        ]
    )
    source[
        "_source_row_id"
    ] = (
        np.arange(
            len(source)
        )
    )

    candidates = (
        source[
            [
                "_source_row_id",
                "_base_key",
                "clean_posted_date"
            ]
        ]
        .merge(
            target[
                [
                    "_base_key",
                    "clean_posted_date"
                ]
            ],
            on="_base_key",
            how="inner",
            suffixes=(
                "_source",
                "_target"
            )
        )
    )

    candidates[
        "date_similarity_pct"
    ] = (
        candidates.apply(
            lambda row:
                calculate_date_similarity(
                    row[
                        "clean_posted_date_source"
                    ],
                    row[
                        "clean_posted_date_target"
                    ]
                ),
            axis=1
        )
    )
    matched_ids = set(
        candidates.loc[
            candidates[
                "date_similarity_pct"
            ]
            >= threshold,
            "_source_row_id"
        ]
    )

    source[
        "is_duplicate"
    ] = (
        source[
            "_source_row_id"
        ]
        .isin(
            matched_ids
        )
    )

    return source

### 10.18.3 Define Deduplication Case 3

In [ ]:
def production_dedup_case3(
    source_df,
    target_df
):

    source = (
        source_df.copy()
    )

    target = (
        target_df.copy()
    )

    source[
        "descriptionTextHash"
    ] = (
        source[
            "clean_description"
        ]
        .apply(
            create_description_text_hash
        )
    )

    target[
        "descriptionTextHash"
    ] = (
        target[
            "clean_description"
        ]
        .apply(
            create_description_text_hash
        )
    )

    fields = [
        "match_company",
        "match_title",
        "match_location",
        "descriptionTextHash"
    ]
    source_valid = (
        source[
            fields
        ]
        .notna()
        .all(axis=1)
    )

    target_valid = (
        target[
            fields
        ]
        .notna()
        .all(axis=1)
    )

    source[
        "_dedup_key"
    ] = None

    target[
        "_dedup_key"
    ] = None
    source.loc[
        source_valid,
        "_dedup_key"
    ] = (
        source.loc[
            source_valid,
            fields
        ]
        .astype(str)
        .agg(
            "|".join,
            axis=1
        )
    )

    target.loc[
        target_valid,
        "_dedup_key"
    ] = (
        target.loc[
            target_valid,
            fields
        ]
        .astype(str)
        .agg(
            "|".join,
            axis=1
        )
    )

    target_keys = set(
        target[
            "_dedup_key"
        ]
        .dropna()
    )
    source[
        "is_duplicate"
    ] = (
        source[
            "_dedup_key"
        ]
        .notna()
        &
        source[
            "_dedup_key"
        ]
        .isin(
            target_keys
        )
    )

    return source

### 10.18.4 Apply the Selected Deduplication Method

In [ ]:
if (
    PRODUCTION_CONFIG[
        "dedup_case"
    ]
    == "Case 1"
):

    production_df = (
        production_dedup_case1(
            production_df,
            production_target_df
        )
    )

elif (
    PRODUCTION_CONFIG[
        "dedup_case"
    ]
    == "Case 2"
):

    production_df = (
        production_dedup_case2(
            production_df,
            production_target_df
        )
    )

elif (
    PRODUCTION_CONFIG[
        "dedup_case"
    ]
    == "Case 3"
):

    production_df = (
        production_dedup_case3(
            production_df,
            production_target_df
        )
    )

else:

    raise ValueError(
        "No valid production "
        "deduplication method selected."
    )

### 10.18.5 Inspect Production Duplicate Decisions

In [ ]:
production_duplicate_count = (
    production_df[
        "is_duplicate"
    ]
    .sum()
)
print(
    "Cross-dataset duplicates identified:",
    production_duplicate_count
)

Cross-dataset duplicates identified: 0


### 10.18.6 Retain New Records Only

In [ ]:
production_new_df = (
    production_df.loc[
        ~production_df[
            "is_duplicate"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)
print(
    "Records remaining after deduplication:",
    len(
        production_new_df
    )
)

Records remaining after deduplication: 20


## 10.19 AI Enrichment

### 10.19.1 Prepare AI input fields

In [ ]:
production_new_df[
    "job_description"
] = (
    production_new_df[
        "clean_description"
    ]
)
production_new_df[
    "job_title"
] = (
    production_new_df[
        "clean_title"
    ]
)
if (
    "industry"
    in production_new_df.columns
):

    production_new_df[
        "source_industry"
    ] = (
        production_new_df[
            "industry"
        ]
        .apply(
            clean_text
        )
    )

else:

    production_new_df[
        "source_industry"
    ] = None

### 10.19.2 Define Combined Notebook AI Production Function

In [ ]:
def production_ai_case1(
    dataframe
):

    df = dataframe.copy()

    outputs = []

    for _, row in (
        df.iterrows()
    ):

        prompt = (
            CASE1_COMBINED_PROMPT
            .format(
                job_title=(
                    row[
                        "job_title"
                    ]
                    or "None"
                ),
                job_description=(
                    row[
                        "job_description"
                    ]
                    or "None"
                ),
                source_industry=(
                    row[
                        "source_industry"
                    ]
                    or "None"
                )
            )
        )

        api_result = (
            call_gemini(
                prompt
            )
        )

        parsed = (
            parse_json_response(
                api_result[
                    "text"
                ]
            )
        )

        validated = (
            validate_combined_result(
                parsed
            )
        )

        outputs.append(
            validated
        )

    return (
        attach_ai_outputs(
            df,
            outputs
        )
    )

### 10.19.3 Define Helper to Attach AI Outputs

In [ ]:
def attach_ai_outputs(
    dataframe,
    outputs
):

    df = dataframe.copy()

    for field in (
        AI_ENRICHMENT_FIELDS
    ):

        df[
            field
        ] = [
            (
                result.get(
                    field
                )
                if isinstance(
                    result,
                    dict
                )
                else None
            )
            for result in outputs
        ]

    return df

### 10.19.4 Define Separate Notebook AI Production Function

In [ ]:
def production_ai_case2(
    dataframe
):

    df = dataframe.copy()

    outputs = []

    for _, row in (
        df.iterrows()
    ):

        result, _ = (
            run_separate_prompts(
                row,
                CASE2_PROMPTS
            )
        )

        outputs.append(
            result
        )

    return (
        attach_ai_outputs(
            df,
            outputs
        )
    )

### 10.19.5 Define Combined JSON AI Production Function

In [ ]:
def production_ai_case3(
    dataframe
):

    df = dataframe.copy()

    outputs = []

    for _, row in (
        df.iterrows()
    ):

        prompt = (
            build_json_combined_prompt(
                row
            )
        )

        api_result = (
            call_gemini(
                prompt,
                model=case3_model
            )
        )

        parsed = (
            parse_json_response(
                api_result[
                    "text"
                ]
            )
        )

        validated = (
            validate_combined_result(
                parsed
            )
        )

        outputs.append(
            validated
        )

    return (
        attach_ai_outputs(
            df,
            outputs
        )
    )

### 10.19.6 Define Separate JSON AI Production Function

In [ ]:
def production_ai_case4(
    dataframe
):

    df = dataframe.copy()

    outputs = []

    for _, row in (
        df.iterrows()
    ):

        result, _ = (
            run_separate_prompts(
                row,
                JSON_PROMPT_STRINGS,
                model=case3_model
            )
        )

        outputs.append(
            result
        )

    return (
        attach_ai_outputs(
            df,
            outputs
        )
    )

### 10.19.7 Apply the Selected AI Method

In [ ]:
if (
    PRODUCTION_CONFIG[
        "ai_case"
    ]
    == "Case 1"
):

    production_enriched_df = (
        production_ai_case1(
            production_new_df
        )
    )

elif (
    PRODUCTION_CONFIG[
        "ai_case"
    ]
    == "Case 2"
):

    production_enriched_df = (
        production_ai_case2(
            production_new_df
        )
    )

elif (
    PRODUCTION_CONFIG[
        "ai_case"
    ]
    == "Case 3"
):

    production_enriched_df = (
        production_ai_case3(
            production_new_df
        )
    )

elif (
    PRODUCTION_CONFIG[
        "ai_case"
    ]
    == "Case 4"
):

    production_enriched_df = (
        production_ai_case4(
            production_new_df
        )
    )

else:

    raise ValueError(
        "No valid production "
        "AI methodology selected."
    )

### 10.19.8 Inspect AI-Enriched Records

In [ ]:
display(
    production_enriched_df[
        [
            "clean_jobKey",
            "clean_title",
            "top_tech_skills",
            "role_cat",
            "function",
            "industry",
            "exp_level",
            "pr_citizenship_req",
            "sec_clearance"
        ]
    ]
    .loc[12:13]
    # .loc[
    #     production_enriched_df[
    #         "exp_level"
    #     ].notna()
    # ]
)

,clean_jobKey,clean_title,top_tech_skills,role_cat,function,industry,exp_level,pr_citizenship_req,sec_clearance
12,fbc1eabe01135c57,Database Admin,"SQL Server, PostgreSQL, Azure, Ansible, GitHub...",Data Engineer,"IT Operations, Software Development",Software,None,None mentioned,False
13,31573efd5c4b4eec,Senior Predictive Analytics Consultant,"Python, R, SQL, MS Fabric, Databricks, Power B...",Data Analyst,"Insurance, Pricing, Actuarial",Insurance,None,None mentioned,False


## 10.20  Create Pipeline Metadata

Generate the pipeline execution timestamp once so all rows in this run receive the same value.

In [ ]:
PIPELINE_RUN_TIMESTAMP = (
    pd.Timestamp.now(
        tz="UTC"
    )
)
print(
    "Pipeline run timestamp:",
    PIPELINE_RUN_TIMESTAMP
)

Pipeline run timestamp: 2026-08-31 05:22:38.366073+00:00


## 10.21 Define Source Metadata Values

In [ ]:
PIPELINE_API_NAME = (
    "Indeed Jobs"
)
PIPELINE_SOURCE = (
    "Indeed"
)

## 10.22 Map Cleaned Fields to Final Target Fields

In [ ]:
final_pipeline_df = (
    pd.DataFrame(
        index=(
            production_enriched_df.index
        )
    )
)

### 10.22.1 Core Identifiers and Metadata

In [ ]:
final_pipeline_df[
    "id"
] = (
    production_enriched_df[
        "clean_jobKey"
    ]
)
final_pipeline_df[
    "api_ini_dt_time"
] = (
    PIPELINE_RUN_TIMESTAMP
)
final_pipeline_df[
    "api_name"
] = (
    PIPELINE_API_NAME
)

### 10.22.2 Core Job Fields

In [ ]:
final_pipeline_df[
    "posted_date"
] = (
    production_enriched_df[
        "clean_posted_date"
    ]
)
final_pipeline_df[
    "job_title"
] = (
    production_enriched_df[
        "clean_title"
    ]
)
final_pipeline_df[
    "company_name"
] = (
    production_enriched_df[
        "clean_company"
    ]
)

### 10.22.3 Location Fields

In [ ]:
final_pipeline_df[
    "city"
] = (
    production_enriched_df[
        "clean_city"
    ]
)
final_pipeline_df[
    "state"
] = (
    production_enriched_df[
        "clean_state"
    ]
)
final_pipeline_df[
    "location"
] = (
    production_enriched_df[
        "clean_location"
    ]
)

final_pipeline_df[
    "location_type"
] = (
    production_enriched_df[
        "location_type"
    ]
)

### 10.22.4 Description and URL

In [ ]:
final_pipeline_df[
    "job_description"
] = (
    production_enriched_df[
        "clean_description"
    ]
)
final_pipeline_df[
    "url"
] = (
    production_enriched_df[
        "clean_url"
    ]
)

### 10.22.5 Source

In [ ]:
final_pipeline_df[ "source" ] = ( PIPELINE_SOURCE )

### 10.22.6 AI-Enriched Fields

In [ ]:
for field in [
    "industry",
    "function",
    "top_tech_skills",
    "exp_level",
    "pr_citizenship_req",
    "sec_clearance",
    "role_cat"
]:

    final_pipeline_df[
        field
    ] = (
        production_enriched_df[
            field
        ]
    )

## 10.23 Enforce Final 20-Field Order

In [ ]:
final_pipeline_df = (
    final_pipeline_df[
        FINAL_TARGET_FIELDS
    ]
    .copy()
)
print(
    "Final output column count:",
    len(
        final_pipeline_df.columns
    )
)

display(
    final_pipeline_df.head(
        2
    ))

Final output column count: 20


,id,api_ini_dt_time,api_name,posted_date,job_title,company_name,city,state,location,job_description,url,source,industry,function,top_tech_skills,exp_level,pr_citizenship_req,sec_clearance,location_type,role_cat
0,9c2916ac1bdaf7e2,2026-08-31 05:22:38.366073+00:00,Indeed Jobs,2026-08-21 00:00:00+00:00,Data Analyst,Department of Education,Melbourne City Centre,VIC,Melbourne City Centre VIC,This position commences 1 January 2027 Are you...,https://au.indeed.com/viewjob?jk=9c2916ac1bdaf7e2,Indeed,Education,"Education, Professional Development, Performan...","Excel, Power BI",None,None mentioned,False,Metro,Data Analyst
1,1651d59366f4f3a1,2026-08-31 05:22:38.366073+00:00,Indeed Jobs,2026-08-21 00:00:00+00:00,Manager - Governance of Indigenous Data and In...,CSIRO,Brisbane,QLD,Brisbane QLD,Acknowledgement of Country CSIRO acknowledges ...,https://au.indeed.com/viewjob?jk=1651d59366f4f3a1,Indeed,Scientific Research,"Governance, Policy Development, Indigenous Rel...",None,None,Citizenship,True,Metro,None


### 10.24 Validate Exact Target Schema

In [ ]:
assert (
    final_pipeline_df.columns.tolist()
    ==
    FINAL_TARGET_FIELDS
), (
    "Final output does not match "
    "the required 20-field schema."
)
print(
    "✓ Final 20-field schema validated."
)

✓ Final 20-field schema validated.


#### 10.25 Validate Final Location Values


In [ ]:
final_invalid_locations = (
    final_pipeline_df[
        "location_type"
    ]
    .notna()
    &
    ~final_pipeline_df[
        "location_type"
    ]
    .isin(
        VALID_LOCATION_TYPES
    )
)
print(
    "Invalid final location types:",
    final_invalid_locations.sum()
)

Invalid final location types: 0


### 10.26 Validate Security-Clearance Type

In [ ]:
invalid_sec_clearance = (
    final_pipeline_df[
        "sec_clearance"
    ]
    .notna()
    &
    ~final_pipeline_df[
        "sec_clearance"
    ]
    .apply(
        lambda x:
            isinstance(
                x,
                bool
            )
    )
)
print(
    "Invalid sec_clearance values:",
    invalid_sec_clearance.sum()
)

Invalid sec_clearance values: 0


### 10.27 Check Final Null Rates

In [ ]:
final_null_profile_df = (
    pd.DataFrame({
        "field":
            FINAL_TARGET_FIELDS,

        "null_count":
            [
                final_pipeline_df[
                    field
                ].isna().sum()
                for field
                in FINAL_TARGET_FIELDS
            ],

        "null_pct":
            [
                round(
                    final_pipeline_df[
                        field
                    ]
                    .isna()
                    .mean()
                    * 100,
                    2
                )
                for field
                in FINAL_TARGET_FIELDS
            ]
    })
)
display(
    final_null_profile_df
)

,field,null_count,null_pct
0,id,0,0.0
1,api_ini_dt_time,0,0.0
2,api_name,0,0.0
3,posted_date,0,0.0
4,job_title,0,0.0
5,company_name,0,0.0
6,city,0,0.0
7,state,0,0.0
8,location,0,0.0
9,job_description,0,0.0


## 10.28 Check Final Record Counts

In [ ]:
pipeline_record_summary = pd.DataFrame({
    "Stage": [
        "Raw source records",
        "After jobKey deduplication",
        "After cross-dataset deduplication",
        "Final enriched records"
    ],

    "Record Count": [
        production_rows_before_jobkey,
        len(
            production_df
        ),
        len(
            production_new_df
        ),
        len(
            final_pipeline_df
        )
    ]
})
display(
    pipeline_record_summary
)

,Stage,Record Count
0,Raw source records,20
1,After jobKey deduplication,20
2,After cross-dataset deduplication,20
3,Final enriched records,20


## 10.29 Check for Final Duplicate IDs

In [ ]:
final_duplicate_ids = (
    final_pipeline_df[
        "id"
    ]
    .notna()
    &
    final_pipeline_df[
        "id"
    ]
    .duplicated(
        keep=False
    )
)
print(
    "Duplicate final IDs:",
    final_duplicate_ids.sum()
)

Duplicate final IDs: 0


## 10.30 Validate Final Row Preservation After Enrichment

AI enrichment should enrich records, not silently remove them.

In [ ]:
assert (
    len(
        production_new_df
    )
    ==
    len(
        final_pipeline_df
    )
), (
    "Records were lost during "
    "AI enrichment or final mapping."
)
print(
    "✓ AI enrichment preserved "
    "all retained records."
)

✓ AI enrichment preserved all retained records.


## 10.31 Create Production Pipeline Summary

In [ ]:
FINAL_PIPELINE_SUMMARY = {
    "location_method":
        PRODUCTION_CONFIG[
            "location_case"
        ],

    "dedup_method":
        PRODUCTION_CONFIG[
            "dedup_case"
        ],

    "ai_method":
        PRODUCTION_CONFIG[
            "ai_case"
        ],

    "source_records":
        production_rows_before_jobkey,

    "jobkey_duplicates_removed":
        (
            production_rows_before_jobkey
            - len(
                production_df
            )
        ),

    "cross_dataset_duplicates_removed":
        int(
            production_duplicate_count
        ),

    "records_ready_for_staging":
        len(
            final_pipeline_df
        )
}
display(
    pd.DataFrame(
        FINAL_PIPELINE_SUMMARY.items(),
        columns=[
            "Metric",
            "Value"
        ]
    )
)

,Metric,Value
0,location_method,Case 2
1,dedup_method,Case 1
2,ai_method,Case 1
3,source_records,20
4,jobkey_duplicates_removed,0
5,cross_dataset_duplicates_removed,0
6,records_ready_for_staging,20


## 10.32 Prepare Staging DataFrame

The final dataframe is now ready for validation and eventual staging-table insertion.

In [ ]:
jobboard_staging_df = (
    final_pipeline_df
    .copy(deep=True)
)
display(
    jobboard_staging_df.loc[11:15]
)

,id,api_ini_dt_time,api_name,posted_date,job_title,company_name,city,state,location,job_description,url,source,industry,function,top_tech_skills,exp_level,pr_citizenship_req,sec_clearance,location_type,role_cat
11,2de17585f6301445,2026-08-31 05:22:38.366073+00:00,Indeed Jobs,2026-08-14 00:00:00+00:00,Data Migration Lead,FinXL,Melbourne City Centre,VIC,Melbourne City Centre VIC,Job details Posted14 August 2026 SalaryNegotia...,https://au.indeed.com/viewjob?jk=2de17585f6301445,Indeed,"Telecommunication, Banking and Finance, Commer...","Customer Transformation, Data Migration",APIs,None,None,False,Metro,Data Engineer
12,fbc1eabe01135c57,2026-08-31 05:22:38.366073+00:00,Indeed Jobs,2026-08-13 00:00:00+00:00,Database Admin,Epicor Software,Sydney,NSW,Sydney NSW,"As a Database Administrator at Epicor , you wi...",https://au.indeed.com/viewjob?jk=fbc1eabe01135c57,Indeed,Software,"IT Operations, Software Development","SQL Server, PostgreSQL, Azure, Ansible, GitHub...",None,None mentioned,False,Metro,Data Engineer
13,31573efd5c4b4eec,2026-08-31 05:22:38.366073+00:00,Indeed Jobs,2026-08-12 00:00:00+00:00,Senior Predictive Analytics Consultant,Guild Group,Melbourne City Centre,VIC,Melbourne City Centre VIC,Job Number: ELMO-00001363 Work type: Full Time...,https://au.indeed.com/viewjob?jk=31573efd5c4b4eec,Indeed,Insurance,"Insurance, Pricing, Actuarial","Python, R, SQL, MS Fabric, Databricks, Power B...",None,None mentioned,False,Metro,Data Analyst
14,fd54fe389005a5d2,2026-08-31 05:22:38.366073+00:00,Indeed Jobs,2026-08-12 00:00:00+00:00,Sr. Data Analyst,FinXL,Melbourne City Centre,VIC,Melbourne City Centre VIC,Job details Posted12 August 2026 SalaryNegotia...,https://au.indeed.com/viewjob?jk=fd54fe389005a5d2,Indeed,"Energy and Utilities,Telecommunications,Bankin...","Energy Retail,Billing,Metering,Tariff Management","SQL,Oracle Utilities CC&B/C1,Databricks",None,None mentioned,False,Metro,Data Analyst
15,6f94f65fc6c95564,2026-08-31 05:22:38.366073+00:00,Indeed Jobs,2026-08-11 00:00:00+00:00,"Group Manager - Network, Storage and Data Centres",Monash University,Clayton,VIC,Clayton VIC,"Group Manager - Network, Storage and Data Cent...",https://au.indeed.com/viewjob?jk=6f94f65fc6c95564,Indeed,Education,"Information Technology, Infrastructure Management",hybrid cloud,None,None,True,Metro,None


## 10.33 Do Not Write Directly to Production

At this point:

jobboard_staging_df

contains the final transformed records.

The dataframe should next undergo final data-quality checks before it is written to jobboard_staging.

No records should be inserted directly into vw_dca_jobboard from this notebook stage.

## 10.34 Section 10 Outcome

Section 10 converts the experimental notebook into a modular end-to-end production workflow.

**The final pipeline:**

-> starts with incoming records from daily_scrape_dev;

-> applies the cleaning and standardisation rules established in Section 5;

-> extracts and validates location information;

-> removes repeated non-null jobKey records from the incoming source;

-> applies the selected location-type methodology from Section 6;

-> compares incoming records against the 48-hour vw_dca_jobboard dataset using the selected deduplication methodology from Section 7;

-> removes records identified as existing production duplicates;

-> applies the selected AI-enrichment architecture from Section 8;

-> maps the resulting data into exactly 20 target fields;

-> validates core schema and output constraints;

-> and prepares the resulting records as jobboard_staging_df.

**The final mapped fields are:**

- id
- api_ini_dt_time
- api_name
- posted_date
- job_title
- company_name
- city
- state
- location
- job_description
- url
- source
- industry
- function
- top_tech_skills
- exp_level
- pr_citizenship_req
- sec_clearance
- location_type
- role_cat

The production workflow remains configuration driven.

The selected methods are controlled through:

**PRODUCTION_CONFIG**

rather than being hard-coded into the pipeline.

This allows the selected methodology to change if additional validation alters the conclusions of Sections 6–9.

Most importantly, Section 10 does not bypass the review requirement. The final output is prepared as:

***jobboard_staging_df***

rather than being written directly into the production job-board dataset.

The next section will perform the final post-processing data-quality assessment, comparing the cleaned and enriched output against the baseline metrics established in Section 4 and determining whether the staging dataset passes the required quality checks.

# 11. Post-Processing Data Quality and Validation

This section performs the final quality assessment of the production-ready staging dataset created in Section 10.

The purpose is to confirm that the final pipeline output is suitable for staging and review by evaluating:

- record counts;
- schema completeness;
- null rates;
- duplicate rates;
- date validity;
- location-type coverage;
- AI-enrichment coverage;
- text-cleaning quality;
- required categorical values;
- and overall pipeline pass/fail status.

This section does not rerun the location, deduplication, or AI-enrichment experiments.

It validates only the final selected production output:

`jobboard_staging_df`

## 11.1 Validate Final Record Count

In [ ]:
final_record_count = (
    len(
        jobboard_staging_df
    )
)
print(
    "Final records ready for staging:",
    final_record_count
)

Final records ready for staging: 20


## 11.2 Validate Final Column Count

The production output must contain exactly 20 target fields.

In [ ]:
final_column_count = (
    len(
        jobboard_staging_df.columns
    )
)
print(
    "Final column count:",
    final_column_count
)
assert (
    final_column_count
    == 20
), (
    "Final staging dataset does not "
    "contain exactly 20 columns."
)

Final column count: 20


## 11.3 Validate Exact Target Schema

In [ ]:
missing_final_fields = [
    field
    for field
    in FINAL_TARGET_FIELDS
    if field
    not in jobboard_staging_df.columns
]
unexpected_final_fields = [
    field
    for field
    in jobboard_staging_df.columns
    if field
    not in FINAL_TARGET_FIELDS
]
print(
    "Missing final fields:",
    missing_final_fields
)

print(
    "Unexpected final fields:",
    unexpected_final_fields
)
assert (
    len(
        missing_final_fields
    )
    == 0
), (
    "Required target fields are missing."
)
assert (
    len(
        unexpected_final_fields
    )
    == 0
), (
    "Unexpected fields exist in "
    "the final staging dataset."
)

Missing final fields: []
Unexpected final fields: []


## 11.4 Final Null-Rate Analysis

In [ ]:
final_null_analysis_df = pd.DataFrame({
    "field":
        FINAL_TARGET_FIELDS,

    "null_count":
        [
            jobboard_staging_df[
                field
            ]
            .isna()
            .sum()
            for field
            in FINAL_TARGET_FIELDS
        ],

    "null_pct":
        [
            round(
                jobboard_staging_df[
                    field
                ]
                .isna()
                .mean()
                * 100,
                2
            )
            if len(
                jobboard_staging_df
            ) > 0
            else 0
            for field
            in FINAL_TARGET_FIELDS
        ]
})
display(
    final_null_analysis_df
)

,field,null_count,null_pct
0,id,0,0.0
1,api_ini_dt_time,0,0.0
2,api_name,0,0.0
3,posted_date,0,0.0
4,job_title,0,0.0
5,company_name,0,0.0
6,city,0,0.0
7,state,0,0.0
8,location,0,0.0
9,job_description,0,0.0


## 11.5 Check Core Required Fields

Some fields are expected to be present for every usable job record.

In [ ]:
CORE_REQUIRED_FIELDS = [
    "id",
    "posted_date",
    "job_title",
    "company_name",
    "location",
    "job_description",
    "url",
    "source"
]

### 11.5.1 Calculate Missing Core Values

In [ ]:
core_required_quality_df = pd.DataFrame({
    "field":
        CORE_REQUIRED_FIELDS,

    "missing_count":
        [
            jobboard_staging_df[
                field
            ]
            .isna()
            .sum()
            for field
            in CORE_REQUIRED_FIELDS
        ],

    "missing_pct":
        [
            round(
                jobboard_staging_df[
                    field
                ]
                .isna()
                .mean()
                * 100,
                2
            )
            if len(
                jobboard_staging_df
            ) > 0
            else 0
            for field
            in CORE_REQUIRED_FIELDS
        ]
})
display(
    core_required_quality_df
)

,field,missing_count,missing_pct
0,id,0,0.0
1,posted_date,0,0.0
2,job_title,0,0.0
3,company_name,0,0.0
4,location,0,0.0
5,job_description,0,0.0
6,url,0,0.0
7,source,0,0.0


## 11.6 Check Duplicate Final IDs

In [ ]:
final_duplicate_id_mask = (
    jobboard_staging_df[
        "id"
    ]
    .notna()
    &
    jobboard_staging_df[
        "id"
    ]
    .duplicated(
        keep=False
    )
)
final_duplicate_id_count = (
    final_duplicate_id_mask.sum()
)
print(
    "Duplicate final ID rows:",
    final_duplicate_id_count
)
1

Duplicate final ID rows: 0


1

## 11.7 Calculate Final Duplicate Rate

In [ ]:
final_duplicate_rate_pct = (
    (
        final_duplicate_id_count
        / final_record_count
        * 100
    )
    if final_record_count > 0
    else 0
)
print(
    f"Final duplicate ID rate: "
    f"{final_duplicate_rate_pct:.2f}%"
)

Final duplicate ID rate: 0.00%


## 11.8 Validate Posting Dates

In [ ]:
final_posted_dates = pd.to_datetime(
    jobboard_staging_df[
        "posted_date"
    ],
    errors="coerce",
    utc=True
)

### 11.8.1 Count Invalid Posting Dates

In [ ]:
invalid_final_dates = (
    jobboard_staging_df[
        "posted_date"
    ]
    .notna()
    &
    final_posted_dates
    .isna()
)
invalid_final_date_count = (
    invalid_final_dates.sum()
)
print(
    "Invalid final posted_date values:",
    invalid_final_date_count
)

Invalid final posted_date values: 0


### 11.8.2 Inspect Final Date Range

In [ ]:
print(
    "Earliest final posted date:",
    final_posted_dates.min()
)
print(
    "Latest final posted date:",
    final_posted_dates.max()
)

Earliest final posted date: 2026-08-10 00:00:00+00:00
Latest final posted date: 2026-08-21 00:00:00+00:00


## 11.9 Validate Pipeline Timestamp

In [ ]:
final_api_times = pd.to_datetime(
    jobboard_staging_df[
        "api_ini_dt_time"
    ],
    errors="coerce",
    utc=True
)
invalid_api_time_count = (
    final_api_times
    .isna()
    .sum()
)
print(
    "Invalid api_ini_dt_time values:",
    invalid_api_time_count
)

Invalid api_ini_dt_time values: 0


## 11.10 Validate `location_type`

The final location value must be one of:

- `Metro`
- `Regional`
- `Remote`

In [ ]:
invalid_location_type_mask = (
    jobboard_staging_df[
        "location_type"
    ]
    .notna()
    &
    ~jobboard_staging_df[
        "location_type"
    ]
    .isin(
        VALID_LOCATION_TYPES
    )
)
invalid_location_type_count = (
    invalid_location_type_mask.sum()
)
print(
    "Invalid location_type values:",
    invalid_location_type_count
)

Invalid location_type values: 0


## 11.11 Calculate Location-Type Coverage

In [ ]:
location_type_coverage_pct = (
    jobboard_staging_df[
        "location_type"
    ]
    .notna()
    .mean()
    * 100
    if final_record_count > 0
    else 0
)
print(
    f"Location type coverage: "
    f"{location_type_coverage_pct:.2f}%"
)

Location type coverage: 100.00%


## 11.12 Inspect Final Location-Type Distribution

In [ ]:
final_location_distribution = (
    jobboard_staging_df[
        "location_type"
    ]
    .value_counts(
        dropna=False
    )
)
display(
    final_location_distribution
    .to_frame(
        "record_count"
    )
)

,record_count
location_type,
Metro,20


## 11.13 Validate Experience-Level Outputs

In [ ]:
invalid_exp_level_mask = (
    jobboard_staging_df[
        "exp_level"
    ]
    .notna()
    &
    ~jobboard_staging_df[
        "exp_level"
    ]
    .isin(
        VALID_EXPERIENCE_LEVELS
    )
)
invalid_exp_level_count = (
    invalid_exp_level_mask.sum()
)
print(
    "Invalid exp_level values:",
    invalid_exp_level_count
)

Invalid exp_level values: 0


## 11.14 Validate `role_cat`

Because `role_cat` may contain one or more comma-separated categories, validate each value individually.

In [ ]:
def valid_role_cat(
    value
):

    if pd.isna(value):
        return True

    values = [
        item.strip()
        for item in str(
            value
        ).split(",")
        if item.strip()
    ]

    return all(
        item
        in VALID_ROLE_CATEGORIES
        for item
        in values
    )
invalid_role_cat_mask = (
    ~jobboard_staging_df[
        "role_cat"
    ]
    .apply(
        valid_role_cat
    )
)
invalid_role_cat_count = (
    invalid_role_cat_mask.sum()
)
print(
    "Invalid role_cat values:",
    invalid_role_cat_count
)

Invalid role_cat values: 0


## 11.15 Validate Residency / Citizenship Requirements

In [ ]:
def valid_pr_requirement(
    value
):

    if pd.isna(value):
        return True

    values = [
        item.strip()
        for item
        in str(
            value
        ).split(",")
        if item.strip()
    ]

    return all(
        item
        in VALID_PR_REQUIREMENTS
        for item
        in values
    )
invalid_pr_requirement_mask = (
    ~jobboard_staging_df[
        "pr_citizenship_req"
    ]
    .apply(
        valid_pr_requirement
    )
)
invalid_pr_requirement_count = (
    invalid_pr_requirement_mask.sum()
)
print(
    "Invalid pr_citizenship_req values:",
    invalid_pr_requirement_count
)

Invalid pr_citizenship_req values: 0


## 11.16 Validate sec_clearance

In [ ]:
def valid_boolean_or_null(
    value
):

    return (
        pd.isna(value)
        or isinstance(
            value,
            (
                bool,
                np.bool_
            )
        )
    )
invalid_sec_clearance_mask = (
    ~jobboard_staging_df[
        "sec_clearance"
    ]
    .apply(
        valid_boolean_or_null
    )
)
invalid_sec_clearance_count = (
    invalid_sec_clearance_mask.sum()
)
print(
    "Invalid sec_clearance values:",
    invalid_sec_clearance_count
)

Invalid sec_clearance values: 0


## 11.17 AI-Enrichment Coverage

Measure the final population rate of the seven enrichment fields.

In [ ]:
FINAL_AI_FIELDS = [
    "industry",
    "function",
    "top_tech_skills",
    "exp_level",
    "pr_citizenship_req",
    "sec_clearance",
    "role_cat"
]
final_ai_coverage_df = pd.DataFrame({
    "field":
        FINAL_AI_FIELDS,

    "populated_count":
        [
            jobboard_staging_df[
                field
            ]
            .notna()
            .sum()
            for field
            in FINAL_AI_FIELDS
        ],

    "coverage_pct":
        [
            round(
                jobboard_staging_df[
                    field
                ]
                .notna()
                .mean()
                * 100,
                2
            )
            if final_record_count > 0
            else 0
            for field
            in FINAL_AI_FIELDS
        ]
})
display(
    final_ai_coverage_df
)

,field,populated_count,coverage_pct
0,industry,20,100.0
1,function,20,100.0
2,top_tech_skills,16,80.0
3,exp_level,1,5.0
4,pr_citizenship_req,16,80.0
5,sec_clearance,20,100.0
6,role_cat,16,80.0


## 11.18 Calculate Overall AI-Enrichment Coverage

In [ ]:
overall_ai_enrichment_coverage_pct = (
    final_ai_coverage_df[
        "coverage_pct"
    ]
    .mean()
)
print(
    f"Average AI enrichment coverage: "
    f"{overall_ai_enrichment_coverage_pct:.2f}%"
)

Average AI enrichment coverage: 77.86%


## 11.19 Validate Description HTML Removal

In [ ]:
final_html_mask = (
    jobboard_staging_df[
        "job_description"
    ]
    .fillna("")
    .astype(str)
    .str.contains(
        r"<[^>]+>",
        regex=True
    )
)
final_html_count = (
    final_html_mask.sum()
)
print(
    "Final descriptions containing HTML:",
    final_html_count
)

Final descriptions containing HTML: 0


## 11.20 Validate Text Whitespace

In [ ]:
FINAL_TEXT_FIELDS = [
    "job_title",
    "company_name",
    "city",
    "state",
    "location",
    "url"
]
final_whitespace_quality = []
for field in (
    FINAL_TEXT_FIELDS
):

    issue_count = (
        jobboard_staging_df[
            field
        ]
        .apply(
            has_whitespace_issue
        )
        .sum()
    )

    final_whitespace_quality.append({
        "field":
            field,

        "whitespace_issue_count":
            int(
                issue_count
            )
    })
final_whitespace_quality_df = pd.DataFrame(
    final_whitespace_quality
)
display(
    final_whitespace_quality_df
)

,field,whitespace_issue_count
0,job_title,0
1,company_name,0
2,city,0
3,state,0
4,location,0
5,url,0


## 11.21 Compare Baseline and Final Quality

Use the baseline metrics stored in Section 4 to show how the pipeline changed the data.

In [ ]:
before_after_quality_df = pd.DataFrame([
    {
        "Metric":
            "HTML descriptions",

        "Before":
            BASELINE_EDA_METRICS.get(
                "html_descriptions",
                np.nan
            ),

        "After":
            final_html_count
    },

    {
        "Metric":
            "Invalid dates",

        "Before":
            BASELINE_EDA_METRICS.get(
                "invalid_dates",
                np.nan
            ),

        "After":
            invalid_final_date_count
    },

    {
        "Metric":
            "Duplicate final IDs",

        "Before":
            BASELINE_EDA_METRICS.get(
                "duplicated_jobkey_group_rows",
                np.nan
            ),

        "After":
            final_duplicate_id_count
    }
])
display(
    before_after_quality_df
)

,Metric,Before,After
0,HTML descriptions,0,0
1,Invalid dates,0,0
2,Duplicate final IDs,0,0


## 11.22 Define Data-Quality Thresholds

Set explicit thresholds for staging readiness.

In [ ]:
DQ_THRESHOLDS = {
    "duplicate_id_rate_max_pct":
        0.0,

    "invalid_date_max_count":
        0,

    "invalid_location_type_max_count":
        0,

    "location_type_coverage_min_pct":
        95.0,

    "html_description_max_count":
        0,

    "invalid_exp_level_max_count":
        0,

    "invalid_role_cat_max_count":
        0,

    "invalid_pr_requirement_max_count":
        0,

    "invalid_sec_clearance_max_count":
        0
}

## 11.23 Run Individual Data-Quality Checks

In [ ]:
dq_checks = []

### Duplicate IDs

In [ ]:
dq_checks.append({
    "check":
        "Duplicate ID rate",

    "value":
        final_duplicate_rate_pct,

    "threshold":
        DQ_THRESHOLDS[
            "duplicate_id_rate_max_pct"
        ],

    "pass":
        final_duplicate_rate_pct
        <= DQ_THRESHOLDS[
            "duplicate_id_rate_max_pct"
        ]
})

### Invalid Dates

In [ ]:
dq_checks.append({
    "check":
        "Invalid posted dates",

    "value":
        invalid_final_date_count,

    "threshold":
        DQ_THRESHOLDS[
            "invalid_date_max_count"
        ],

    "pass":
        invalid_final_date_count
        <= DQ_THRESHOLDS[
            "invalid_date_max_count"
        ]
})

### Invalid Location Types

In [ ]:
dq_checks.append({
    "check":
        "Invalid location types",

    "value":
        invalid_location_type_count,

    "threshold":
        DQ_THRESHOLDS[
            "invalid_location_type_max_count"
        ],

    "pass":
        invalid_location_type_count
        <= DQ_THRESHOLDS[
            "invalid_location_type_max_count"
        ]
})

### Location Coverage

In [ ]:
dq_checks.append({
    "check":
        "Location type coverage %",

    "value":
        location_type_coverage_pct,

    "threshold":
        DQ_THRESHOLDS[
            "location_type_coverage_min_pct"
        ],

    "pass":
        location_type_coverage_pct
        >= DQ_THRESHOLDS[
            "location_type_coverage_min_pct"
        ]
})

### HTML

In [ ]:
dq_checks.append({
    "check":
        "HTML descriptions",

    "value":
        final_html_count,

    "threshold":
        DQ_THRESHOLDS[
            "html_description_max_count"
        ],

    "pass":
        final_html_count
        <= DQ_THRESHOLDS[
            "html_description_max_count"
        ]
})

### Experience Level

In [ ]:
dq_checks.append({
    "check":
        "Invalid experience levels",

    "value":
        invalid_exp_level_count,

    "threshold":
        DQ_THRESHOLDS[
            "invalid_exp_level_max_count"
        ],

    "pass":
        invalid_exp_level_count
        <= DQ_THRESHOLDS[
            "invalid_exp_level_max_count"
        ]
})

### Role Category

In [ ]:
dq_checks.append({
    "check":
        "Invalid role categories",

    "value":
        invalid_role_cat_count,

    "threshold":
        DQ_THRESHOLDS[
            "invalid_role_cat_max_count"
        ],

    "pass":
        invalid_role_cat_count
        <= DQ_THRESHOLDS[
            "invalid_role_cat_max_count"
        ]
})

Residency / Citizenship Requirement

In [ ]:
dq_checks.append({
    "check":
        "Invalid residency requirements",

    "value":
        invalid_pr_requirement_count,

    "threshold":
        DQ_THRESHOLDS[
            "invalid_pr_requirement_max_count"
        ],

    "pass":
        invalid_pr_requirement_count
        <= DQ_THRESHOLDS[
            "invalid_pr_requirement_max_count"
        ]
})

### Security Clearance

In [ ]:
dq_checks.append({
    "check":
        "Invalid security clearance values",

    "value":
        invalid_sec_clearance_count,

    "threshold":
        DQ_THRESHOLDS[
            "invalid_sec_clearance_max_count"
        ],

    "pass":
        invalid_sec_clearance_count
        <= DQ_THRESHOLDS[
            "invalid_sec_clearance_max_count"
        ]
})

## 11.24 Build Data-Quality Results Table

In [ ]:
dq_results_df = pd.DataFrame(
    dq_checks
)
dq_results_df[
    "status"
] = np.where(
    dq_results_df[
        "pass"
    ],
    "PASS",
    "FAIL"
)
display(
    dq_results_df
)

,check,value,threshold,pass,status
0,Duplicate ID rate,0.0,0.0,True,PASS
1,Invalid posted dates,0.0,0.0,True,PASS
2,Invalid location types,0.0,0.0,True,PASS
3,Location type coverage %,100.0,95.0,True,PASS
4,HTML descriptions,0.0,0.0,True,PASS
5,Invalid experience levels,0.0,0.0,True,PASS
6,Invalid role categories,0.0,0.0,True,PASS
7,Invalid residency requirements,0.0,0.0,True,PASS
8,Invalid security clearance values,0.0,0.0,True,PASS


## 11.25 Calculate Overall Data-Quality Status

In [ ]:
overall_dq_pass = (
    dq_results_df[
        "pass"
    ]
    .all()
)
print(
    "Overall DQ result:",
    (
        "PASS"
        if overall_dq_pass
        else "FAIL"
    )
)

Overall DQ result: PASS


## 11.26 Create RAG Status

Use:

- GREEN — all checks pass;
- AMBER — one or two checks fail;
- RED — more than two checks fail.

In [ ]:
failed_dq_checks = (
    (~dq_results_df[
        "pass"
    ])
    .sum()
)
if (
    failed_dq_checks
    == 0
):

    DQ_RAG_STATUS = (
        "GREEN"
    )

elif (
    failed_dq_checks
    <= 2
):

    DQ_RAG_STATUS = (
        "AMBER"
    )

else:

    DQ_RAG_STATUS = (
        "RED"
    )
print(
    "DQ RAG status:",
    DQ_RAG_STATUS
)

DQ RAG status: GREEN


## 11.27 Create Final Pipeline Quality Summary

In [ ]:
FINAL_DQ_SUMMARY = {
    "records_ready_for_staging":
        final_record_count,

    "column_count":
        final_column_count,

    "duplicate_id_rate_pct":
        final_duplicate_rate_pct,

    "invalid_dates":
        int(
            invalid_final_date_count
        ),

    "location_type_coverage_pct":
        float(
            location_type_coverage_pct
        ),

    "invalid_location_types":
        int(
            invalid_location_type_count
        ),

    "ai_enrichment_coverage_pct":
        float(
            overall_ai_enrichment_coverage_pct
        ),

    "html_descriptions":
        int(
            final_html_count
        ),

    "failed_dq_checks":
        int(
            failed_dq_checks
        ),

    "overall_pass":
        bool(
            overall_dq_pass
        ),

    "rag_status":
        DQ_RAG_STATUS
}
display(
    pd.DataFrame(
        FINAL_DQ_SUMMARY.items(),
        columns=[
            "Metric",
            "Value"
        ]
    )
)

,Metric,Value
0,records_ready_for_staging,20
1,column_count,20
2,duplicate_id_rate_pct,0.0
3,invalid_dates,0
4,location_type_coverage_pct,100.0
5,invalid_location_types,0
6,ai_enrichment_coverage_pct,77.857143
7,html_descriptions,0
8,failed_dq_checks,0
9,overall_pass,True


## 11.28 Identify Failed Checks for Review

In [ ]:
failed_checks_df = (
    dq_results_df.loc[
        ~dq_results_df[
            "pass"
        ]
    ]
    .copy()
)
if len(
    failed_checks_df
) > 0:

    display(
        failed_checks_df
    )

else:

    print(
        "✓ No data-quality checks failed."
    )

✓ No data-quality checks failed.


## 11.29 Staging Readiness Decision

In [ ]:
STAGING_READY = (
    overall_dq_pass
)
if STAGING_READY:

    print(
        "✓ Dataset passed final validation "
        "and is ready for staging."
    )

else:

    print(
        "⚠ Dataset failed one or more "
        "quality checks and requires review "
        "before staging."
    )

✓ Dataset passed final validation and is ready for staging.


## 11.30 Preserve Final Validated Dataset

Do not automatically discard records that fail individual enrichment checks.

Preserve the complete staging dataframe for review.

In [ ]:
validated_staging_df = (
    jobboard_staging_df
    .copy(deep=True)
)
print(
    "Validated staging records:",
    len(
        validated_staging_df
    )
)

Validated staging records: 20


## 11.31 Section 11 Outcome

Section 11 performs the final data-quality assessment of the output generated by the production pipeline.

The section verifies that the resulting staging dataset:

- contains exactly the required 20 target fields;
- contains no unexpected schema fields;
has acceptable null rates;
- contains no repeated final id values;
- contains valid posting dates;
- contains valid pipeline timestamps;
- restricts location_type to Metro, Regional, or Remote;
- measures final location-type coverage;
- validates the allowed exp_level values;
- validates the allowed role_cat values;
- validates residency and citizenship outputs;
- verifies sec_clearance as a boolean value;
- measures coverage across all AI-enrichment fields;
- confirms that HTML has been removed from job descriptions;
- checks for remaining whitespace issues;
- and compares selected final-quality measures against the raw EDA baseline.

The final quality controls are summarised through:

`dq_results_df`

and the overall staging condition is represented by:

`DQ_RAG_STATUS`

using:

`GREEN`
All configured quality checks passed

`AMBER`
One or two checks failed and require review

`RED`
More than two checks failed

The final staging-readiness flag is stored as:

`STAGING_READY`

The validated dataset remains available as:

`validated_staging_df`

This dataset should only be written to jobboard_staging once the required quality checks and review process have been satisfied.

The final section of the notebook can now summarise the project results, selected methodologies, measured improvements, limitations, and recommendations for production deployment.

#12. Final Results, Conclusions and Recommendations

This section consolidates the outputs of the complete pipeline and provides the final project assessment.

The notebook has progressed through four distinct stages:

Source Understanding
        
        ↓
Cleaning and Standardisation
        
        ↓
Method Experimentation and Selection
        
        ↓
Production Pipeline and Data-Quality Validation

The final assessment focuses on:

- selected production methodologies;
- experimental findings;
- pipeline record movement;
- final data quality;
- enrichment coverage;
- staging readiness;
- known limitations;
- and recommendations for production deployment.

No transformation or enrichment is performed in this section.

## 12.1 Final Methodology Summary

Create a concise record of the methodologies selected for the final production pipeline.

In [ ]:
FINAL_METHOD_SELECTION = {
    "Location Type":
        PRODUCTION_CONFIG[
            "location_case"
        ],

    "Deduplication":
        PRODUCTION_CONFIG[
            "dedup_case"
        ],

    "AI Enrichment":
        PRODUCTION_CONFIG[
            "ai_case"
        ]
}
final_methodology_df = pd.DataFrame(
    FINAL_METHOD_SELECTION.items(),
    columns=[
        "Pipeline Component",
        "Selected Method"
    ]
)
display(
    final_methodology_df
)

,Pipeline Component,Selected Method
0,Location Type,Case 2
1,Deduplication,Case 1
2,AI Enrichment,Case 1


##12.2 Document Selected Method Descriptions

Map the selected case identifiers to meaningful methodology descriptions.

In [ ]:
LOCATION_METHOD_DESCRIPTIONS = {
    "Case 1":
        "AI-only location classification",

    "Case 2":
        "GCCSA spatial classification "
        "with AI fallback"
}
DEDUP_METHOD_DESCRIPTIONS = {
    "Case 1":
        "Company + job title + location",

    "Case 2":
        "Company + job title + location "
        "+ posting-date similarity",

    "Case 3":
        "Company + job title + location "
        "+ generated description hash"
}
AI_METHOD_DESCRIPTIONS = {
    "Case 1":
        "Combined notebook-defined prompt",

    "Case 2":
        "Separate notebook-defined prompts",

    "Case 3":
        "Combined JSON-defined prompt",

    "Case 4":
        "Separate JSON-defined prompts"
}

## 12.3 Build Final Production Method Table

In [ ]:
selected_method_details_df = pd.DataFrame([
    {
        "Pipeline Component":
            "Location Type",

        "Selected Case":
            PRODUCTION_CONFIG[
                "location_case"
            ],

        "Method":
            LOCATION_METHOD_DESCRIPTIONS.get(
                PRODUCTION_CONFIG[
                    "location_case"
                ],
                "Not selected"
            )
    },

    {
        "Pipeline Component":
            "Deduplication",

        "Selected Case":
            PRODUCTION_CONFIG[
                "dedup_case"
            ],

        "Method":
            DEDUP_METHOD_DESCRIPTIONS.get(
                PRODUCTION_CONFIG[
                    "dedup_case"
                ],
                "Not selected"
            )
    },

    {
        "Pipeline Component":
            "AI Enrichment",

        "Selected Case":
            PRODUCTION_CONFIG[
                "ai_case"
            ],

        "Method":
            AI_METHOD_DESCRIPTIONS.get(
                PRODUCTION_CONFIG[
                    "ai_case"
                ],
                "Not selected"
            )
    }
])
display(
    selected_method_details_df
)

,Pipeline Component,Selected Case,Method
0,Location Type,Case 2,GCCSA spatial classification with AI fallback
1,Deduplication,Case 1,Company + job title + location
2,AI Enrichment,Case 1,Combined notebook-defined prompt


## 12.4 Summarise Production Record Movement

The following table shows how records moved through the production pipeline.

In [ ]:
final_record_flow_df = pd.DataFrame([
    {
        "Pipeline Stage":
            "Incoming source",

        "Record Count":
            production_rows_before_jobkey
    },

    {
        "Pipeline Stage":
            "After source jobKey deduplication",

        "Record Count":
            len(
                production_df
            )
    },

    {
        "Pipeline Stage":
            "After cross-dataset deduplication",

        "Record Count":
            len(
                production_new_df
            )
    },

    {
        "Pipeline Stage":
            "Final staging output",

        "Record Count":
            len(
                validated_staging_df
            )
    }
])
display(
    final_record_flow_df
)

,Pipeline Stage,Record Count
0,Incoming source,20
1,After source jobKey deduplication,20
2,After cross-dataset deduplication,20
3,Final staging output,20


## 12.5 Calculate Records Removed by Deduplication

In [ ]:
source_jobkey_duplicates_removed = (
    production_rows_before_jobkey
    - len(
        production_df
    )
)
cross_dataset_duplicates_removed = (
    production_duplicate_count
)
total_duplicates_removed = (
    source_jobkey_duplicates_removed
    + cross_dataset_duplicates_removed
)
print(
    "Source jobKey duplicates removed:",
    source_jobkey_duplicates_removed
)

print(
    "Cross-dataset duplicates removed:",
    cross_dataset_duplicates_removed
)

print(
    "Total duplicate records removed:",
    total_duplicates_removed
)

Source jobKey duplicates removed: 0
Cross-dataset duplicates removed: 0
Total duplicate records removed: 0


##12.6 Final Data-Quality Summary

Use the outputs generated in Section 11.

In [ ]:
final_quality_summary_df = pd.DataFrame([
    {
        "Metric":
            "Records ready for staging",

        "Value":
            final_record_count
    },

    {
        "Metric":
            "Final column count",

        "Value":
            final_column_count
    },

    {
        "Metric":
            "Duplicate ID rate (%)",

        "Value":
            round(
                final_duplicate_rate_pct,
                2
            )
    },

    {
        "Metric":
            "Invalid posting dates",

        "Value":
            int(
                invalid_final_date_count
            )
    },

    {
        "Metric":
            "Location type coverage (%)",

        "Value":
            round(
                location_type_coverage_pct,
                2
            )
    },

    {
        "Metric":
            "Invalid location types",

        "Value":
            int(
                invalid_location_type_count
            )
    },

    {
        "Metric":
            "AI enrichment coverage (%)",

        "Value":
            round(
                overall_ai_enrichment_coverage_pct,
                2
            )
    },

    {
        "Metric":
            "Descriptions containing HTML",

        "Value":
            int(
                final_html_count
            )
    },

    {
        "Metric":
            "Failed DQ checks",

        "Value":
            int(
                failed_dq_checks
            )
    },

    {
        "Metric":
            "RAG status",

        "Value":
            DQ_RAG_STATUS
    }
])
display(
    final_quality_summary_df
)

,Metric,Value
0,Records ready for staging,20
1,Final column count,20
2,Duplicate ID rate (%),0.0
3,Invalid posting dates,0
4,Location type coverage (%),100.0
5,Invalid location types,0
6,AI enrichment coverage (%),77.86
7,Descriptions containing HTML,0
8,Failed DQ checks,0
9,RAG status,GREEN


## 12.7 Final AI-Enrichment Coverage by Field

Coverage should be reviewed individually because overall average coverage can hide weak performance in individual enrichment fields.

In [ ]:
final_ai_summary_df = (
    final_ai_coverage_df
    .sort_values(
        by="coverage_pct",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)
display(
    final_ai_summary_df
)

,field,populated_count,coverage_pct
0,industry,20,100.0
1,function,20,100.0
2,sec_clearance,20,100.0
3,top_tech_skills,16,80.0
4,pr_citizenship_req,16,80.0
5,role_cat,16,80.0
6,exp_level,1,5.0


## 12.8 Identify Lowest-Coverage AI Fields

In [ ]:
lowest_ai_coverage = (
    final_ai_summary_df
    .sort_values(
        by="coverage_pct",
        ascending=True
    )
    .head(3)
)
display(
    lowest_ai_coverage
)

,field,populated_count,coverage_pct
6,exp_level,1,5.0
3,top_tech_skills,16,80.0
4,pr_citizenship_req,16,80.0


## 12.9 Final Location Distribution

In [ ]:
final_location_summary_df = (
    jobboard_staging_df[
        "location_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "location_type"
    )
    .reset_index(
        name="record_count"
    )
)
if (
    final_record_count > 0
):

    final_location_summary_df[
        "percentage"
    ] = (
        final_location_summary_df[
            "record_count"
        ]
        / final_record_count
        * 100
    ).round(2)

else:

    final_location_summary_df[
        "percentage"
    ] = 0
display(
    final_location_summary_df
)

,location_type,record_count,percentage
0,Metro,20,100.0


## 12.10 Final Null-Rate Ranking

Identify the target fields with the highest remaining null rates.

In [ ]:
final_null_ranking_df = (
    final_null_analysis_df
    .sort_values(
        by="null_pct",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)
display(
    final_null_ranking_df
)

,field,null_count,null_pct
0,exp_level,19,95.0
1,role_cat,4,20.0
2,pr_citizenship_req,4,20.0
3,top_tech_skills,4,20.0
4,posted_date,0,0.0
5,api_name,0,0.0
6,api_ini_dt_time,0,0.0
7,id,0,0.0
8,job_title,0,0.0
9,company_name,0,0.0


## 12.11 Final Pipeline Status

In [ ]:
if STAGING_READY:

    FINAL_PIPELINE_STATUS = (
        "READY FOR STAGING"
    )

elif (
    DQ_RAG_STATUS
    == "AMBER"
):

    FINAL_PIPELINE_STATUS = (
        "REVIEW REQUIRED"
    )

else:

    FINAL_PIPELINE_STATUS = (
        "NOT READY FOR STAGING"
    )
print(
    "Final pipeline status:",
    FINAL_PIPELINE_STATUS
)

Final pipeline status: READY FOR STAGING


## 12.12 Final Project Result

In [ ]:
FINAL_PROJECT_RESULT = {
    "location_method":
        LOCATION_METHOD_DESCRIPTIONS.get(
            PRODUCTION_CONFIG[
                "location_case"
            ]
        ),

    "deduplication_method":
        DEDUP_METHOD_DESCRIPTIONS.get(
            PRODUCTION_CONFIG[
                "dedup_case"
            ]
        ),

    "ai_enrichment_method":
        AI_METHOD_DESCRIPTIONS.get(
            PRODUCTION_CONFIG[
                "ai_case"
            ]
        ),

    "source_records":
        int(
            production_rows_before_jobkey
        ),

    "duplicates_removed":
        int(
            total_duplicates_removed
        ),

    "final_records":
        int(
            final_record_count
        ),

    "location_coverage_pct":
        round(
            float(
                location_type_coverage_pct
            ),
            2
        ),

    "ai_enrichment_coverage_pct":
        round(
            float(
                overall_ai_enrichment_coverage_pct
            ),
            2
        ),

    "dq_rag_status":
        DQ_RAG_STATUS,

    "pipeline_status":
        FINAL_PIPELINE_STATUS
}
display(
    pd.DataFrame(
        FINAL_PROJECT_RESULT.items(),
        columns=[
            "Metric",
            "Result"
        ]
    )
)

,Metric,Result
0,location_method,GCCSA spatial classification with AI fallback
1,deduplication_method,Company + job title + location
2,ai_enrichment_method,Combined notebook-defined prompt
3,source_records,20
4,duplicates_removed,0
5,final_records,20
6,location_coverage_pct,100.0
7,ai_enrichment_coverage_pct,77.86
8,dq_rag_status,GREEN
9,pipeline_status,READY FOR STAGING


## 12.13 Experimental Findings

The experimentation stage was necessary to avoid selecting transformation methods based only on implementation convenience.

The tests in Sections 6–9 compared alternative approaches using:

- manually validated output quality;
- execution time;
- API-call volume;
- token consumption;
- resource use;
- output coverage;
- operational reliability;
- and maintainability.

### 12.13.1 Location-Type Experiment

The location experiment compared:

**Case 1**
AI-only classification

**Case 2**
GCCSA spatial classification
+
AI fallback

The hybrid approach provides an important architectural advantage where deterministic geographic evidence is available.

Instead of requesting an AI decision for every record, the spatial method can classify records that resolve successfully through the GCCSA geography and reserve AI calls for unresolved records.

This reduces unnecessary dependence on generative AI while maintaining an AI fallback for records that cannot be resolved deterministically.

The final production implementation therefore uses only the location methodology selected in Section 9.

### 12.13.2 Deduplication Experiment

The deduplication experiment evaluated:

**Case 1**

Company
+
Job title
+
Location

**Case 2**

Company
+
Job title
+
Location
+
Posting-date similarity

**Case 3**

Company
+
Job title
+
Location
+
Generated description hash

The production pipeline also performs an initial jobKey duplicate removal within the incoming source before cross-dataset comparison.

The cross-dataset test is then performed against records retrieved from the previous 48-hour window of vw_dca_jobboard.

Where experimental validation does not contain sufficient positive duplicate examples, conclusions regarding recall and F1 should be treated cautiously.

A method should not be considered proven at identifying duplicates simply because it achieves high accuracy on a validation set dominated by non-duplicate records.

### 12.13.3 AI-Enrichment Experiment

The AI-enrichment experiment compared four prompting architectures:

**Case 1**

Combined prompt
defined directly in the notebook

**Case 2**

Separate prompts
defined directly in the notebook

**Case 3**

Combined prompt
constructed from JSON prompt definitions

**Case 4**

Separate prompts
loaded from JSON definitions

The experiment evaluates an important production trade-off:

Output quality
        vs
API calls
        vs
Token consumption
        vs
Execution time
        vs
Maintainability

The final production pipeline does not execute all four cases.

Only the selected enrichment architecture is used.

The experimental implementations remain in the notebook solely to demonstrate why that methodology was selected.

## 12.14 Key Pipeline Strengths

The final design has several strengths.

Deterministic processing where possible

Cleaning, standardisation, date parsing, hashing, source-level deduplication and spatial location classification are handled deterministically where appropriate.

This improves reproducibility and reduces unnecessary AI dependence.

AI used for semantic enrichment

Gemini is reserved for tasks requiring interpretation of job content, including:

- industry;
- business function;
- technical skills;
- experience level;
- residency or citizenship requirements;
- security-clearance requirements;
- and role category.

**Independent production workflow**

The production pipeline independently retrieves its source and target data.

It does not depend on temporary EDA or experimental dataframes for production execution.

**48-hour cross-dataset comparison**

Incoming Indeed jobs are compared against the recent target dataset instead of being deduplicated only within the current batch.

Staging before production

Transformed records are prepared for:

jobboard_staging

rather than being written directly into:

vw_dca_jobboard

This allows quality review before records reach the production dataset.

**Explicit quality controls**

The final pipeline validates:

- schema;
- record count;
- duplicates;
- dates;
- categorical outputs;
- location coverage;
- AI coverage;
- HTML cleaning;
- and text quality.

## 12.15 Known Limitations

The pipeline also has limitations that should be documented rather than hidden.

Location Classification

GCCSA geography primarily distinguishes Greater Capital City Statistical Areas from the rest of a state or territory.

It does not by itself provide a complete geographic definition of remoteness.

The AI fallback therefore remains important for unresolved locations, and the classification methodology should continue to be validated as location diversity increases.

**Deduplication Validation**

Deduplication performance depends heavily on the composition of the manually labelled benchmark.

If the benchmark contains few or no actual duplicates, metrics such as recall and F1 cannot meaningfully demonstrate duplicate-detection capability.

Future validation should deliberately contain both:

Known duplicate records
+
Known non-duplicate records
AI-Enrichment Quality

AI enrichment is probabilistic.

A successful API response and valid output format do not guarantee that the semantic classification is correct.

AI quality should therefore continue to be monitored using manually reviewed samples.

Multi-Value AI Fields

Fields such as:

- top_tech_skills
- role_cat
- function
- industry
- pr_citizenship_req

may contain more than one valid value.

Exact-match accuracy can penalise partially correct predictions.

AI-enrichment cost and execution time scale with:

- number of incoming records;
- prompt length;
- output length;
- number of prompts per record;
- and model pricing.

Production monitoring should therefore track token consumption and API calls rather than assuming the experimental cost remains constant.

## 12.16 Production Recommendations

The following controls are recommended before fully automating the pipeline.

1. Retain staging review

Continue writing transformed data to jobboard_staging before promoting records into the final job-board dataset.

2. Expand manual validation datasets

The current manually reviewed samples should be expanded over time, particularly for:

- duplicate identification;
- remote/regional classification;
- ambiguous job roles;
- and AI-enrichment fields.

3. Monitor AI quality by field

Do not rely only on overall enrichment accuracy.

Track quality separately for:

- industry
- function
- top_tech_skills
- exp_level
- pr_citizenship_req
- sec_clearance
- role_cat

4. Track pipeline metrics for every run

At minimum, retain:

- source record count;
- records removed by jobKey;
- cross-dataset duplicates removed;
- records sent to AI;
- API calls;
- token consumption;
- processing duration;
- AI coverage;
- location coverage;
- final staging record count;
- DQ failures;
- and final RAG status.

5. Review thresholds periodically

The DQ thresholds defined in Section 11 should initially act as explicit controls but should be reviewed after sufficient production history becomes available.

6. Investigate failed records rather than silently dropping them

Records with unresolved enrichment or quality issues should remain available for review unless there is an explicit business rule requiring removal.


12.17 Recommended Production Monitoring Metrics

In [ ]:
PRODUCTION_MONITORING_METRICS = {
    "pipeline_timestamp":
        PIPELINE_RUN_TIMESTAMP,

    "source_records":
        int(
            production_rows_before_jobkey
        ),

    "jobkey_duplicates_removed":
        int(
            source_jobkey_duplicates_removed
        ),

    "cross_dataset_duplicates_removed":
        int(
            cross_dataset_duplicates_removed
        ),

    "final_staging_records":
        int(
            final_record_count
        ),

    "duplicate_rate_pct":
        round(
            float(
                final_duplicate_rate_pct
            ),
            2
        ),

    "location_coverage_pct":
        round(
            float(
                location_type_coverage_pct
            ),
            2
        ),

    "ai_enrichment_coverage_pct":
        round(
            float(
                overall_ai_enrichment_coverage_pct
            ),
            2
        ),

    "failed_dq_checks":
        int(
            failed_dq_checks
        ),

    "rag_status":
        DQ_RAG_STATUS,

    "staging_ready":
        bool(
            STAGING_READY
        )
}
production_monitoring_df = pd.DataFrame(
    PRODUCTION_MONITORING_METRICS.items(),
    columns=[
        "Metric",
        "Value"
    ]
)
display(
    production_monitoring_df
)

,Metric,Value
0,pipeline_timestamp,2026-08-31 05:22:38.366073+00:00
1,source_records,20
2,jobkey_duplicates_removed,0
3,cross_dataset_duplicates_removed,0
4,final_staging_records,20
5,duplicate_rate_pct,0.0
6,location_coverage_pct,100.0
7,ai_enrichment_coverage_pct,77.86
8,failed_dq_checks,0
9,rag_status,GREEN


## 12.18 Final Recommendation

In [ ]:
if STAGING_READY:

    final_recommendation = (
        "The current pipeline output satisfies "
        "the configured data-quality controls "
        "and can proceed to the staging layer."
    )

elif (
    DQ_RAG_STATUS
    == "AMBER"
):

    final_recommendation = (
        "The pipeline completed, but one or "
        "more quality controls require manual "
        "review before the data proceeds."
    )

else:

    final_recommendation = (
        "The current pipeline output should "
        "not proceed until the failed quality "
        "controls have been investigated."
    )
print(
    final_recommendation
)

The current pipeline output satisfies the configured data-quality controls and can proceed to the staging layer.


## 12.19 Final Conclusion

The completed notebook provides an end-to-end framework for transforming Indeed job data from:

`daily_scrape_dev`

into the agreed 20-field job-board schema.

The workflow combines:

BigQuery extraction
        
        ↓
Data cleaning
        
        ↓
Location standardisation
        
        ↓
Source-level deduplication
        
        ↓
48-hour cross-dataset deduplication
        
        ↓
AI enrichment
        
        ↓
Target mapping
       
        ↓
Data-quality validation
        
        ↓
Staging readiness

The experimental components of the notebook provide evidence for methodology selection, while the final production pipeline contains only the methodology chosen for operational use.

This separation is important because experimentation and production serve different purposes:


**Experiments**
Determine which method should be used.

**Production pipeline**
Executes the selected method consistently.

The final output is therefore not automatically treated as production-ready simply because the pipeline executes successfully.

Instead, readiness is determined through explicit quality checks and represented by:

`STAGING_READY`

`DQ_RAG_STATUS`

`FINAL_PIPELINE_STATUS`

This provides a reproducible and auditable basis for deciding whether each processed batch should proceed to jobboard_staging or be reviewed before further loading.

#13. Pipeline Results Dashboard

This dashboard provides a compact visual summary of the complete Indeed data pipeline.

It combines:

production record flow;
selected methodologies;
deduplication results;
location-type results;
AI-enrichment coverage;
final data-quality status;
and a sample of the staging output.

The dashboard uses only results already produced by the notebook and does not rerun the pipeline.

##13.1 Install Dashboard Dependencies

In [ ]:
!pip install -q plotly

##13.2 Import Dashboard Libraries

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display, HTML

##13.3 Dashboard Title

In [ ]:
display(
    HTML(
        """
        <div style="
            padding:18px;
            border-radius:10px;
            background:#f5f7fa;
            margin-bottom:20px;
        ">
            <h2 style="margin:0;">
                Indeed Pipeline Results Dashboard
            </h2>

            <p style="
                margin-top:8px;
                margin-bottom:0;
                color:#555;
            ">
                End-to-end overview of cleaning,
                deduplication, location classification,
                AI enrichment and staging readiness.
            </p>
        </div>
        """
    )
)

##13.4 KPI Cards

Create high-level metrics from the final pipeline.

In [ ]:
dashboard_kpis = {
    "Source Records":
        int(
            production_rows_before_jobkey
        ),

    "Final Staging Records":
        int(
            final_record_count
        ),

    "Duplicates Removed":
        int(
            total_duplicates_removed
        ),

    "Location Coverage":
        f"{location_type_coverage_pct:.1f}%",

    "AI Coverage":
        f"{overall_ai_enrichment_coverage_pct:.1f}%",

    "Failed DQ Checks":
        int(
            failed_dq_checks
        ),

    "RAG Status":
        DQ_RAG_STATUS
}
kpi_html = """
<div style="
    display:grid;
    grid-template-columns:repeat(4, 1fr);
    gap:12px;
    margin-bottom:20px;
">
"""

for metric, value in dashboard_kpis.items():

    kpi_html += f"""
    <div style="
        border:1px solid #ddd;
        border-radius:8px;
        padding:14px;
        background:white;
    ">
        <div style="
            font-size:13px;
            color:#666;
        ">
            {metric}
        </div>

        <div style="
            font-size:24px;
            font-weight:bold;
            margin-top:5px;
        ">
            {value}
        </div>
    </div>
    """

kpi_html += "</div>"
display(
    HTML(
        kpi_html
    )
)

##13.5 Pipeline Flow

Create a simple visual showing the complete production process.

In [ ]:
pipeline_stages = [
    "Raw Source",
    "Cleaning",
    "jobKey Dedup",
    "Location Type",
    "48h Cross-Dataset Dedup",
    "AI Enrichment",
    "20-Field Mapping",
    "DQ Validation",
    "Staging"
]
pipeline_flow_html = """
<div style="
    display:flex;
    flex-wrap:wrap;
    align-items:center;
    gap:8px;
    margin-bottom:25px;
">
"""

for index, stage in enumerate(
    pipeline_stages
):

    pipeline_flow_html += f"""
    <div style="
        padding:10px 14px;
        border:1px solid #bbb;
        border-radius:7px;
        background:white;
        font-weight:600;
        font-size:13px;
    ">
        {stage}
    </div>
    """

    if index < len(
        pipeline_stages
    ) - 1:

        pipeline_flow_html += """
        <div style="
            font-size:20px;
            color:#777;
        ">
            →
        </div>
        """

pipeline_flow_html += "</div>"
display(
    HTML(
        pipeline_flow_html
    )
)

##13.6 Selected Production Methods

Use the final selected methodologies from Section 12.

In [ ]:
display( selected_method_details_df )

,Pipeline Component,Selected Case,Method
0,Location Type,Case 2,GCCSA spatial classification with AI fallback
1,Deduplication,Case 1,Company + job title + location
2,AI Enrichment,Case 1,Combined notebook-defined prompt


##13.7 Production Record Flow Chart

Show how records move through the production pipeline.

In [ ]:
fig = px.bar(
    final_record_flow_df,
    x="Pipeline Stage",
    y="Record Count",
    text="Record Count",
    title="Production Record Flow"
)
fig.update_layout(
    xaxis_title="",
    yaxis_title="Records"
)
fig.show()

##13.8 Deduplication Summary

Create a compact summary showing both deduplication stages.

In [ ]:
dedup_dashboard_df = pd.DataFrame({
    "Stage": [
        "Source jobKey deduplication",
        "48h cross-dataset deduplication"
    ],

    "Duplicates Removed": [
        source_jobkey_duplicates_removed,
        cross_dataset_duplicates_removed
    ]
})
display(
    dedup_dashboard_df
)

,Stage,Duplicates Removed
0,Source jobKey deduplication,0
1,48h cross-dataset deduplication,0


##13.9 Deduplication Experiment Comparison

Show the three tested cross-dataset approaches.

Use your existing deduplication comparison dataframe.

In [ ]:
display( dedup_comparison_df )

,clean_jobKey,clean_company,clean_title,clean_location,clean_posted_date,case1_duplicate,case2_duplicate,case3_duplicate,all_cases_agree
0,9c2916ac1bdaf7e2,Department of Education,Data Analyst,Melbourne City Centre VIC,2026-08-21 00:00:00+00:00,False,False,False,True
1,1651d59366f4f3a1,CSIRO,Manager - Governance of Indigenous Data and In...,Brisbane QLD,2026-08-21 00:00:00+00:00,False,False,False,True
2,d140df231910c633,Amazon.com,"Technical Program Manager, Data Center Delivery",Sydney NSW,2026-08-21 00:00:00+00:00,False,False,False,True
3,28e7a32d55dffa4f,Wesfarmers,Data Scientist - Hybrid,Melbourne VIC,2026-08-21 00:00:00+00:00,False,False,False,True
4,10f4851d92195dbd,ANZ,Data Analyst,Melbourne VIC,2026-08-20 00:00:00+00:00,False,False,False,True
5,4a76befd023692bf,CBRE,Data & AI Governance Lead,Sydney NSW,2026-08-20 00:00:00+00:00,False,False,False,True
6,d021ae0217a5e5d5,oOh!,CRM data and analytics Manager,North Sydney NSW,2026-08-20 00:00:00+00:00,False,False,False,True
7,750cb75c113c1030,NSW Government,Business Analyst,Sydney NSW,2026-08-20 00:00:00+00:00,False,False,False,True
8,8c4ce1fb43cadf86,Colliers,Database & Sales Support,Parramatta NSW,2026-08-19 00:00:00+00:00,False,False,False,True
9,67860caf9d737fa9,Commonwealth Bank of Australia,Senior Manager Data Lead and Process Improvement,Sydney NSW,2026-08-17 00:00:00+00:00,False,False,False,True


##13.10 Location Method Comparison

In [ ]:
display( location_ranking_df )

,Case,Approach,Quality Score,Coverage Score,Execution Efficiency Score,Resource Efficiency Score,Reliability Score,Maintainability Score,Weighted Score
1,Case 2,GCCSA + AI Fallback,100.0,100.0,100.0,100.0,100.0,80,98.0
0,Case 1,AI Only,100.0,100.0,0.0,0.0,100.0,90,74.0


##13.11 AI Experiment Comparison

In [ ]:
display( ai_ranking_df )

,Case,Approach,Quality Score,Coverage Score,Execution Efficiency Score,Resource Efficiency Score,Operational Reliability Score,Maintainability Score,Weighted Score
0,Case 1,Combined - Notebook,50.714286,78.571429,98.229074,100.000000,100.0,70,73.805790
2,Case 3,Combined - JSON,54.285714,78.571429,100.000000,0.000000,100.0,95,68.000000
3,Case 4,Separate - JSON,50.714286,78.571429,4.202426,5.481216,100.0,90,52.249914
1,Case 2,Separate - Notebook,49.285714,83.571429,0.000000,5.917138,100.0,75,50.341714


##13.12 AI Coverage by Field

In [ ]:
fig = px.bar(
    final_ai_summary_df,
    x="field",
    y="coverage_pct",
    text="coverage_pct",
    title="AI Enrichment Coverage by Field"
)
fig.update_layout(
    xaxis_title="AI-Enriched Field",
    yaxis_title="Coverage (%)",
    yaxis_range=[
        0,
        100
    ]
)
fig.show()

##13.13 Location-Type Distribution

In [ ]:
fig = px.pie(
    final_location_summary_df,
    names="location_type",
    values="record_count",
    title="Final Location-Type Distribution"
)
fig.show()

##13.14 Data-Quality Results

In [ ]:
display( dq_results_df[ [ "check", "value", "threshold", "status" ] ] )

,check,value,threshold,status
0,Duplicate ID rate,0.0,0.0,PASS
1,Invalid posted dates,0.0,0.0,PASS
2,Invalid location types,0.0,0.0,PASS
3,Location type coverage %,100.0,95.0,PASS
4,HTML descriptions,0.0,0.0,PASS
5,Invalid experience levels,0.0,0.0,PASS
6,Invalid role categories,0.0,0.0,PASS
7,Invalid residency requirements,0.0,0.0,PASS
8,Invalid security clearance values,0.0,0.0,PASS


##13.15 DQ Status Summary

In [ ]:
status_colour = (
    "#d4edda"
    if DQ_RAG_STATUS == "GREEN"
    else (
        "#fff3cd"
        if DQ_RAG_STATUS == "AMBER"
        else "#f8d7da"
    )
)
display(
    HTML(
        f"""
        <div style="
            padding:18px;
            border-radius:10px;
            background:{status_colour};
            margin-top:15px;
            margin-bottom:20px;
        ">

            <div style="
                font-size:14px;
            ">
                Final Data Quality Status
            </div>

            <div style="
                font-size:30px;
                font-weight:bold;
            ">
                {DQ_RAG_STATUS}
            </div>

            <div style="
                font-size:14px;
                margin-top:5px;
            ">
                Failed checks:
                {failed_dq_checks}
            </div>

        </div>
        """
    )
)

##13.16 Final Staging Table

Show a small sample of the actual final output.

In [ ]:
display( jobboard_staging_df.head(3) )

,id,api_ini_dt_time,api_name,posted_date,job_title,company_name,city,state,location,job_description,url,source,industry,function,top_tech_skills,exp_level,pr_citizenship_req,sec_clearance,location_type,role_cat
0,9c2916ac1bdaf7e2,2026-08-31 05:22:38.366073+00:00,Indeed Jobs,2026-08-21 00:00:00+00:00,Data Analyst,Department of Education,Melbourne City Centre,VIC,Melbourne City Centre VIC,This position commences 1 January 2027 Are you...,https://au.indeed.com/viewjob?jk=9c2916ac1bdaf7e2,Indeed,Education,"Education, Professional Development, Performan...","Excel, Power BI",None,None mentioned,False,Metro,Data Analyst
1,1651d59366f4f3a1,2026-08-31 05:22:38.366073+00:00,Indeed Jobs,2026-08-21 00:00:00+00:00,Manager - Governance of Indigenous Data and In...,CSIRO,Brisbane,QLD,Brisbane QLD,Acknowledgement of Country CSIRO acknowledges ...,https://au.indeed.com/viewjob?jk=1651d59366f4f3a1,Indeed,Scientific Research,"Governance, Policy Development, Indigenous Rel...",None,None,Citizenship,True,Metro,None
2,d140df231910c633,2026-08-31 05:22:38.366073+00:00,Indeed Jobs,2026-08-21 00:00:00+00:00,"Technical Program Manager, Data Center Delivery",Amazon.com,Sydney,NSW,Sydney NSW,DESCRIPTION AWS Infrastructure Services owns t...,https://au.indeed.com/viewjob?jk=d140df231910c633,Indeed,Cloud Computing,Infrastructure Management,AWS,None,None mentioned,False,Metro,None


##13.17 AI-Enriched Listing

In [ ]:
display(production_enriched_df.loc[11:13])

,companyIndustry,companyCeo,companyLinks,companyFounded,companyDescription,companyNumEmployees,companyAddresses,jobUrl,language,locale,...,job_description,job_title,source_industry,top_tech_skills,role_cat,function,industry,exp_level,pr_citizenship_req,sec_clearance
11,BANKS_AND_FINANCIAL_SERVICES,"{'startDate': None, 'photoUrl': None, 'name': ...",{'corporateWebsite': 'http://www.finxl.com.au'...,{'year': 2003},"FinXL started in 2003, with a passion for usin...",201 to 500,[Sydney],https://au.indeed.com/viewjob?jk=2de17585f6301445,en,au,...,Job details Posted14 August 2026 SalaryNegotia...,Data Migration Lead,None,APIs,Data Engineer,"Customer Transformation, Data Migration","Telecommunication, Banking and Finance, Commer...",None,None,False
12,None,"{'startDate': None, 'photoUrl': 'https://d2q79...","{'corporateWebsite': 'https://www.epicor.com',...",{'year': 1984},We’re here for the hard-working businesses tha...,"5,001 to 10,000","[Austin, TX]",https://au.indeed.com/viewjob?jk=fbc1eabe01135c57,en,au,...,"As a Database Administrator at Epicor , you wi...",Database Admin,None,"SQL Server, PostgreSQL, Azure, Ansible, GitHub...",Data Engineer,"IT Operations, Software Development",Software,None,None mentioned,False
13,None,"{'startDate': None, 'photoUrl': None, 'name': ...",{'corporateWebsite': 'https://www.guildgroup.c...,{'year': 1963},"In 1961, a South Australian pharmacist, Mr Ala...",201 to 500,"[Level 15, 171 Collins Street\r\nMelbourne VIC...",https://au.indeed.com/viewjob?jk=31573efd5c4b4eec,en,au,...,Job Number: ELMO-00001363 Work type: Full Time...,Senior Predictive Analytics Consultant,None,"Python, R, SQL, MS Fabric, Databricks, Power B...",Data Analyst,"Insurance, Pricing, Actuarial",Insurance,None,None mentioned,False


##13.18 Before / After Cleaning Example

In [ ]:
display(cleaning_comparison_df)

,Field,Raw,Cleaned,Changed?
0,jobKey,9c2916ac1bdaf7e2,9c2916ac1bdaf7e2,False
1,companyName,Department of Education,Department of Education,False
2,title,Data Analyst,Data Analyst,False
3,descriptionText,This position commences 1 January 2027\n\nAre ...,This position commences 1 January 2027 Are you...,True
4,jobUrl,https://au.indeed.com/viewjob?jk=9c2916ac1bdaf7e2,https://au.indeed.com/viewjob?jk=9c2916ac1bdaf7e2,False
5,location,"{'latitude': -37.818, 'formattedAddressShort':...",Melbourne City Centre VIC,True
6,city,Melbourne City Centre,Melbourne City Centre,False
7,state,"{'latitude': -37.818, 'formattedAddressShort':...",None,True
8,datePublished,2026-08-21,2026-08-21 00:00:00+00:00,True


##13.19 Final Dashboard Summary

The dashboard summarises the complete pipeline:

Raw Indeed Data

        ↓
Cleaning & Standardisation

        ↓
Source jobKey Deduplication

        ↓
Location-Type Classification

        ↓
48-Hour Cross-Dataset Deduplication

        ↓
Gemini AI Enrichment

        ↓
20-Field Target Mapping

        ↓
Data-Quality Validation

        ↓
jobboard_staging
